# Constrained Decoding Engine — Benchmark & Demos (Colab / Kaggle)

Runs **baseline vs. prompt-and-retry vs. constrained** on a real model
(default **Qwen2.5-0.5B-Instruct**), then the **Phase 3** demos: the per-state
token index speedup, EBNF tool-call generation, and streaming.

**Colab:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all.
**Kaggle:** Settings → Accelerator → **GPU T4**, then Run All.

The engine source is embedded below (identical to the tested repo) — no clone
or upload needed.

## 1. Setup

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jsonschema"], check=True)
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE,
      "|", (torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU only (slower latency)"))
if DEVICE == "cpu":
    print("TIP: enable a GPU runtime for latency numbers (correctness is identical on CPU).")

## 2. Write the engine source (embedded)

In [ ]:
import os, base64
FILES = {
"engine/__init__.py": "IiIiQ29uc3RyYWluZWQgRGVjb2RpbmcgRW5naW5lLgoKQSBsaWJyYXJ5IHRoYXQgZ3VhcmFudGVlcyBhbiBMTE0ncyBvdXRwdXQgY29uZm9ybXMgdG8gYSBzY2hlbWEgb3IgZ3JhbW1hciBieQptYXNraW5nIGludmFsaWQgdG9rZW5zIGF0IGVhY2ggZGVjb2RlIHN0ZXAsIHNvIG91dHB1dCBpcyBjb3JyZWN0IGJ5CmNvbnN0cnVjdGlvbiByYXRoZXIgdGhhbiBjb3JyZWN0LWlmLWx1Y2t5LgoKUHVibGljIHN1cmZhY2UgZm9yIFBoYXNlIDEgKHJlZ2V4IC8gZmluaXRlLWF1dG9tYXRvbiksIFBoYXNlIDIgKEpTT04gc2NoZW1hIHZpYQphIHB1c2hkb3duIGF1dG9tYXRvbiksIGFuZCBQaGFzZSAzIChwZXItc3RhdGUgdG9rZW4gaW5kZXgsIEVCTkYgZ3JhbW1hcnMsCnN0cmVhbWluZykuCiIiIgoKZnJvbSBlbmdpbmUuYXV0b21hdG9uIGltcG9ydCBORkEsIERGQSwgU3RhdGUKZnJvbSBlbmdpbmUucmVnZXhfY29tcGlsZXIgaW1wb3J0IGNvbXBpbGVfcmVnZXgKZnJvbSBlbmdpbmUucHVzaGRvd24gaW1wb3J0IFBEQSwgR3JhbW1hciwgY29tcGlsZV9ncmFtbWFyCmZyb20gZW5naW5lLmpzb25fc2NoZW1hIGltcG9ydCBidWlsZF9qc29uX2F1dG9tYXRvbiwgYnVpbGRfanNvbl9wcm9jZXNzb3IsIHNjaGVtYV90b19ncmFtbWFyCmZyb20gZW5naW5lLnRva2VuX2luZGV4IGltcG9ydCBUb2tlbkluZGV4CmZyb20gZW5naW5lLmVibmYgaW1wb3J0IGNvbXBpbGVfZWJuZiwgbG9hZF9lYm5mLCBidWlsZF9lYm5mX3Byb2Nlc3Nvcgpmcm9tIGVuZ2luZS5zdHJlYW1pbmcgaW1wb3J0IFN0cmVhbVZhbGlkYXRvciwgc3RyZWFtX2NvbnN0cmFpbmVkLCBzdHJlYW1fZ2VuZXJhdGVfaGYKCl9fYWxsX18gPSBbCiAgICAiTkZBIiwKICAgICJERkEiLAogICAgIlN0YXRlIiwKICAgICJjb21waWxlX3JlZ2V4IiwKICAgICJQREEiLAogICAgIkdyYW1tYXIiLAogICAgImNvbXBpbGVfZ3JhbW1hciIsCiAgICAiYnVpbGRfanNvbl9hdXRvbWF0b24iLAogICAgImJ1aWxkX2pzb25fcHJvY2Vzc29yIiwKICAgICJzY2hlbWFfdG9fZ3JhbW1hciIsCiAgICAiVG9rZW5JbmRleCIsCiAgICAiY29tcGlsZV9lYm5mIiwKICAgICJsb2FkX2VibmYiLAogICAgImJ1aWxkX2VibmZfcHJvY2Vzc29yIiwKICAgICJTdHJlYW1WYWxpZGF0b3IiLAogICAgInN0cmVhbV9jb25zdHJhaW5lZCIsCiAgICAic3RyZWFtX2dlbmVyYXRlX2hmIiwKXQo=",
"engine/config.py": "IiIiU2hhcmVkIGNvbmZpZ3VyYXRpb24gY29uc3RhbnRzLgoKVGhlIG1vZGVsIGlkIGxpdmVzIGJlaGluZCBhIHNpbmdsZSBjb25zdGFudCBzbyBpdCBpcyB0cml2aWFsbHkgc3dhcHBhYmxlCih0aGUgcGxhbidzIHJlcXVpcmVtZW50KS4gRGVmYXVsdCBpcyBRd2VuMi41LTAuNUItSW5zdHJ1Y3Q6IHVuZ2F0ZWQsIH4xR0IsCnJ1bnMgb24gQ1BVLiBPdmVycmlkZSB3aXRoIHRoZSBDREVfTU9ERUwgZW52IHZhciB3aXRob3V0IHRvdWNoaW5nIGNvZGUuCiIiIgoKaW1wb3J0IG9zCgojIERlZmF1bHQgb3Blbi1zb3VyY2UgaW5zdHJ1Y3QgbW9kZWwuIFN3YXAgdmlhIGBleHBvcnQgQ0RFX01PREVMPS4uLmAuCkRFRkFVTFRfTU9ERUwgPSBvcy5lbnZpcm9uLmdldCgiQ0RFX01PREVMIiwgIlF3ZW4vUXdlbjIuNS0wLjVCLUluc3RydWN0IikKCiMgR2VuZXJhdGlvbiBkZWZhdWx0cyB1c2VkIGJ5IHRoZSBiZW5jaG1hcmsgaGFybmVzcy4KTUFYX05FV19UT0tFTlMgPSAyNTYKIyBHcmVlZHkgYnkgZGVmYXVsdCBzbyB0aGUgYmFzZWxpbmUgdnMgY29uc3RyYWluZWQgY29tcGFyaXNvbiBpcyBkZXRlcm1pbmlzdGljLgpET19TQU1QTEUgPSBGYWxzZQpURU1QRVJBVFVSRSA9IDEuMAoKCmRlZiBwaWNrX2RldmljZSgpOgogICAgIiIiUmV0dXJuICdjdWRhJyB3aGVuIGEgR1BVIGlzIHByZXNlbnQsIGVsc2UgJ2NwdScuIEtlZXBzIGNvZGUgZGV2aWNlLWFnbm9zdGljLiIiIgogICAgdHJ5OgogICAgICAgIGltcG9ydCB0b3JjaAoKICAgICAgICBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgICAgICByZXR1cm4gImN1ZGEiCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHBhc3MKICAgIHJldHVybiAiY3B1Igo=",
"engine/automaton.py": "IiIiRmluaXRlIGF1dG9tYXRhOiBORkEgd2l0aCBlcHNpbG9uIG1vdmVzLCBUaG9tcHNvbiBmcmFnbWVudHMsIGFuZCBzdWJzZXQKY29uc3RydWN0aW9uIHRvIGEgREZBLgoKRGVzaWduIG5vdGUg4oCUIHRoZSBPVEhFUiBzZW50aW5lbAotLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQpBIHJlZ2V4IHJhbmdlcyBvdmVyIGFuIHVuYm91bmRlZCBjaGFyYWN0ZXIgc2V0LCBidXQgYSBERkEgbmVlZHMgYSBmaW5pdGUKYWxwaGFiZXQuIFdlIGNvbGxlY3QgdGhlICpjb25jcmV0ZSogY2hhcmFjdGVycyBuYW1lZCBpbiB0aGUgcGF0dGVybiBpbnRvIGFuCmV4cGxpY2l0IGBgYWxwaGFiZXRgYCBhbmQgYWRkIG9uZSBzZW50aW5lbCBzeW1ib2wgOmRhdGE6YE9USEVSYCBzdGFuZGluZyBmb3IKImFueSBjaGFyYWN0ZXIgbm90IGluIHRoZSBhbHBoYWJldCIuIGBgLmBgIGFuZCBuZWdhdGVkIGNsYXNzZXMgbWF0Y2ggT1RIRVI7CmxpdGVyYWxzIGFuZCBwb3NpdGl2ZSBjbGFzc2VzIGRvIG5vdC4gQXQgcnVuIHRpbWUgYSBjb25jcmV0ZSBjaGFyYWN0ZXIgYGBjYGAgaXMKbWFwcGVkIHRvIGBgY2BgIGlmIGl0IGlzIGluIHRoZSBhbHBoYWJldCwgZWxzZSB0byBPVEhFUi4gVGhpcyBsZXRzIGEgZmluaXRlCmF1dG9tYXRvbiBkZWNpZGUgbWVtYmVyc2hpcCBmb3Igc3RyaW5ncyBvdmVyIHRoZSBmdWxsIFVuaWNvZGUgcmFuZ2UuCgpUaGVzZSBjbGFzc2VzIGFyZSBwdXJlIFB5dGhvbiAobm8gdG9yY2gpLCBzbyB0aGUgd2hvbGUgcmVnZXggcGF0aCBpcyB1bml0CnRlc3RhYmxlIHdpdGhvdXQgYSBtb2RlbC4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MsIGZpZWxkCmZyb20gdHlwaW5nIGltcG9ydCBEaWN0LCBGcm96ZW5TZXQsIEl0ZXJhYmxlLCBMaXN0LCBPcHRpb25hbCwgU2V0LCBUdXBsZQoKCmNsYXNzIF9PdGhlcjoKICAgICIiIlNlbnRpbmVsIHN5bWJvbCBmb3IgJ2FueSBjaGFyYWN0ZXIgb3V0c2lkZSB0aGUgcGF0dGVybidzIGFscGhhYmV0Jy4iIiIKCiAgICBfaW5zdGFuY2UgPSBOb25lCgogICAgZGVmIF9fbmV3X18oY2xzKToKICAgICAgICBpZiBjbHMuX2luc3RhbmNlIGlzIE5vbmU6CiAgICAgICAgICAgIGNscy5faW5zdGFuY2UgPSBzdXBlcigpLl9fbmV3X18oY2xzKQogICAgICAgIHJldHVybiBjbHMuX2luc3RhbmNlCgogICAgZGVmIF9fcmVwcl9fKHNlbGYpOiAgIyBwcmFnbWE6IG5vIGNvdmVyIC0gY29zbWV0aWMKICAgICAgICByZXR1cm4gIk9USEVSIgoKCiM6IFNpbmdsZXRvbiBzZW50aW5lbCAoc2VlIG1vZHVsZSBkb2NzdHJpbmcpLgpPVEhFUiA9IF9PdGhlcigpCgojIEEgc3ltYm9sIGlzIGVpdGhlciBhIHNpbmdsZS1jaGFyYWN0ZXIgc3RyaW5nIG9yIHRoZSBPVEhFUiBzZW50aW5lbC4KU3ltYm9sID0gb2JqZWN0CgojIEEgc2VudGluZWwgREZBIHN0YXRlIGlkIG1lYW5pbmcgImRlYWQiIChubyBhY2NlcHQgcmVhY2hhYmxlLCBjb25zdW1lcyBub3RoaW5nKS4KREVBRCA9IC0xCgoKQGRhdGFjbGFzcwpjbGFzcyBTdGF0ZToKICAgICIiIkFuIE5GQSBzdGF0ZS4gYGBlcHNgYCBhcmUgZXBzaWxvbiB0YXJnZXRzOyBgYG1vdmVzYGAgYXJlIChzeW1ib2xzLCB0YXJnZXQpCiAgICBjb25zdW1pbmcgdHJhbnNpdGlvbnMgd2hlcmUgYGBzeW1ib2xzYGAgaXMgYSBmcm96ZW5zZXQgb3ZlciB0aGUgc3ltYm9sCiAgICB1bml2ZXJzZSAoY29uY3JldGUgY2hhcnMgYW5kL29yIE9USEVSKS4iIiIKCiAgICBpZDogaW50CiAgICBlcHM6IFNldFtpbnRdID0gZmllbGQoZGVmYXVsdF9mYWN0b3J5PXNldCkKICAgIG1vdmVzOiBMaXN0W1R1cGxlW0Zyb3plblNldCwgaW50XV0gPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKCgpjbGFzcyBORkE6CiAgICAiIiJFcHNpbG9uLU5GQSB3aXRoIGEgc2luZ2xlIHN0YXJ0IGFuZCBzaW5nbGUgYWNjZXB0IHN0YXRlIChUaG9tcHNvbiBmb3JtKS4iIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgYWxwaGFiZXQ6IFNldFtzdHJdKToKICAgICAgICBzZWxmLnN0YXRlczogRGljdFtpbnQsIFN0YXRlXSA9IHt9CiAgICAgICAgc2VsZi5zdGFydDogaW50ID0gLTEKICAgICAgICBzZWxmLmFjY2VwdDogaW50ID0gLTEKICAgICAgICAjIENvbmNyZXRlIGNoYXJhY3RlcnMgbmFtZWQgYW55d2hlcmUgaW4gdGhlIHBhdHRlcm4uCiAgICAgICAgc2VsZi5hbHBoYWJldDogU2V0W3N0cl0gPSBzZXQoYWxwaGFiZXQpCgogICAgIyAtLSBjb25zdHJ1Y3Rpb24gaGVscGVycyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiAgICBkZWYgbmV3X3N0YXRlKHNlbGYpIC0+IGludDoKICAgICAgICBzaWQgPSBsZW4oc2VsZi5zdGF0ZXMpCiAgICAgICAgc2VsZi5zdGF0ZXNbc2lkXSA9IFN0YXRlKHNpZCkKICAgICAgICByZXR1cm4gc2lkCgogICAgZGVmIGFkZF9lcHMoc2VsZiwgc3JjOiBpbnQsIGRzdDogaW50KSAtPiBOb25lOgogICAgICAgIHNlbGYuc3RhdGVzW3NyY10uZXBzLmFkZChkc3QpCgogICAgZGVmIGFkZF9tb3ZlKHNlbGYsIHNyYzogaW50LCBzeW1ib2xzOiBGcm96ZW5TZXQsIGRzdDogaW50KSAtPiBOb25lOgogICAgICAgIHNlbGYuc3RhdGVzW3NyY10ubW92ZXMuYXBwZW5kKChzeW1ib2xzLCBkc3QpKQoKICAgICMgLS0gc3Vic2V0IGNvbnN0cnVjdGlvbiAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogICAgZGVmIGVwc2lsb25fY2xvc3VyZShzZWxmLCBpZHM6IEl0ZXJhYmxlW2ludF0pIC0+IEZyb3plblNldFtpbnRdOgogICAgICAgIHN0YWNrID0gbGlzdChpZHMpCiAgICAgICAgc2VlbjogU2V0W2ludF0gPSBzZXQoc3RhY2spCiAgICAgICAgd2hpbGUgc3RhY2s6CiAgICAgICAgICAgIHMgPSBzdGFjay5wb3AoKQogICAgICAgICAgICBmb3Igbnh0IGluIHNlbGYuc3RhdGVzW3NdLmVwczoKICAgICAgICAgICAgICAgIGlmIG54dCBub3QgaW4gc2VlbjoKICAgICAgICAgICAgICAgICAgICBzZWVuLmFkZChueHQpCiAgICAgICAgICAgICAgICAgICAgc3RhY2suYXBwZW5kKG54dCkKICAgICAgICByZXR1cm4gZnJvemVuc2V0KHNlZW4pCgogICAgZGVmIHRvX2RmYShzZWxmKSAtPiAiREZBIjoKICAgICAgICAiIiJTdWJzZXQgY29uc3RydWN0aW9uLiBQcm9kdWNlcyBhIERGQSBvdmVyIGBgYWxwaGFiZXQg4oiqIHtPVEhFUn1gYC4iIiIKICAgICAgICB1bml2ZXJzZTogTGlzdFtTeW1ib2xdID0gbGlzdChzZWxmLmFscGhhYmV0KSArIFtPVEhFUl0KCiAgICAgICAgc3RhcnRfc2V0ID0gc2VsZi5lcHNpbG9uX2Nsb3N1cmUoW3NlbGYuc3RhcnRdKQogICAgICAgIHNldF90b19pZDogRGljdFtGcm96ZW5TZXRbaW50XSwgaW50XSA9IHtzdGFydF9zZXQ6IDB9CiAgICAgICAgaWRfdG9fc2V0OiBMaXN0W0Zyb3plblNldFtpbnRdXSA9IFtzdGFydF9zZXRdCiAgICAgICAgdHJhbnNpdGlvbnM6IERpY3RbVHVwbGVbaW50LCBTeW1ib2xdLCBpbnRdID0ge30KICAgICAgICB3b3JrbGlzdCA9IFtzdGFydF9zZXRdCgogICAgICAgIHdoaWxlIHdvcmtsaXN0OgogICAgICAgICAgICBjdXIgPSB3b3JrbGlzdC5wb3AoKQogICAgICAgICAgICBjdXJfaWQgPSBzZXRfdG9faWRbY3VyXQogICAgICAgICAgICBmb3Igc3ltIGluIHVuaXZlcnNlOgogICAgICAgICAgICAgICAgIyBORkEgc3RhdGVzIHJlYWNoYWJsZSBieSBjb25zdW1pbmcgb25lIG9jY3VycmVuY2Ugb2YgYHN5bWAuCiAgICAgICAgICAgICAgICB0YXJnZXRzOiBTZXRbaW50XSA9IHNldCgpCiAgICAgICAgICAgICAgICBmb3IgcyBpbiBjdXI6CiAgICAgICAgICAgICAgICAgICAgZm9yIHN5bWJvbHMsIGRzdCBpbiBzZWxmLnN0YXRlc1tzXS5tb3ZlczoKICAgICAgICAgICAgICAgICAgICAgICAgaWYgc3ltIGluIHN5bWJvbHM6CiAgICAgICAgICAgICAgICAgICAgICAgICAgICB0YXJnZXRzLmFkZChkc3QpCiAgICAgICAgICAgICAgICBpZiBub3QgdGFyZ2V0czoKICAgICAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICAgICAgY2xvc3VyZSA9IHNlbGYuZXBzaWxvbl9jbG9zdXJlKHRhcmdldHMpCiAgICAgICAgICAgICAgICBpZiBjbG9zdXJlIG5vdCBpbiBzZXRfdG9faWQ6CiAgICAgICAgICAgICAgICAgICAgc2V0X3RvX2lkW2Nsb3N1cmVdID0gbGVuKGlkX3RvX3NldCkKICAgICAgICAgICAgICAgICAgICBpZF90b19zZXQuYXBwZW5kKGNsb3N1cmUpCiAgICAgICAgICAgICAgICAgICAgd29ya2xpc3QuYXBwZW5kKGNsb3N1cmUpCiAgICAgICAgICAgICAgICB0cmFuc2l0aW9uc1soY3VyX2lkLCBzeW0pXSA9IHNldF90b19pZFtjbG9zdXJlXQoKICAgICAgICBhY2NlcHQgPSB7CiAgICAgICAgICAgIHNpZAogICAgICAgICAgICBmb3Igc2lkLCBuZmFfc2V0IGluIGVudW1lcmF0ZShpZF90b19zZXQpCiAgICAgICAgICAgIGlmIHNlbGYuYWNjZXB0IGluIG5mYV9zZXQKICAgICAgICB9CiAgICAgICAgcmV0dXJuIERGQSgKICAgICAgICAgICAgbl9zdGF0ZXM9bGVuKGlkX3RvX3NldCksCiAgICAgICAgICAgIHN0YXJ0PTAsCiAgICAgICAgICAgIGFjY2VwdD1hY2NlcHQsCiAgICAgICAgICAgIHRyYW5zaXRpb25zPXRyYW5zaXRpb25zLAogICAgICAgICAgICBhbHBoYWJldD1zZXQoc2VsZi5hbHBoYWJldCksCiAgICAgICAgKQoKCmNsYXNzIERGQToKICAgICIiIkRldGVybWluaXN0aWMgYXV0b21hdG9uIG92ZXIgYGBhbHBoYWJldCDiiKoge09USEVSfWBgLgoKICAgIFRoZSB0d28gb3BlcmF0aW9ucyB0aGUgdG9rZW4gYnJpZGdlIHJlbGllcyBvbiBhcmUgOm1ldGg6YHN0ZXBgIChhZHZhbmNlIGJ5IGEKICAgIGNvbmNyZXRlIGNoYXJhY3RlciwgcmV0dXJuaW5nIDpkYXRhOmBERUFEYCB3aGVuIHRoZSBwYXRoIGRpZXMpIGFuZCB0aGUKICAgIDphdHRyOmBsaXZlX3N0YXRlc2Agc2V0IChzdGF0ZXMgZnJvbSB3aGljaCBhbiBhY2NlcHRpbmcgc3RhdGUgaXMgc3RpbGwKICAgIHJlYWNoYWJsZSDigJQgdXNlZCB0byByZWplY3QgdG9rZW5zIHRoYXQgd291bGQgc3RyYW5kIHVzKS4iIiIKCiAgICBkZWYgX19pbml0X18oCiAgICAgICAgc2VsZiwKICAgICAgICBuX3N0YXRlczogaW50LAogICAgICAgIHN0YXJ0OiBpbnQsCiAgICAgICAgYWNjZXB0OiBTZXRbaW50XSwKICAgICAgICB0cmFuc2l0aW9uczogRGljdFtUdXBsZVtpbnQsIFN5bWJvbF0sIGludF0sCiAgICAgICAgYWxwaGFiZXQ6IFNldFtzdHJdLAogICAgKToKICAgICAgICBzZWxmLm5fc3RhdGVzID0gbl9zdGF0ZXMKICAgICAgICBzZWxmLnN0YXJ0ID0gc3RhcnQKICAgICAgICBzZWxmLmFjY2VwdCA9IHNldChhY2NlcHQpCiAgICAgICAgc2VsZi50cmFuc2l0aW9ucyA9IHRyYW5zaXRpb25zCiAgICAgICAgc2VsZi5hbHBoYWJldCA9IHNldChhbHBoYWJldCkKICAgICAgICBzZWxmLmxpdmVfc3RhdGVzID0gc2VsZi5fY29tcHV0ZV9saXZlKCkKCiAgICAjIC0tIHF1ZXJpZXMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KICAgIGRlZiBfc3ltYm9sKHNlbGYsIGNoOiBzdHIpIC0+IFN5bWJvbDoKICAgICAgICByZXR1cm4gY2ggaWYgY2ggaW4gc2VsZi5hbHBoYWJldCBlbHNlIE9USEVSCgogICAgZGVmIHN0ZXAoc2VsZiwgc3RhdGU6IGludCwgY2g6IHN0cikgLT4gaW50OgogICAgICAgICIiIkFkdmFuY2Ugb25lIGNoYXJhY3Rlci4gUmV0dXJucyB0aGUgbmV4dCBzdGF0ZSBpZCwgb3IgOmRhdGE6YERFQURgLiIiIgogICAgICAgIGlmIHN0YXRlID09IERFQUQ6CiAgICAgICAgICAgIHJldHVybiBERUFECiAgICAgICAgcmV0dXJuIHNlbGYudHJhbnNpdGlvbnMuZ2V0KChzdGF0ZSwgc2VsZi5fc3ltYm9sKGNoKSksIERFQUQpCgogICAgZGVmIHJ1bihzZWxmLCBzdGF0ZTogaW50LCB0ZXh0OiBzdHIpIC0+IGludDoKICAgICAgICAiIiJBZHZhbmNlIHRocm91Z2ggYSB3aG9sZSBzdHJpbmc7IERFQUQgaWYgYW55IGNoYXJhY3RlciBpcyByZWplY3RlZC4iIiIKICAgICAgICBmb3IgY2ggaW4gdGV4dDoKICAgICAgICAgICAgc3RhdGUgPSBzZWxmLnN0ZXAoc3RhdGUsIGNoKQogICAgICAgICAgICBpZiBzdGF0ZSA9PSBERUFEOgogICAgICAgICAgICAgICAgcmV0dXJuIERFQUQKICAgICAgICByZXR1cm4gc3RhdGUKCiAgICBkZWYgaXNfZGVhZChzZWxmLCBzdGF0ZTogaW50KSAtPiBib29sOgogICAgICAgIHJldHVybiBzdGF0ZSA9PSBERUFECgogICAgZGVmIGlzX2FjY2VwdGluZyhzZWxmLCBzdGF0ZTogaW50KSAtPiBib29sOgogICAgICAgIHJldHVybiBzdGF0ZSBpbiBzZWxmLmFjY2VwdAoKICAgIGRlZiBpc19saXZlKHNlbGYsIHN0YXRlOiBpbnQpIC0+IGJvb2w6CiAgICAgICAgIiIiVHJ1ZSBpZiBhbiBhY2NlcHRpbmcgc3RhdGUgaXMgc3RpbGwgcmVhY2hhYmxlIGZyb20gYGBzdGF0ZWBgLgoKICAgICAgICBBIGxpdmUsIG5vbi1hY2NlcHRpbmcgc3RhdGUgbWVhbnMgInZhbGlkIHNvIGZhciwga2VlcCBnb2luZyI7IGEgZGVhZAogICAgICAgIHN0YXRlIG1lYW5zIHRoZSBzdHJpbmcgY2FuIG5ldmVyIGJlIGNvbXBsZXRlZCBpbnRvIGEgbWF0Y2guIiIiCiAgICAgICAgcmV0dXJuIHN0YXRlIGluIHNlbGYubGl2ZV9zdGF0ZXMKCiAgICBkZWYgYWNjZXB0cyhzZWxmLCB0ZXh0OiBzdHIpIC0+IGJvb2w6CiAgICAgICAgcmV0dXJuIHNlbGYuaXNfYWNjZXB0aW5nKHNlbGYucnVuKHNlbGYuc3RhcnQsIHRleHQpKQoKICAgICMgLS0gaW50ZXJuYWxzIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogICAgZGVmIF9jb21wdXRlX2xpdmUoc2VsZikgLT4gU2V0W2ludF06CiAgICAgICAgIiIiUmV2ZXJzZS1yZWFjaGFiaWxpdHkgZnJvbSBhY2NlcHRpbmcgc3RhdGVzIG92ZXIgYWxsIHRyYW5zaXRpb25zLiIiIgogICAgICAgIHByZWRzOiBEaWN0W2ludCwgU2V0W2ludF1dID0ge3M6IHNldCgpIGZvciBzIGluIHJhbmdlKHNlbGYubl9zdGF0ZXMpfQogICAgICAgIGZvciAoc3JjLCBfc3ltKSwgZHN0IGluIHNlbGYudHJhbnNpdGlvbnMuaXRlbXMoKToKICAgICAgICAgICAgcHJlZHNbZHN0XS5hZGQoc3JjKQogICAgICAgIGxpdmU6IFNldFtpbnRdID0gc2V0KHNlbGYuYWNjZXB0KQogICAgICAgIHN0YWNrID0gbGlzdChzZWxmLmFjY2VwdCkKICAgICAgICB3aGlsZSBzdGFjazoKICAgICAgICAgICAgcyA9IHN0YWNrLnBvcCgpCiAgICAgICAgICAgIGZvciBwIGluIHByZWRzW3NdOgogICAgICAgICAgICAgICAgaWYgcCBub3QgaW4gbGl2ZToKICAgICAgICAgICAgICAgICAgICBsaXZlLmFkZChwKQogICAgICAgICAgICAgICAgICAgIHN0YWNrLmFwcGVuZChwKQogICAgICAgIHJldHVybiBsaXZlCg==",
"engine/regex_compiler.py": "IiIiUmVnZXggLT4gREZBLCBoYW5kLXJvbGxlZC4KClBpcGVsaW5lOiByZWN1cnNpdmUtZGVzY2VudCBwYXJzZSBpbnRvIGFuIEFTVCwgVGhvbXBzb24gY29uc3RydWN0aW9uIGludG8gYW4KZXBzaWxvbi1ORkEsIHRoZW4gc3Vic2V0IGNvbnN0cnVjdGlvbiAoOm1ldGg6YE5GQS50b19kZmFgKSBpbnRvIGEgREZBLgoKU3VwcG9ydGVkIHN1YnNldCAoZW5vdWdoIGZvciBkYXRlcywgZW51bXMsIGZpeGVkIGZvcm1hdHMsIGFuZCBKU09OIHNjYWxhcnMpOgogICAgbGl0ZXJhbHMgYW5kIGVzY2FwZXMgIFxcLiBcXGQgXFx3IFxccyBcXEQgXFxXIFxcUyBcXG4gXFx0IFxcciBhbmQgXFw8bWV0YT4KICAgIC4gKGFueSBjaGFyYWN0ZXIpICAgICBjaGFyIGNsYXNzZXMgW2FiY10gW2Etel0gW14wLTldIHdpdGggcmFuZ2VzCiAgICBxdWFudGlmaWVycyAgICAgICAgICAgKiAgKyAgPyAge219ICB7bSx9ICB7bSxufQogICAgZ3JvdXBpbmcgICAgICAgICAgICAgICggLi4uICkgICAgICAgIGFsdGVybmF0aW9uICBhfGIKQ29uY2F0ZW5hdGlvbiBpcyBpbXBsaWNpdC4gQW5jaG9ycywgYmFja3JlZmVyZW5jZXMsIGFuZCBsb29rYXJvdW5kIGFyZSBvdXQgb2YKc2NvcGUgKGRvY3VtZW50ZWQgaW4gdGhlIFJFQURNRSkuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmZyb20gdHlwaW5nIGltcG9ydCBMaXN0LCBPcHRpb25hbCwgU2V0Cgpmcm9tIGVuZ2luZS5hdXRvbWF0b24gaW1wb3J0IE5GQSwgT1RIRVIsIERGQQoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tICMKIyBBU1QKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gIwpAZGF0YWNsYXNzCmNsYXNzIF9TcGVjOgogICAgIiIiQSBvbmUtc3ltYm9sIG1hdGNoZXIgcmVzb2x2ZWQgYWdhaW5zdCB0aGUgZmluYWwgYWxwaGFiZXQgYXQgYnVpbGQgdGltZS4iIiIKCiAgICBraW5kOiBzdHIgICMgImxpdCIgfCAiYW55IiB8ICJjbGFzcyIKICAgIGNoYXJzOiBTZXRbc3RyXSAgIyBmb3IgImxpdCIvImNsYXNzIjogdGhlIG5hbWVkIGNoYXJhY3RlcnMKICAgIG5lZ2F0ZWQ6IGJvb2wgPSBGYWxzZSAgIyBmb3IgImNsYXNzIgoKICAgIGRlZiBzeW1ib2xzKHNlbGYsIGFscGhhYmV0OiBTZXRbc3RyXSk6CiAgICAgICAgaWYgc2VsZi5raW5kID09ICJsaXQiOgogICAgICAgICAgICByZXR1cm4gZnJvemVuc2V0KHNlbGYuY2hhcnMpCiAgICAgICAgaWYgc2VsZi5raW5kID09ICJhbnkiOgogICAgICAgICAgICByZXR1cm4gZnJvemVuc2V0KHNldChhbHBoYWJldCkgfCB7T1RIRVJ9KQogICAgICAgICMgY2xhc3MKICAgICAgICBpZiBub3Qgc2VsZi5uZWdhdGVkOgogICAgICAgICAgICByZXR1cm4gZnJvemVuc2V0KHNlbGYuY2hhcnMpCiAgICAgICAgcmV0dXJuIGZyb3plbnNldCgoc2V0KGFscGhhYmV0KSAtIHNlbGYuY2hhcnMpIHwge09USEVSfSkKCgpAZGF0YWNsYXNzCmNsYXNzIF9MZWFmOgogICAgc3BlYzogX1NwZWMKCgpAZGF0YWNsYXNzCmNsYXNzIF9FbXB0eToKICAgIHBhc3MKCgpAZGF0YWNsYXNzCmNsYXNzIF9Db25jYXQ6CiAgICBwYXJ0czogTGlzdFtvYmplY3RdCgoKQGRhdGFjbGFzcwpjbGFzcyBfQWx0OgogICAgb3B0aW9uczogTGlzdFtvYmplY3RdCgoKQGRhdGFjbGFzcwpjbGFzcyBfUmVwZWF0OgogICAgbm9kZTogb2JqZWN0CiAgICBtOiBpbnQKICAgIG46IE9wdGlvbmFsW2ludF0gICMgTm9uZSA9IHVuYm91bmRlZAoKCiMgQ2hhcmFjdGVyLWNsYXNzIGV4cGFuc2lvbnMgZm9yIGVzY2FwZXMuCl9ESUdJVFMgPSBzZXQoIjAxMjM0NTY3ODkiKQpfV09SRCA9IHNldCgiYWJjZGVmZ2hpamtsbW5vcHFyc3R1dnd4eXpBQkNERUZHSElKS0xNTk9QUVJTVFVWV1hZWjAxMjM0NTY3ODlfIikKX1NQQUNFID0gc2V0KCIgXHRcblxyXGZcdiIpCgoKY2xhc3MgX1BhcnNlcjoKICAgICIiIlJlY3Vyc2l2ZS1kZXNjZW50IHBhcnNlci4gQ29sbGVjdHMgY29uY3JldGUgY2hhcmFjdGVycyBpbnRvIGBgYWxwaGFiZXRgYAogICAgc28gdGhlIGF1dG9tYXRvbidzIHN5bWJvbCB1bml2ZXJzZSBpcyBrbm93biBiZWZvcmUgTkZBIGNvbnN0cnVjdGlvbi4iIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgcGF0dGVybjogc3RyKToKICAgICAgICBzZWxmLnMgPSBwYXR0ZXJuCiAgICAgICAgc2VsZi5pID0gMAogICAgICAgIHNlbGYuYWxwaGFiZXQ6IFNldFtzdHJdID0gc2V0KCkKCiAgICAjIC0tIGxvdy1sZXZlbCAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KICAgIGRlZiBfcGVlayhzZWxmKSAtPiBPcHRpb25hbFtzdHJdOgogICAgICAgIHJldHVybiBzZWxmLnNbc2VsZi5pXSBpZiBzZWxmLmkgPCBsZW4oc2VsZi5zKSBlbHNlIE5vbmUKCiAgICBkZWYgX25leHQoc2VsZikgLT4gc3RyOgogICAgICAgIGNoID0gc2VsZi5zW3NlbGYuaV0KICAgICAgICBzZWxmLmkgKz0gMQogICAgICAgIHJldHVybiBjaAoKICAgIGRlZiBfZW9mKHNlbGYpIC0+IGJvb2w6CiAgICAgICAgcmV0dXJuIHNlbGYuaSA+PSBsZW4oc2VsZi5zKQoKICAgIGRlZiBfbm90ZShzZWxmLCBjaGFycykgLT4gTm9uZToKICAgICAgICBzZWxmLmFscGhhYmV0LnVwZGF0ZShjaGFycykKCiAgICAjIC0tIGdyYW1tYXI6IGFsdCAtPiBjb25jYXQgKCd8JyBjb25jYXQpKiAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KICAgIGRlZiBwYXJzZShzZWxmKToKICAgICAgICBub2RlID0gc2VsZi5fYWx0KCkKICAgICAgICBpZiBub3Qgc2VsZi5fZW9mKCk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJ1bmV4cGVjdGVkIHtzZWxmLl9wZWVrKCkhcn0gYXQgcG9zaXRpb24ge3NlbGYuaX0iKQogICAgICAgIHJldHVybiBub2RlCgogICAgZGVmIF9hbHQoc2VsZik6CiAgICAgICAgb3B0aW9ucyA9IFtzZWxmLl9jb25jYXQoKV0KICAgICAgICB3aGlsZSBzZWxmLl9wZWVrKCkgPT0gInwiOgogICAgICAgICAgICBzZWxmLl9uZXh0KCkKICAgICAgICAgICAgb3B0aW9ucy5hcHBlbmQoc2VsZi5fY29uY2F0KCkpCiAgICAgICAgcmV0dXJuIG9wdGlvbnNbMF0gaWYgbGVuKG9wdGlvbnMpID09IDEgZWxzZSBfQWx0KG9wdGlvbnMpCgogICAgZGVmIF9jb25jYXQoc2VsZik6CiAgICAgICAgcGFydHMgPSBbXQogICAgICAgIHdoaWxlIFRydWU6CiAgICAgICAgICAgIGNoID0gc2VsZi5fcGVlaygpCiAgICAgICAgICAgIGlmIGNoIGlzIE5vbmUgb3IgY2ggaW4gInwpIjoKICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgIHBhcnRzLmFwcGVuZChzZWxmLl9yZXBlYXQoKSkKICAgICAgICBpZiBub3QgcGFydHM6CiAgICAgICAgICAgIHJldHVybiBfRW1wdHkoKQogICAgICAgIHJldHVybiBwYXJ0c1swXSBpZiBsZW4ocGFydHMpID09IDEgZWxzZSBfQ29uY2F0KHBhcnRzKQoKICAgIGRlZiBfcmVwZWF0KHNlbGYpOgogICAgICAgIGF0b20gPSBzZWxmLl9hdG9tKCkKICAgICAgICBjaCA9IHNlbGYuX3BlZWsoKQogICAgICAgIGlmIGNoID09ICIqIjoKICAgICAgICAgICAgc2VsZi5fbmV4dCgpCiAgICAgICAgICAgIHJldHVybiBfUmVwZWF0KGF0b20sIDAsIE5vbmUpCiAgICAgICAgaWYgY2ggPT0gIisiOgogICAgICAgICAgICBzZWxmLl9uZXh0KCkKICAgICAgICAgICAgcmV0dXJuIF9SZXBlYXQoYXRvbSwgMSwgTm9uZSkKICAgICAgICBpZiBjaCA9PSAiPyI6CiAgICAgICAgICAgIHNlbGYuX25leHQoKQogICAgICAgICAgICByZXR1cm4gX1JlcGVhdChhdG9tLCAwLCAxKQogICAgICAgIGlmIGNoID09ICJ7IjoKICAgICAgICAgICAgcmV0dXJuIHNlbGYuX2JyYWNlKGF0b20pCiAgICAgICAgcmV0dXJuIGF0b20KCiAgICBkZWYgX2JyYWNlKHNlbGYsIGF0b20pOgogICAgICAgIGFzc2VydCBzZWxmLl9uZXh0KCkgPT0gInsiCiAgICAgICAgZGlnaXRzID0gIiIKICAgICAgICB3aGlsZSBzZWxmLl9wZWVrKCkgaXMgbm90IE5vbmUgYW5kIHNlbGYuX3BlZWsoKS5pc2RpZ2l0KCk6CiAgICAgICAgICAgIGRpZ2l0cyArPSBzZWxmLl9uZXh0KCkKICAgICAgICBtID0gaW50KGRpZ2l0cykgaWYgZGlnaXRzIGVsc2UgMAogICAgICAgIG46IE9wdGlvbmFsW2ludF0gPSBtCiAgICAgICAgaWYgc2VsZi5fcGVlaygpID09ICIsIjoKICAgICAgICAgICAgc2VsZi5fbmV4dCgpCiAgICAgICAgICAgIGQyID0gIiIKICAgICAgICAgICAgd2hpbGUgc2VsZi5fcGVlaygpIGlzIG5vdCBOb25lIGFuZCBzZWxmLl9wZWVrKCkuaXNkaWdpdCgpOgogICAgICAgICAgICAgICAgZDIgKz0gc2VsZi5fbmV4dCgpCiAgICAgICAgICAgIG4gPSBpbnQoZDIpIGlmIGQyIGVsc2UgTm9uZQogICAgICAgIGlmIHNlbGYuX3BlZWsoKSAhPSAifSI6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJ1bnRlcm1pbmF0ZWQge3t9fSBxdWFudGlmaWVyIGF0IHBvc2l0aW9uIHtzZWxmLml9IikKICAgICAgICBzZWxmLl9uZXh0KCkgICMgY29uc3VtZSB9CiAgICAgICAgcmV0dXJuIF9SZXBlYXQoYXRvbSwgbSwgbikKCiAgICBkZWYgX2F0b20oc2VsZik6CiAgICAgICAgY2ggPSBzZWxmLl9wZWVrKCkKICAgICAgICBpZiBjaCA9PSAiKCI6CiAgICAgICAgICAgIHNlbGYuX25leHQoKQogICAgICAgICAgICBub2RlID0gc2VsZi5fYWx0KCkKICAgICAgICAgICAgaWYgc2VsZi5fcGVlaygpICE9ICIpIjoKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInVuYmFsYW5jZWQgcGFyZW50aGVzaXMiKQogICAgICAgICAgICBzZWxmLl9uZXh0KCkKICAgICAgICAgICAgcmV0dXJuIG5vZGUKICAgICAgICBpZiBjaCA9PSAiWyI6CiAgICAgICAgICAgIHJldHVybiBzZWxmLl9jaGFyX2NsYXNzKCkKICAgICAgICBpZiBjaCA9PSAiLiI6CiAgICAgICAgICAgIHNlbGYuX25leHQoKQogICAgICAgICAgICByZXR1cm4gX0xlYWYoX1NwZWMoImFueSIsIHNldCgpKSkKICAgICAgICBpZiBjaCA9PSAiXFwiOgogICAgICAgICAgICByZXR1cm4gc2VsZi5fZXNjYXBlKCkKICAgICAgICBpZiBjaCBpbiAiKXwqKz97fSI6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJ1bmV4cGVjdGVkIHtjaCFyfSBhdCBwb3NpdGlvbiB7c2VsZi5pfSIpCiAgICAgICAgIyBsaXRlcmFsCiAgICAgICAgc2VsZi5fbmV4dCgpCiAgICAgICAgc2VsZi5fbm90ZShbY2hdKQogICAgICAgIHJldHVybiBfTGVhZihfU3BlYygibGl0Iiwge2NofSkpCgogICAgZGVmIF9lc2NhcGUoc2VsZik6CiAgICAgICAgYXNzZXJ0IHNlbGYuX25leHQoKSA9PSAiXFwiCiAgICAgICAgaWYgc2VsZi5fZW9mKCk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInRyYWlsaW5nIGJhY2tzbGFzaCIpCiAgICAgICAgY2ggPSBzZWxmLl9uZXh0KCkKICAgICAgICBtYXBwaW5nID0gewogICAgICAgICAgICAiZCI6ICgiY2xhc3MiLCBfRElHSVRTLCBGYWxzZSksCiAgICAgICAgICAgICJEIjogKCJjbGFzcyIsIF9ESUdJVFMsIFRydWUpLAogICAgICAgICAgICAidyI6ICgiY2xhc3MiLCBfV09SRCwgRmFsc2UpLAogICAgICAgICAgICAiVyI6ICgiY2xhc3MiLCBfV09SRCwgVHJ1ZSksCiAgICAgICAgICAgICJzIjogKCJjbGFzcyIsIF9TUEFDRSwgRmFsc2UpLAogICAgICAgICAgICAiUyI6ICgiY2xhc3MiLCBfU1BBQ0UsIFRydWUpLAogICAgICAgIH0KICAgICAgICBpZiBjaCBpbiBtYXBwaW5nOgogICAgICAgICAgICBraW5kLCBjaGFycywgbmVnID0gbWFwcGluZ1tjaF0KICAgICAgICAgICAgc2VsZi5fbm90ZShjaGFycykKICAgICAgICAgICAgcmV0dXJuIF9MZWFmKF9TcGVjKCJjbGFzcyIsIHNldChjaGFycyksIG5lZykpCiAgICAgICAgbGl0ZXJhbCA9IHsibiI6ICJcbiIsICJ0IjogIlx0IiwgInIiOiAiXHIiLCAiZiI6ICJcZiIsICJ2IjogIlx2IiwgIjAiOiAiXDAifQogICAgICAgIHJlYWwgPSBsaXRlcmFsLmdldChjaCwgY2gpICAjIFw8bWV0YT4gLT4gdGhlIG1ldGFjaGFyYWN0ZXIgbGl0ZXJhbGx5CiAgICAgICAgc2VsZi5fbm90ZShbcmVhbF0pCiAgICAgICAgcmV0dXJuIF9MZWFmKF9TcGVjKCJsaXQiLCB7cmVhbH0pKQoKICAgIGRlZiBfY2hhcl9jbGFzcyhzZWxmKToKICAgICAgICBhc3NlcnQgc2VsZi5fbmV4dCgpID09ICJbIgogICAgICAgIG5lZ2F0ZWQgPSBGYWxzZQogICAgICAgIGlmIHNlbGYuX3BlZWsoKSA9PSAiXiI6CiAgICAgICAgICAgIHNlbGYuX25leHQoKQogICAgICAgICAgICBuZWdhdGVkID0gVHJ1ZQogICAgICAgIGNoYXJzOiBTZXRbc3RyXSA9IHNldCgpCiAgICAgICAgaWYgc2VsZi5fcGVlaygpID09ICJdIjogICMgbGl0ZXJhbCBdIGFzIGZpcnN0IG1lbWJlcgogICAgICAgICAgICBjaGFycy5hZGQoc2VsZi5fbmV4dCgpKQogICAgICAgIHdoaWxlIHNlbGYuX3BlZWsoKSBpcyBub3QgTm9uZSBhbmQgc2VsZi5fcGVlaygpICE9ICJdIjoKICAgICAgICAgICAgYyA9IHNlbGYuX2NsYXNzX2NoYXIoKQogICAgICAgICAgICAjIHJhbmdlIGEtegogICAgICAgICAgICBpZiBzZWxmLl9wZWVrKCkgPT0gIi0iIGFuZCBzZWxmLmkgKyAxIDwgbGVuKHNlbGYucykgYW5kIHNlbGYuc1tzZWxmLmkgKyAxXSAhPSAiXSI6CiAgICAgICAgICAgICAgICBzZWxmLl9uZXh0KCkgICMgY29uc3VtZSAnLScKICAgICAgICAgICAgICAgIGVuZCA9IHNlbGYuX2NsYXNzX2NoYXIoKQogICAgICAgICAgICAgICAgZm9yIGNvZGUgaW4gcmFuZ2Uob3JkKGMpLCBvcmQoZW5kKSArIDEpOgogICAgICAgICAgICAgICAgICAgIGNoYXJzLmFkZChjaHIoY29kZSkpCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICBjaGFycy5hZGQoYykKICAgICAgICBpZiBzZWxmLl9wZWVrKCkgIT0gIl0iOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJ1bnRlcm1pbmF0ZWQgY2hhcmFjdGVyIGNsYXNzIikKICAgICAgICBzZWxmLl9uZXh0KCkgICMgY29uc3VtZSBdCiAgICAgICAgc2VsZi5fbm90ZShjaGFycykKICAgICAgICByZXR1cm4gX0xlYWYoX1NwZWMoImNsYXNzIiwgY2hhcnMsIG5lZ2F0ZWQpKQoKICAgIGRlZiBfY2xhc3NfY2hhcihzZWxmKSAtPiBzdHI6CiAgICAgICAgY2ggPSBzZWxmLl9uZXh0KCkKICAgICAgICBpZiBjaCA9PSAiXFwiOgogICAgICAgICAgICBpZiBzZWxmLl9lb2YoKToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInRyYWlsaW5nIGJhY2tzbGFzaCBpbiBjbGFzcyIpCiAgICAgICAgICAgIGUgPSBzZWxmLl9uZXh0KCkKICAgICAgICAgICAgcmV0dXJuIHsibiI6ICJcbiIsICJ0IjogIlx0IiwgInIiOiAiXHIiLCAiZiI6ICJcZiIsICJ2IjogIlx2In0uZ2V0KGUsIGUpCiAgICAgICAgcmV0dXJuIGNoCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gIwojIFRob21wc29uIGNvbnN0cnVjdGlvbjogQVNUIC0+IE5GQSBmcmFnbWVudAojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSAjCmRlZiBfYnVpbGQobm9kZSwgbmZhOiBORkEpOgogICAgIiIiUmV0dXJuIChzdGFydCwgYWNjZXB0KSBmb3IgYGBub2RlYGAsIGFkZGluZyBzdGF0ZXMgdG8gYGBuZmFgYC4KCiAgICBDYWxsZWQgcmVwZWF0ZWRseSBvbiB0aGUgc2FtZSBzdWJ0cmVlIGZvciBgYHttLG59YGAgZXhwYW5zaW9uLCBlYWNoIGNhbGwKICAgIG1pbnRpbmcgZnJlc2ggc3RhdGVzIOKAlCBzbyBjb3BpZXMgc2hhcmUgbm8gc3RhdGUuIiIiCiAgICBpZiBpc2luc3RhbmNlKG5vZGUsIF9FbXB0eSk6CiAgICAgICAgcywgYSA9IG5mYS5uZXdfc3RhdGUoKSwgbmZhLm5ld19zdGF0ZSgpCiAgICAgICAgbmZhLmFkZF9lcHMocywgYSkKICAgICAgICByZXR1cm4gcywgYQoKICAgIGlmIGlzaW5zdGFuY2Uobm9kZSwgX0xlYWYpOgogICAgICAgIHMsIGEgPSBuZmEubmV3X3N0YXRlKCksIG5mYS5uZXdfc3RhdGUoKQogICAgICAgIG5mYS5hZGRfbW92ZShzLCBub2RlLnNwZWMuc3ltYm9scyhuZmEuYWxwaGFiZXQpLCBhKQogICAgICAgIHJldHVybiBzLCBhCgogICAgaWYgaXNpbnN0YW5jZShub2RlLCBfQ29uY2F0KToKICAgICAgICBzdGFydCA9IHByZXZfYWNjZXB0ID0gTm9uZQogICAgICAgIGZvciBwYXJ0IGluIG5vZGUucGFydHM6CiAgICAgICAgICAgIHMsIGEgPSBfYnVpbGQocGFydCwgbmZhKQogICAgICAgICAgICBpZiBzdGFydCBpcyBOb25lOgogICAgICAgICAgICAgICAgc3RhcnQgPSBzCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICBuZmEuYWRkX2VwcyhwcmV2X2FjY2VwdCwgcykKICAgICAgICAgICAgcHJldl9hY2NlcHQgPSBhCiAgICAgICAgcmV0dXJuIHN0YXJ0LCBwcmV2X2FjY2VwdAoKICAgIGlmIGlzaW5zdGFuY2Uobm9kZSwgX0FsdCk6CiAgICAgICAgcywgYSA9IG5mYS5uZXdfc3RhdGUoKSwgbmZhLm5ld19zdGF0ZSgpCiAgICAgICAgZm9yIG9wdCBpbiBub2RlLm9wdGlvbnM6CiAgICAgICAgICAgIG9zXywgb2EgPSBfYnVpbGQob3B0LCBuZmEpCiAgICAgICAgICAgIG5mYS5hZGRfZXBzKHMsIG9zXykKICAgICAgICAgICAgbmZhLmFkZF9lcHMob2EsIGEpCiAgICAgICAgcmV0dXJuIHMsIGEKCiAgICBpZiBpc2luc3RhbmNlKG5vZGUsIF9SZXBlYXQpOgogICAgICAgIHJldHVybiBfYnVpbGRfcmVwZWF0KG5vZGUsIG5mYSkKCiAgICByYWlzZSBUeXBlRXJyb3IoZiJ1bmtub3duIEFTVCBub2RlIHtub2RlIXJ9IikKCgpkZWYgX2J1aWxkX3JlcGVhdChub2RlOiBfUmVwZWF0LCBuZmE6IE5GQSk6CiAgICBtLCBuID0gbm9kZS5tLCBub2RlLm4KICAgIHN0YXJ0ID0gbmZhLm5ld19zdGF0ZSgpCiAgICBjdXIgPSBzdGFydAogICAgIyBgbWAgbWFuZGF0b3J5IGNvcGllcy4KICAgIGZvciBfIGluIHJhbmdlKG0pOgogICAgICAgIHMsIGEgPSBfYnVpbGQobm9kZS5ub2RlLCBuZmEpCiAgICAgICAgbmZhLmFkZF9lcHMoY3VyLCBzKQogICAgICAgIGN1ciA9IGEKICAgIGlmIG4gaXMgTm9uZToKICAgICAgICAjIFVuYm91bmRlZCB0YWlsOiBhIEtsZWVuZSBzdGFyIG92ZXIgb25lIG1vcmUgY29weS4KICAgICAgICBsb29wX3N0YXJ0ID0gbmZhLm5ld19zdGF0ZSgpCiAgICAgICAgbmZhLmFkZF9lcHMoY3VyLCBsb29wX3N0YXJ0KQogICAgICAgIHMsIGEgPSBfYnVpbGQobm9kZS5ub2RlLCBuZmEpCiAgICAgICAgbmZhLmFkZF9lcHMobG9vcF9zdGFydCwgcykKICAgICAgICBuZmEuYWRkX2VwcyhhLCBsb29wX3N0YXJ0KQogICAgICAgIGFjY2VwdCA9IG5mYS5uZXdfc3RhdGUoKQogICAgICAgIG5mYS5hZGRfZXBzKGxvb3Bfc3RhcnQsIGFjY2VwdCkKICAgICAgICByZXR1cm4gc3RhcnQsIGFjY2VwdAogICAgIyBCb3VuZGVkIHRhaWw6IChuIC0gbSkgb3B0aW9uYWwgY29waWVzLCBlYWNoIHNraXBwYWJsZSB0byB0aGUgZW5kLgogICAgYWNjZXB0ID0gbmZhLm5ld19zdGF0ZSgpCiAgICBuZmEuYWRkX2VwcyhjdXIsIGFjY2VwdCkKICAgIGZvciBfIGluIHJhbmdlKG4gLSBtKToKICAgICAgICBzLCBhID0gX2J1aWxkKG5vZGUubm9kZSwgbmZhKQogICAgICAgIG5mYS5hZGRfZXBzKGN1ciwgcykKICAgICAgICBuZmEuYWRkX2VwcyhhLCBhY2NlcHQpCiAgICAgICAgY3VyID0gYQogICAgcmV0dXJuIHN0YXJ0LCBhY2NlcHQKCgpkZWYgY29tcGlsZV9yZWdleChwYXR0ZXJuOiBzdHIpIC0+IERGQToKICAgICIiIkNvbXBpbGUgYGBwYXR0ZXJuYGAgKGEgc3VwcG9ydGVkLXN1YnNldCByZWdleCkgaW50byBhIDpjbGFzczpgREZBYC4iIiIKICAgIHBhcnNlciA9IF9QYXJzZXIocGF0dGVybikKICAgIGFzdCA9IHBhcnNlci5wYXJzZSgpCiAgICBuZmEgPSBORkEocGFyc2VyLmFscGhhYmV0KQogICAgc3RhcnQsIGFjY2VwdCA9IF9idWlsZChhc3QsIG5mYSkKICAgIG5mYS5zdGFydCwgbmZhLmFjY2VwdCA9IHN0YXJ0LCBhY2NlcHQKICAgIHJldHVybiBuZmEudG9fZGZhKCkKCgpkZWYgcGFyc2VfcmVnZXhfYXN0KHBhdHRlcm46IHN0cik6CiAgICAiIiJQYXJzZSBgYHBhdHRlcm5gYCBhbmQgcmV0dXJuIGBgKGFzdCwgYWxwaGFiZXQpYGAuCgogICAgRXhwb3NlcyB0aGUgcmVnZXggQVNUIHNvIG90aGVyIGNvbXBpbGVycyAoZS5nLiB0aGUgUGhhc2UtMyBFQk5GIGZyb250ZW5kKQogICAgY2FuIHRyYW5zbGF0ZSBhIGBgL3JlZ2V4L2BgIHRlcm1pbmFsIGludG8gcHVzaGRvd24gZ3JhbW1hciBjb21iaW5hdG9ycywKICAgIHJldXNpbmcgdGhpcyBwYXJzZXIgaW5zdGVhZCBvZiBhIHNlY29uZCByZWdleCBpbXBsZW1lbnRhdGlvbi4iIiIKICAgIHBhcnNlciA9IF9QYXJzZXIocGF0dGVybikKICAgIGFzdCA9IHBhcnNlci5wYXJzZSgpCiAgICByZXR1cm4gYXN0LCBwYXJzZXIuYWxwaGFiZXQK",
"engine/token_bridge.py": "IiIiVGhlIGNoYXIgPC0+IHRva2VuIGJyaWRnZTogdGhlIGhhcmQgcGFydCBvZiB0aGUgcHJvamVjdC4KClRoZSBhdXRvbWF0b24gcmVhc29ucyBpbiAqY2hhcmFjdGVycyo7IHRoZSBtb2RlbCBlbWl0cyAqdG9rZW5zKiAoc3Vid29yZHMpLCBhbmQKYSBzaW5nbGUgdG9rZW4gY2FuIHNwYW4gc2V2ZXJhbCB0cmFuc2l0aW9ucyBvciBzdHJhZGRsZSBhIHN0cnVjdHVyYWwgYm91bmRhcnkuCkZvciBhIGdpdmVuIGF1dG9tYXRvbiBzdGF0ZSB3ZSBtdXN0IGNvbXB1dGUgdGhlIHNldCBvZiB0b2tlbiBpZHMgd2hvc2UgZGVjb2RlZAp0ZXh0IGtlZXBzIHRoZSBhdXRvbWF0b24gb24gYSBsaXZlIHBhdGguCgpUaGlzIG1vZHVsZSBpcyBkZWxpYmVyYXRlbHkgYXV0b21hdG9uLWFnbm9zdGljOiBpdCB0YWxrcyB0byBhbnkgb2JqZWN0IGV4cG9zaW5nCmBgc3RlcChzdGF0ZSwgY2gpIC0+IHN0YXRlYGAsIGBgaXNfZGVhZGBgLCBgYGlzX2xpdmVgYCwgYGBpc19hY2NlcHRpbmdgYCBhbmQgYQpgYHN0YXJ0YGAgYXR0cmlidXRlLiBUaGUgUGhhc2UtMSA6Y2xhc3M6YH5lbmdpbmUuYXV0b21hdG9uLkRGQWAgYW5kIHRoZSBQaGFzZS0yCnB1c2hkb3duIGF1dG9tYXRvbiBib3RoIHNhdGlzZnkgaXQsIHNvIHRoZSBicmlkZ2UgYW5kIG1hc2tlciBhcmUgcmV1c2VkIGFzLWlzLgoKUGhhc2UgMSB1c2VzIHRoZSBuYWl2ZSwgY29ycmVjdCwgTyh2b2NhYikgcGVyLXN0ZXAgc2Nhbi4gUGhhc2UgMyByZXBsYWNlcyB0aGUKc2NhbiB3aXRoIGEgcHJlY29tcHV0ZWQgcGVyLXN0YXRlIGluZGV4ICg6bW9kOmBlbmdpbmUudG9rZW5faW5kZXhgKSBmb3Igc3BlZWQ7CnRoZSBzZW1hbnRpY3MgZGVmaW5lZCBoZXJlIGFyZSB0aGUgc3BlY2lmaWNhdGlvbiB0aGF0IGluZGV4IG11c3QgbWF0Y2guCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBmaWVsZApmcm9tIHR5cGluZyBpbXBvcnQgRGljdCwgSXRlcmFibGUsIE9wdGlvbmFsLCBTZXQsIFR1cGxlCgojIFRoZSBVbmljb2RlIHJlcGxhY2VtZW50IGNoYXJhY3RlciBtYXJrcyB0b2tlbnMgdGhhdCBkZWNvZGUgdG8gcGFydGlhbCBieXRlcwojIChieXRlLWxldmVsIEJQRSBmcmFnbWVudHMpLiBXZSBkcm9wIHRoZW0gZnJvbSB0aGUgY29uc3RyYWluZWQgdm9jYWJ1bGFyeToKIyBlbWl0dGluZyBvbmUgbWlkLXBhdHRlcm4gY2FuIG5ldmVyIGJlIHJlYXNvbmVkIGFib3V0IGF0IHRoZSBjaGFyYWN0ZXIgbGV2ZWwuCl9SRVBMQUNFTUVOVCA9ICLvv70iCgoKQGRhdGFjbGFzcwpjbGFzcyBUb2tlblZvY2FiOgogICAgIiIiVG9rZW4gaWQgLT4gZGVjb2RlZCB0ZXh0LCBwbHVzIHRoZSBFT1MgaWQgdXNlZCB0byBlbmQgZ2VuZXJhdGlvbi4KCiAgICBCdWlsdCBvbmNlIHBlciB0b2tlbml6ZXIgYW5kIHJldXNlZCBhY3Jvc3Mgc2NoZW1hcy4gQ29uc3RydWN0IGZyb20gYSByZWFsCiAgICB0b2tlbml6ZXIgd2l0aCA6bWV0aDpgZnJvbV9oZmAsIG9yIGZyb20gZXhwbGljaXQgcGFpcnMgd2l0aCA6bWV0aDpgZnJvbV9wYWlyc2AKICAgIGZvciB0ZXN0cyB0aGF0IG5lZWQgbm8gbW9kZWwuIiIiCgogICAgcGllY2VzOiBEaWN0W2ludCwgc3RyXQogICAgZW9zX2lkOiBpbnQKICAgIHNwZWNpYWxfaWRzOiBTZXRbaW50XSA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1zZXQpCgogICAgQGNsYXNzbWV0aG9kCiAgICBkZWYgZnJvbV9wYWlycyhjbHMsIHBhaXJzOiBJdGVyYWJsZVtUdXBsZVtpbnQsIHN0cl1dLCBlb3NfaWQ6IGludCkgLT4gIlRva2VuVm9jYWIiOgogICAgICAgIHJldHVybiBjbHMocGllY2VzPXt0aWQ6IHR4dCBmb3IgdGlkLCB0eHQgaW4gcGFpcnN9LCBlb3NfaWQ9ZW9zX2lkKQoKICAgIEBjbGFzc21ldGhvZAogICAgZGVmIGZyb21faGYoY2xzLCB0b2tlbml6ZXIpIC0+ICJUb2tlblZvY2FiIjoKICAgICAgICAiIiJEZWNvZGUgZXZlcnkgdm9jYWJ1bGFyeSB0b2tlbiB0byBpdHMgdGV4dC4KCiAgICAgICAgU3BlY2lhbC9hZGRlZCB0b2tlbnMgKG90aGVyIHRoYW4gRU9TKSBhcmUgZXhjbHVkZWQgc28gdGhlIG1vZGVsIGNhbgogICAgICAgIG5ldmVyIHNhdGlzZnkgdGhlIGdyYW1tYXIgYnkgZW1pdHRpbmcgYSBjb250cm9sIHRva2VuLiBVbmRlY29kYWJsZQogICAgICAgIHBhcnRpYWwtYnl0ZSB0b2tlbnMgYXJlIGV4Y2x1ZGVkIHRvbyAoc2VlIGBgX1JFUExBQ0VNRU5UYGApLiIiIgogICAgICAgIGVvc19pZCA9IHRva2VuaXplci5lb3NfdG9rZW5faWQKICAgICAgICBzcGVjaWFsID0gc2V0KGdldGF0dHIodG9rZW5pemVyLCAiYWxsX3NwZWNpYWxfaWRzIiwgW10pIG9yIFtdKQogICAgICAgIHBpZWNlczogRGljdFtpbnQsIHN0cl0gPSB7fQogICAgICAgIHZvY2FiX3NpemUgPSBsZW4odG9rZW5pemVyKQogICAgICAgIGZvciB0aWQgaW4gcmFuZ2Uodm9jYWJfc2l6ZSk6CiAgICAgICAgICAgIGlmIHRpZCBpbiBzcGVjaWFsIGFuZCB0aWQgIT0gZW9zX2lkOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgdGV4dCA9IHRva2VuaXplci5kZWNvZGUoW3RpZF0sIGNsZWFuX3VwX3Rva2VuaXphdGlvbl9zcGFjZXM9RmFsc2UpCiAgICAgICAgICAgIGlmIHRleHQgPT0gIiIgb3IgX1JFUExBQ0VNRU5UIGluIHRleHQ6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBwaWVjZXNbdGlkXSA9IHRleHQKICAgICAgICBwaWVjZXMucG9wKGVvc19pZCwgTm9uZSkgICMgRU9TIGhhbmRsZWQgc3RydWN0dXJhbGx5LCBub3QgYXMgYSBwaWVjZQogICAgICAgIHJldHVybiBjbHMocGllY2VzPXBpZWNlcywgZW9zX2lkPWVvc19pZCwgc3BlY2lhbF9pZHM9c3BlY2lhbCkKCgpkZWYgY29uc3VtZShhdXRvbWF0b24sIHN0YXRlLCB0ZXh0OiBzdHIpOgogICAgIiIiRmVlZCBldmVyeSBjaGFyYWN0ZXIgb2YgYGB0ZXh0YGAgZnJvbSBgYHN0YXRlYGAuIFJldHVybnMgdGhlIHJlc3VsdGluZwogICAgc3RhdGUsIHdoaWNoIG1heSBiZSB0aGUgYXV0b21hdG9uJ3MgZGVhZCBzZW50aW5lbCBpZiB0aGUgdGV4dCBpcyByZWplY3RlZC4iIiIKICAgIGZvciBjaCBpbiB0ZXh0OgogICAgICAgIHN0YXRlID0gYXV0b21hdG9uLnN0ZXAoc3RhdGUsIGNoKQogICAgICAgIGlmIGF1dG9tYXRvbi5pc19kZWFkKHN0YXRlKToKICAgICAgICAgICAgcmV0dXJuIHN0YXRlCiAgICByZXR1cm4gc3RhdGUKCgpkZWYgYWR2YW5jZShhdXRvbWF0b24sIHN0YXRlLCB0ZXh0OiBzdHIpOgogICAgIiIiQWR2YW5jZSB0aGUgYXV0b21hdG9uIGJ5IGEgY2hvc2VuIHRva2VuJ3MgdGV4dCAoYXNzdW1lZCBhbHJlYWR5IGxlZ2FsKS4iIiIKICAgIHJldHVybiBjb25zdW1lKGF1dG9tYXRvbiwgc3RhdGUsIHRleHQpCgoKZGVmIGFsbG93ZWRfdG9rZW5faWRzKGF1dG9tYXRvbiwgc3RhdGUsIHZvY2FiOiBUb2tlblZvY2FiKSAtPiBTZXRbaW50XToKICAgICIiIlNldCBvZiB0b2tlbiBpZHMgbGVnYWwgZnJvbSBgYHN0YXRlYGAuCgogICAgQSBub3JtYWwgdG9rZW4gaXMgbGVnYWwgaWZmIGl0cyB3aG9sZSBkZWNvZGVkIHRleHQgaXMgY29uc3VtYWJsZSB3aXRob3V0CiAgICBkeWluZyBhbmQgbGFuZHMgaW4gYSAqbGl2ZSogc3RhdGUgKG9uZSBmcm9tIHdoaWNoIGFuIGFjY2VwdCBpcyBzdGlsbAogICAgcmVhY2hhYmxlKS4gVGhlIEVPUyB0b2tlbiBpcyBsZWdhbCBpZmYgYGBzdGF0ZWBgIGlzIGFjY2VwdGluZyDigJQgaS5lLiB0aGUKICAgIG91dHB1dCBpcyBhbHJlYWR5IGNvbXBsZXRlLCBzbyBzdG9wcGluZyBpcyB2YWxpZC4gVGhlIHJldHVybmVkIHNldCBpcyBuZXZlcgogICAgZW1wdHkgZm9yIGEgcmVhY2hhYmxlIHN0YXRlIHVubGVzcyB0aGUgYXV0b21hdG9uIGlzIGdlbnVpbmVseSBzdHVjaywgd2hpY2gKICAgIHRoZSBtYXNrZXIgdHJlYXRzIGFzIGFuIGVycm9yIHRvIHN1cmZhY2UgcmF0aGVyIHRoYW4gaGlkZS4iIiIKICAgIGFsbG93ZWQ6IFNldFtpbnRdID0gc2V0KCkKICAgIGZvciB0aWQsIHBpZWNlIGluIHZvY2FiLnBpZWNlcy5pdGVtcygpOgogICAgICAgIGVuZCA9IGNvbnN1bWUoYXV0b21hdG9uLCBzdGF0ZSwgcGllY2UpCiAgICAgICAgaWYgbm90IGF1dG9tYXRvbi5pc19kZWFkKGVuZCkgYW5kIGF1dG9tYXRvbi5pc19saXZlKGVuZCk6CiAgICAgICAgICAgIGFsbG93ZWQuYWRkKHRpZCkKICAgIGlmIGF1dG9tYXRvbi5pc19hY2NlcHRpbmcoc3RhdGUpOgogICAgICAgIGFsbG93ZWQuYWRkKHZvY2FiLmVvc19pZCkKICAgIHJldHVybiBhbGxvd2VkCg==",
"engine/masker.py": "IiIiVGhlIExvZ2l0c1Byb2Nlc3NvciB0aGF0IGVuZm9yY2VzIHRoZSBncmFtbWFyIGF0IGRlY29kZSB0aW1lLgoKT24gZXZlcnkgc3RlcCBpdCAoMSkgYWR2YW5jZXMgdGhlIGF1dG9tYXRvbiBieSB3aGF0ZXZlciB0b2tlbihzKSB0aGUgbW9kZWwganVzdApwcm9kdWNlZCwgKDIpIGRlcml2ZXMgdGhlIGxlZ2FsIHRva2VuIHNldCBmcm9tIHRoZSBjdXJyZW50IHN0YXRlIHZpYSB0aGUgdG9rZW4KYnJpZGdlLCBhbmQgKDMpIHNldHMgZXZlcnkgaWxsZWdhbCB0b2tlbidzIGxvZ2l0IHRvIC1pbmYgc28gc2FtcGxpbmcgY2FuIG9ubHkKcGljayBhIGxlZ2FsIG9uZS4gT3V0cHV0IGNhbiB0aGVyZWZvcmUgbmV2ZXIgbGVhdmUgdGhlIGdyYW1tYXIuCgpJbnZhcmlhbnRzIChmcm9tIHRoZSBndWlkZSdzIGdvdGNoYXMpOgogICogTmV2ZXIgbWFzayAqZXZlcnkqIHRva2VuIHRvIC1pbmYuIFdoZW4gdGhlIHN0YXRlIGlzIGFjY2VwdGluZywgRU9TIGlzIGFsd2F5cwogICAgbGVnYWwsIHNvIHRoZSBtb2RlbCBjYW4gc3RvcC4gQSBzdGF0ZSB3aXRoIG5vIGxlZ2FsIGNvbnRpbnVhdGlvbiBhbmQgbm8gbGVnYWwKICAgIHN0b3AgaXMgYW4gYXV0b21hdG9uIGJ1Zywgc3VyZmFjZWQgYXMgYW4gZXJyb3IgcmF0aGVyIHRoYW4gc2lsZW50bHkgaHVuZy4KICAqIEVPUyBpcyBsZWdhbCBleGFjdGx5IHdoZW4gdGhlIGF1dG9tYXRvbiBpcyBpbiBhbiBhY2NlcHRpbmcgc3RhdGUuCgpCYXRjaGluZzogdGhlIGJlbmNobWFyayBoYXJuZXNzIGRlY29kZXMgb25lIHNlcXVlbmNlIGF0IGEgdGltZSAoYmF0Y2ggc2l6ZSAxKSwKd2hpY2ggaXMgd2hhdCB0aGlzIHByb2Nlc3NvciBzdXBwb3J0cy4gTXVsdGktYmVhbS9tdWx0aS1zZXF1ZW5jZSBjb25zdHJhaW5lZApkZWNvZGluZyBuZWVkcyBwZXItYmVhbSBzdGF0ZSBhbmQgaXMgb3V0IG9mIHNjb3BlIGZvciB0aGUgaGFybmVzcy4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIHR5cGluZyBpbXBvcnQgT3B0aW9uYWwKCmltcG9ydCB0b3JjaApmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgTG9naXRzUHJvY2Vzc29yCgpmcm9tIGVuZ2luZS50b2tlbl9icmlkZ2UgaW1wb3J0IFRva2VuVm9jYWIsIGFkdmFuY2UsIGFsbG93ZWRfdG9rZW5faWRzCgoKY2xhc3MgQXV0b21hdG9uTG9naXRzUHJvY2Vzc29yKExvZ2l0c1Byb2Nlc3Nvcik6CiAgICAiIiJDb25zdHJhaW4gZ2VuZXJhdGlvbiB0byBhbiBhdXRvbWF0b24uIFdvcmtzIHdpdGggYW55IGF1dG9tYXRvbiBleHBvc2luZwogICAgYGBzdGFydGBgLCBgYHN0ZXBgYCwgYGBpc19kZWFkYGAsIGBgaXNfbGl2ZWBgLCBgYGlzX2FjY2VwdGluZ2BgICh0aGUgREZBIGFuZAogICAgdGhlIHB1c2hkb3duIGF1dG9tYXRvbiBib3RoIGRvKS4iIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgYXV0b21hdG9uLCB2b2NhYjogVG9rZW5Wb2NhYiwgYWxsb3dlZF9mbj1hbGxvd2VkX3Rva2VuX2lkcyk6CiAgICAgICAgc2VsZi5hdXRvbWF0b24gPSBhdXRvbWF0b24KICAgICAgICBzZWxmLnZvY2FiID0gdm9jYWIKICAgICAgICBzZWxmLl9hbGxvd2VkX2ZuID0gYWxsb3dlZF9mbgogICAgICAgIHNlbGYucmVzZXQoKQoKICAgIGRlZiByZXNldChzZWxmKSAtPiBOb25lOgogICAgICAgICIiIlJlc2V0IHRvIHRoZSBzdGFydCBzdGF0ZS4gQ2FsbCBiZXR3ZWVuIGluZGVwZW5kZW50IGdlbmVyYXRpb25zLiIiIgogICAgICAgIHNlbGYuc3RhdGUgPSBzZWxmLmF1dG9tYXRvbi5zdGFydAogICAgICAgIHNlbGYuX3Byb21wdF9sZW46IE9wdGlvbmFsW2ludF0gPSBOb25lCiAgICAgICAgc2VsZi5fY29uc3VtZWQgPSAwCgogICAgIyAtLSBzdGF0ZSB0cmFja2luZyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiAgICBkZWYgX3N5bmNfc3RhdGUoc2VsZiwgc2VxOiB0b3JjaC5UZW5zb3IpIC0+IE5vbmU6CiAgICAgICAgIiIiQWR2YW5jZSB0aGUgYXV0b21hdG9uIGJ5IGFueSB0b2tlbnMgZ2VuZXJhdGVkIHNpbmNlIHRoZSBsYXN0IGNhbGwuIiIiCiAgICAgICAgaWYgc2VsZi5fcHJvbXB0X2xlbiBpcyBOb25lOgogICAgICAgICAgICAjIEZpcnN0IGNhbGw6IGV2ZXJ5dGhpbmcgcHJlc2VudCBpcyB0aGUgcHJvbXB0LCBub3RoaW5nIGdlbmVyYXRlZC4KICAgICAgICAgICAgc2VsZi5fcHJvbXB0X2xlbiA9IGludChzZXEuc2hhcGVbMF0pCiAgICAgICAgICAgIHJldHVybgogICAgICAgIG5ld19zdGFydCA9IHNlbGYuX3Byb21wdF9sZW4gKyBzZWxmLl9jb25zdW1lZAogICAgICAgIGZvciB0aWQgaW4gc2VxW25ld19zdGFydDpdLnRvbGlzdCgpOgogICAgICAgICAgICBpZiB0aWQgPT0gc2VsZi52b2NhYi5lb3NfaWQ6CiAgICAgICAgICAgICAgICBjb250aW51ZSAgIyBFT1MgZW5kcyBnZW5lcmF0aW9uOyBpdCBjb25zdW1lcyBubyBncmFtbWFyIGNoYXJhY3RlcnMKICAgICAgICAgICAgcGllY2UgPSBzZWxmLnZvY2FiLnBpZWNlcy5nZXQodGlkLCAiIikKICAgICAgICAgICAgaWYgcGllY2U6CiAgICAgICAgICAgICAgICBzZWxmLnN0YXRlID0gYWR2YW5jZShzZWxmLmF1dG9tYXRvbiwgc2VsZi5zdGF0ZSwgcGllY2UpCiAgICAgICAgICAgIHNlbGYuX2NvbnN1bWVkICs9IDEKCiAgICAjIC0tIHRoZSBob29rIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KICAgIGRlZiBfX2NhbGxfXyhzZWxmLCBpbnB1dF9pZHM6IHRvcmNoLkxvbmdUZW5zb3IsIHNjb3JlczogdG9yY2guRmxvYXRUZW5zb3IpIC0+IHRvcmNoLkZsb2F0VGVuc29yOgogICAgICAgIHNlcSA9IGlucHV0X2lkc1swXQogICAgICAgIHNlbGYuX3N5bmNfc3RhdGUoc2VxKQoKICAgICAgICBhbGxvd2VkID0gc2VsZi5fYWxsb3dlZF9mbihzZWxmLmF1dG9tYXRvbiwgc2VsZi5zdGF0ZSwgc2VsZi52b2NhYikKICAgICAgICBpZiBub3QgYWxsb3dlZDoKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKAogICAgICAgICAgICAgICAgImNvbnN0cmFpbmVkIGRlY29kaW5nIHJlYWNoZWQgYSBzdGF0ZSB3aXRoIG5vIGxlZ2FsIHRva2VuIGFuZCBubyAiCiAgICAgICAgICAgICAgICAibGVnYWwgc3RvcCAoc3RhdGU9JXIpLiBUaGlzIGlzIGFuIGF1dG9tYXRvbi9icmlkZ2UgYnVnLCBub3QgYW4gIgogICAgICAgICAgICAgICAgImFjY2VwdGFibGUgb3V0cHV0LiIgJSAoc2VsZi5zdGF0ZSwpCiAgICAgICAgICAgICkKCiAgICAgICAgaWR4ID0gdG9yY2gudGVuc29yKHNvcnRlZChhbGxvd2VkKSwgZGV2aWNlPXNjb3Jlcy5kZXZpY2UsIGR0eXBlPXRvcmNoLmxvbmcpCiAgICAgICAgbWFza2VkID0gdG9yY2guZnVsbF9saWtlKHNjb3JlcywgZmxvYXQoIi1pbmYiKSkKICAgICAgICBtYXNrZWRbOiwgaWR4XSA9IHNjb3Jlc1s6LCBpZHhdCiAgICAgICAgcmV0dXJuIG1hc2tlZAoKCmRlZiBidWlsZF9yZWdleF9wcm9jZXNzb3IoCiAgICBwYXR0ZXJuOiBzdHIsIHZvY2FiOiBUb2tlblZvY2FiLCBpbmRleGVkOiBib29sID0gRmFsc2UsIHByZWNvbXB1dGU6IGJvb2wgPSBGYWxzZQopIC0+IEF1dG9tYXRvbkxvZ2l0c1Byb2Nlc3NvcjoKICAgICIiIkNvbnZlbmllbmNlOiBjb21waWxlIGBgcGF0dGVybmBgIGFuZCB3cmFwIGl0IGluIGEgcHJvY2Vzc29yLgoKICAgIGBgaW5kZXhlZGBgIGFkZHMgdGhlIFBoYXNlLTMgcGVyLXN0YXRlIHRva2VuIGNhY2hlOyBgYHByZWNvbXB1dGVgYCBmaWxscyBpdAogICAgZWFnZXJseSB1cCBmcm9udCAoY2hlYXAgZm9yIGEgREZBKS4iIiIKICAgIGZyb20gZW5naW5lLnJlZ2V4X2NvbXBpbGVyIGltcG9ydCBjb21waWxlX3JlZ2V4CgogICAgYXV0b21hdG9uID0gY29tcGlsZV9yZWdleChwYXR0ZXJuKQogICAgYWxsb3dlZF9mbiA9IGFsbG93ZWRfdG9rZW5faWRzCiAgICBpZiBpbmRleGVkOgogICAgICAgIGZyb20gZW5naW5lLnRva2VuX2luZGV4IGltcG9ydCBUb2tlbkluZGV4CgogICAgICAgIGlkeCA9IFRva2VuSW5kZXgoYXV0b21hdG9uLCB2b2NhYikKICAgICAgICBpZiBwcmVjb21wdXRlOgogICAgICAgICAgICBpZHgucHJlY29tcHV0ZSgpCiAgICAgICAgYWxsb3dlZF9mbiA9IGlkeAogICAgcmV0dXJuIEF1dG9tYXRvbkxvZ2l0c1Byb2Nlc3NvcihhdXRvbWF0b24sIHZvY2FiLCBhbGxvd2VkX2ZuPWFsbG93ZWRfZm4pCg==",
"engine/pushdown.py": "IiIiUHVzaGRvd24gYXV0b21hdG9uIChQaGFzZSAyKSBidWlsdCBhcyBhIFJlY3Vyc2l2ZSBUcmFuc2l0aW9uIE5ldHdvcmsuCgpBIGZpeGVkIEpTT04gc2NoZW1hIGhhcyAqYm91bmRlZCogbmVzdGluZywgc28gaXQgaXMgdGVjaG5pY2FsbHkgcmVndWxhciDigJQgYnV0CmNvbXBpbGluZyBkZWVwIG5lc3RpbmcgaW50byBvbmUgZmxhdCBERkEgZXhwbG9kZXMsIGFuZCBpdCB3b3VsZCBub3QgZ2VuZXJhbGlzZQp0byB0aGUgcmVjdXJzaXZlIEVCTkYgZ3JhbW1hcnMgb2YgUGhhc2UgMy4gU28gd2UgdXNlIGEgcmVhbCBzdGFjayBtYWNoaW5lOiB0aGUKZ3JhbW1hciBpcyBhIHNldCBvZiBuYW1lZCBydWxlcywgYSBgYENhbGxgYCBlZGdlIHB1c2hlcyBhIHJldHVybiBwb2ludCBhbmQganVtcHMKaW50byBhIHN1Yi1ydWxlLCBhbmQgcmVhY2hpbmcgYSBydWxlJ3MgZXhpdCBwb3BzIGJhY2suIFRoZSBzdGFjayBpcyBleGFjdGx5IHRoZQoibmVzdGVkIGJyYWNlcy9icmFja2V0cyIgZGVwdGggdGhlIGd1aWRlIGRlc2NyaWJlcy4KCkNydWNpYWxseSwgbmVzdGluZyBpcyBkcml2ZW4gYnkgKmdyYW1tYXIgc3RydWN0dXJlKiwgbm90IGJ5IHNjYW5uaW5nIGZvciBicmFjZQpjaGFyYWN0ZXJzIOKAlCBzbyBhIGBge2BgIGluc2lkZSBhIHN0cmluZyBsaXRlcmFsIGlzIGp1c3QgYW4gb3JkaW5hcnkgY2hhcmFjdGVyIGluCnRoZSBTVFJJTkcgcnVsZSBhbmQgbmV2ZXIgdG91Y2hlcyB0aGUgc3RhY2suIFRoYXQgaXMgd2hhdCBtYWtlcyB0aGUgaGFyZAoidG9rZW4gc3RyYWRkbGluZyBhIHN0cnVjdHVyYWwgYm91bmRhcnkiIGNhc2UgZmFsbCBvdXQgY29ycmVjdGx5LgoKVGhlIGF1dG9tYXRvbiBleHBvc2VzIHRoZSBzYW1lIHN1cmZhY2UgYXMgdGhlIFBoYXNlLTEgREZBIChgYHN0YXJ0YGAsIGBgc3RlcGBgLApgYGlzX2RlYWRgYCwgYGBpc19saXZlYGAsIGBgaXNfYWNjZXB0aW5nYGApLCBzbyA6bW9kOmBlbmdpbmUudG9rZW5fYnJpZGdlYCBhbmQKOm1vZDpgZW5naW5lLm1hc2tlcmAgZHJpdmUgaXQgdW5jaGFuZ2VkLiBBICJzdGF0ZSIgaGVyZSBpcyBhIGZyb3plbnNldCBvZgpgYChub2RlLCBzdGFjaylgYCBjb25maWd1cmF0aW9ucyDigJQgdGhlIG5vbmRldGVybWluaXN0aWMgUERBIHNpbXVsYXRlZCBhcyBhIHNldCwKdGhlIHB1c2hkb3duIGFuYWxvZ3VlIG9mIHN1YnNldCBjb25zdHJ1Y3Rpb24uCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBjb2xsZWN0aW9ucyBpbXBvcnQgZGVmYXVsdGRpY3QKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBmaWVsZApmcm9tIHR5cGluZyBpbXBvcnQgRGljdCwgRnJvemVuU2V0LCBMaXN0LCBPcHRpb25hbCwgU2V0LCBUdXBsZQoKZnJvbSBlbmdpbmUuYXV0b21hdG9uIGltcG9ydCBPVEhFUgoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tICMKIyBHcmFtbWFyIGNvbWJpbmF0b3JzCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tICMKQGRhdGFjbGFzcwpjbGFzcyBDaGFyczoKICAgICIiIkNvbnN1bWUgb25lIGNoYXJhY3RlciBkZXNjcmliZWQgYnkgYGBzcGVjYGAuCgogICAgc3BlYyBpcyBvbmUgb2Y6ICgibGl0IiwgZnJvemVuc2V0KSwgKCJzZXQiLCBmcm96ZW5zZXQpLCAoImFueSIsKSwKICAgICgiYW55YnV0IiwgZnJvemVuc2V0KS4iIiIKCiAgICBzcGVjOiB0dXBsZQoKCkBkYXRhY2xhc3MKY2xhc3MgTGl0OgogICAgIiIiTWF0Y2ggYW4gZXhhY3Qgc3RyaW5nIChhIHNlcXVlbmNlIG9mIHNpbmdsZS1jaGFyYWN0ZXIgbWF0Y2hlcykuIiIiCgogICAgdGV4dDogc3RyCgoKQGRhdGFjbGFzcwpjbGFzcyBTZXE6CiAgICBwYXJ0czogbGlzdAoKCkBkYXRhY2xhc3MKY2xhc3MgQWx0OgogICAgb3B0aW9uczogbGlzdAoKCkBkYXRhY2xhc3MKY2xhc3MgT3B0OgogICAgbm9kZTogb2JqZWN0CgoKQGRhdGFjbGFzcwpjbGFzcyBTdGFyOgogICAgbm9kZTogb2JqZWN0CgoKQGRhdGFjbGFzcwpjbGFzcyBQbHVzOgogICAgbm9kZTogb2JqZWN0CgoKQGRhdGFjbGFzcwpjbGFzcyBFbXB0eToKICAgIHBhc3MKCgpAZGF0YWNsYXNzCmNsYXNzIENhbGw6CiAgICAiIiJFbnRlciBhIG5hbWVkIHJ1bGUsIHB1c2hpbmcgYSByZXR1cm4gcG9pbnQgKHRoZSBzdGFjay11c2luZyBlZGdlKS4iIiIKCiAgICBydWxlOiBzdHIKCgpAZGF0YWNsYXNzCmNsYXNzIEdyYW1tYXI6CiAgICBydWxlczogRGljdFtzdHIsIG9iamVjdF0KICAgIHN0YXJ0OiBzdHIKCgojIENvbnZlbmllbmNlIGNvbnN0cnVjdG9ycyBmb3IgY29tbW9uIGNoYXJhY3RlciBtYXRjaGVycy4KZGVmIGxpdF9jaGFyKGM6IHN0cikgLT4gQ2hhcnM6CiAgICByZXR1cm4gQ2hhcnMoKCJsaXQiLCBmcm96ZW5zZXQoW2NdKSkpCgoKZGVmIGNoYXJfc2V0KGNoYXJzKSAtPiBDaGFyczoKICAgIHJldHVybiBDaGFycygoInNldCIsIGZyb3plbnNldChjaGFycykpKQoKCmRlZiBhbnlfY2hhcigpIC0+IENoYXJzOgogICAgcmV0dXJuIENoYXJzKCgiYW55IiwpKQoKCmRlZiBhbnlfYnV0KGNoYXJzKSAtPiBDaGFyczoKICAgIHJldHVybiBDaGFycygoImFueWJ1dCIsIGZyb3plbnNldChjaGFycykpKQoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tICMKIyBDb21waWxhdGlvbjogY29tYmluYXRvcnMgLT4gUlROIGVkZ2VzCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tICMKIyBFZGdlIGVuY29kaW5ncyAoc3RvcmVkIHBlciBzb3VyY2Ugc3RhdGUpOgojICAgKCJjaGFyIiwgc3BlY19vcl9mcm96ZW5zZXQsIHRhcmdldCkgIGNvbnN1bWUgb25lIGNoYXIKIyAgICgiZXBzIiwgdGFyZ2V0KSAgICAgICAgICAgICAgICAgICAgICBlcHNpbG9uIG1vdmUKIyAgICgiY2FsbCIsIGNhbGxlZV9lbnRyeSwgcmV0dXJuX3N0YXRlKSBwdXNoIHJldHVybl9zdGF0ZSwgZW50ZXIgY2FsbGVlCmNsYXNzIF9CdWlsZGVyOgogICAgZGVmIF9faW5pdF9fKHNlbGYpOgogICAgICAgIHNlbGYuZWRnZXM6IERpY3RbaW50LCBsaXN0XSA9IGRlZmF1bHRkaWN0KGxpc3QpCiAgICAgICAgc2VsZi5fbiA9IDAKICAgICAgICBzZWxmLmFscGhhYmV0OiBTZXRbc3RyXSA9IHNldCgpCiAgICAgICAgc2VsZi5yZXR1cm5zOiBTZXRbaW50XSA9IHNldCgpCiAgICAgICAgc2VsZi5ydWxlX2VudHJ5OiBEaWN0W3N0ciwgaW50XSA9IHt9CiAgICAgICAgc2VsZi5ydWxlX2V4aXQ6IERpY3Rbc3RyLCBpbnRdID0ge30KICAgICAgICBzZWxmLl9jaGFyX2VkZ2VzOiBMaXN0W1R1cGxlW2ludCwgaW50XV0gPSBbXSAgIyAoc3RhdGUsIGluZGV4KSB0byByZXNvbHZlCgogICAgZGVmIG5ld19zdGF0ZShzZWxmKSAtPiBpbnQ6CiAgICAgICAgcyA9IHNlbGYuX24KICAgICAgICBzZWxmLl9uICs9IDEKICAgICAgICByZXR1cm4gcwoKICAgIGRlZiBfbm90ZShzZWxmLCBjaGFycykgLT4gTm9uZToKICAgICAgICBzZWxmLmFscGhhYmV0LnVwZGF0ZShjaGFycykKCiAgICBkZWYgYWRkX2NoYXIoc2VsZiwgc3JjOiBpbnQsIHNwZWM6IHR1cGxlLCBkc3Q6IGludCkgLT4gTm9uZToKICAgICAgICAjIFJlY29yZCBjb25jcmV0ZSBjaGFyYWN0ZXJzIHNvIHRoZSBnbG9iYWwgYWxwaGFiZXQgaXMgY29tcGxldGUgYmVmb3JlCiAgICAgICAgIyAiYW55Ii8iYW55YnV0IiBzcGVjcyBhcmUgcmVzb2x2ZWQgYWdhaW5zdCBpdC4KICAgICAgICBraW5kID0gc3BlY1swXQogICAgICAgIGlmIGtpbmQgaW4gKCJsaXQiLCAic2V0Iik6CiAgICAgICAgICAgIHNlbGYuX25vdGUoc3BlY1sxXSkKICAgICAgICBlbGlmIGtpbmQgPT0gImFueWJ1dCI6CiAgICAgICAgICAgIHNlbGYuX25vdGUoc3BlY1sxXSkKICAgICAgICBzZWxmLmVkZ2VzW3NyY10uYXBwZW5kKFsiY2hhciIsIHNwZWMsIGRzdF0pCiAgICAgICAgc2VsZi5fY2hhcl9lZGdlcy5hcHBlbmQoKHNyYywgbGVuKHNlbGYuZWRnZXNbc3JjXSkgLSAxKSkKCiAgICBkZWYgYWRkX2VwcyhzZWxmLCBzcmM6IGludCwgZHN0OiBpbnQpIC0+IE5vbmU6CiAgICAgICAgc2VsZi5lZGdlc1tzcmNdLmFwcGVuZCgoImVwcyIsIGRzdCkpCgogICAgZGVmIGFkZF9jYWxsKHNlbGYsIHNyYzogaW50LCBlbnRyeTogaW50LCByZXQ6IGludCkgLT4gTm9uZToKICAgICAgICBzZWxmLmVkZ2VzW3NyY10uYXBwZW5kKCgiY2FsbCIsIGVudHJ5LCByZXQpKQoKICAgICMgLS0gY29tcGlsZSBhIGNvbWJpbmF0b3Igbm9kZSB0byAoZW50cnksIGV4aXQpIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiAgICBkZWYgY29tcGlsZShzZWxmLCBub2RlKSAtPiBUdXBsZVtpbnQsIGludF06CiAgICAgICAgaWYgaXNpbnN0YW5jZShub2RlLCBFbXB0eSk6CiAgICAgICAgICAgIGUsIHggPSBzZWxmLm5ld19zdGF0ZSgpLCBzZWxmLm5ld19zdGF0ZSgpCiAgICAgICAgICAgIHNlbGYuYWRkX2VwcyhlLCB4KQogICAgICAgICAgICByZXR1cm4gZSwgeAogICAgICAgIGlmIGlzaW5zdGFuY2Uobm9kZSwgQ2hhcnMpOgogICAgICAgICAgICBlLCB4ID0gc2VsZi5uZXdfc3RhdGUoKSwgc2VsZi5uZXdfc3RhdGUoKQogICAgICAgICAgICBzZWxmLmFkZF9jaGFyKGUsIG5vZGUuc3BlYywgeCkKICAgICAgICAgICAgcmV0dXJuIGUsIHgKICAgICAgICBpZiBpc2luc3RhbmNlKG5vZGUsIExpdCk6CiAgICAgICAgICAgIGlmIG5vZGUudGV4dCA9PSAiIjoKICAgICAgICAgICAgICAgIHJldHVybiBzZWxmLmNvbXBpbGUoRW1wdHkoKSkKICAgICAgICAgICAgZSA9IHNlbGYubmV3X3N0YXRlKCkKICAgICAgICAgICAgY3VyID0gZQogICAgICAgICAgICBmb3IgY2ggaW4gbm9kZS50ZXh0OgogICAgICAgICAgICAgICAgbnh0ID0gc2VsZi5uZXdfc3RhdGUoKQogICAgICAgICAgICAgICAgc2VsZi5hZGRfY2hhcihjdXIsICgibGl0IiwgZnJvemVuc2V0KFtjaF0pKSwgbnh0KQogICAgICAgICAgICAgICAgY3VyID0gbnh0CiAgICAgICAgICAgIHJldHVybiBlLCBjdXIKICAgICAgICBpZiBpc2luc3RhbmNlKG5vZGUsIFNlcSk6CiAgICAgICAgICAgIGlmIG5vdCBub2RlLnBhcnRzOgogICAgICAgICAgICAgICAgcmV0dXJuIHNlbGYuY29tcGlsZShFbXB0eSgpKQogICAgICAgICAgICBlbnRyeSA9IHByZXZfZXhpdCA9IE5vbmUKICAgICAgICAgICAgZm9yIHBhcnQgaW4gbm9kZS5wYXJ0czoKICAgICAgICAgICAgICAgIHBlLCBweCA9IHNlbGYuY29tcGlsZShwYXJ0KQogICAgICAgICAgICAgICAgaWYgZW50cnkgaXMgTm9uZToKICAgICAgICAgICAgICAgICAgICBlbnRyeSA9IHBlCiAgICAgICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgICAgIHNlbGYuYWRkX2VwcyhwcmV2X2V4aXQsIHBlKQogICAgICAgICAgICAgICAgcHJldl9leGl0ID0gcHgKICAgICAgICAgICAgcmV0dXJuIGVudHJ5LCBwcmV2X2V4aXQKICAgICAgICBpZiBpc2luc3RhbmNlKG5vZGUsIEFsdCk6CiAgICAgICAgICAgIGUsIHggPSBzZWxmLm5ld19zdGF0ZSgpLCBzZWxmLm5ld19zdGF0ZSgpCiAgICAgICAgICAgIGZvciBvcHQgaW4gbm9kZS5vcHRpb25zOgogICAgICAgICAgICAgICAgb2UsIG94ID0gc2VsZi5jb21waWxlKG9wdCkKICAgICAgICAgICAgICAgIHNlbGYuYWRkX2VwcyhlLCBvZSkKICAgICAgICAgICAgICAgIHNlbGYuYWRkX2VwcyhveCwgeCkKICAgICAgICAgICAgcmV0dXJuIGUsIHgKICAgICAgICBpZiBpc2luc3RhbmNlKG5vZGUsIE9wdCk6CiAgICAgICAgICAgIGUsIHggPSBzZWxmLm5ld19zdGF0ZSgpLCBzZWxmLm5ld19zdGF0ZSgpCiAgICAgICAgICAgIG5lLCBueCA9IHNlbGYuY29tcGlsZShub2RlLm5vZGUpCiAgICAgICAgICAgIHNlbGYuYWRkX2VwcyhlLCBuZSkKICAgICAgICAgICAgc2VsZi5hZGRfZXBzKG54LCB4KQogICAgICAgICAgICBzZWxmLmFkZF9lcHMoZSwgeCkgICMgc2tpcAogICAgICAgICAgICByZXR1cm4gZSwgeAogICAgICAgIGlmIGlzaW5zdGFuY2Uobm9kZSwgU3Rhcik6CiAgICAgICAgICAgIGUsIHggPSBzZWxmLm5ld19zdGF0ZSgpLCBzZWxmLm5ld19zdGF0ZSgpCiAgICAgICAgICAgIG5lLCBueCA9IHNlbGYuY29tcGlsZShub2RlLm5vZGUpCiAgICAgICAgICAgIHNlbGYuYWRkX2VwcyhlLCBuZSkKICAgICAgICAgICAgc2VsZi5hZGRfZXBzKGUsIHgpICAjIHplcm8gdGltZXMKICAgICAgICAgICAgc2VsZi5hZGRfZXBzKG54LCBuZSkgICMgbG9vcAogICAgICAgICAgICBzZWxmLmFkZF9lcHMobngsIHgpICAjIHN0b3AKICAgICAgICAgICAgcmV0dXJuIGUsIHgKICAgICAgICBpZiBpc2luc3RhbmNlKG5vZGUsIFBsdXMpOgogICAgICAgICAgICBlLCB4ID0gc2VsZi5uZXdfc3RhdGUoKSwgc2VsZi5uZXdfc3RhdGUoKQogICAgICAgICAgICBuZSwgbnggPSBzZWxmLmNvbXBpbGUobm9kZS5ub2RlKQogICAgICAgICAgICBzZWxmLmFkZF9lcHMoZSwgbmUpCiAgICAgICAgICAgIHNlbGYuYWRkX2VwcyhueCwgbmUpICAjIGxvb3AKICAgICAgICAgICAgc2VsZi5hZGRfZXBzKG54LCB4KSAgIyBzdG9wIChhdCBsZWFzdCBvbmNlKQogICAgICAgICAgICByZXR1cm4gZSwgeAogICAgICAgIGlmIGlzaW5zdGFuY2Uobm9kZSwgQ2FsbCk6CiAgICAgICAgICAgIGUsIHggPSBzZWxmLm5ld19zdGF0ZSgpLCBzZWxmLm5ld19zdGF0ZSgpCiAgICAgICAgICAgIHNlbGYuYWRkX2NhbGwoZSwgc2VsZi5ydWxlX2VudHJ5W25vZGUucnVsZV0sIHgpCiAgICAgICAgICAgIHJldHVybiBlLCB4CiAgICAgICAgcmFpc2UgVHlwZUVycm9yKGYidW5rbm93biBncmFtbWFyIG5vZGUge25vZGUhcn0iKQoKICAgIGRlZiByZXNvbHZlX3NwZWNzKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgIiIiVHVybiBjaGFyIHNwZWNzIGludG8gY29uY3JldGUgc3ltYm9sIGZyb3plbnNldHMgbm93IHRoZSBhbHBoYWJldCBpcwogICAgICAgIGZpbmFsIChgYGFueWBgL2BgYW55YnV0YGAgbmVlZCB0aGUgd2hvbGUgYWxwaGFiZXQgdG8gYmUga25vd24pLiIiIgogICAgICAgIGZvciBzdGF0ZSwgaWR4IGluIHNlbGYuX2NoYXJfZWRnZXM6CiAgICAgICAgICAgIGVkZ2UgPSBzZWxmLmVkZ2VzW3N0YXRlXVtpZHhdCiAgICAgICAgICAgIHNwZWMgPSBlZGdlWzFdCiAgICAgICAgICAgIGtpbmQgPSBzcGVjWzBdCiAgICAgICAgICAgIGlmIGtpbmQgPT0gImxpdCIgb3Iga2luZCA9PSAic2V0IjoKICAgICAgICAgICAgICAgIHN5bXMgPSBmcm96ZW5zZXQoc3BlY1sxXSkKICAgICAgICAgICAgZWxpZiBraW5kID09ICJhbnkiOgogICAgICAgICAgICAgICAgc3ltcyA9IGZyb3plbnNldChzZXQoc2VsZi5hbHBoYWJldCkgfCB7T1RIRVJ9KQogICAgICAgICAgICBlbGlmIGtpbmQgPT0gImFueWJ1dCI6CiAgICAgICAgICAgICAgICBzeW1zID0gZnJvemVuc2V0KChzZXQoc2VsZi5hbHBoYWJldCkgLSBzcGVjWzFdKSB8IHtPVEhFUn0pCiAgICAgICAgICAgIGVsc2U6ICAjIHByYWdtYTogbm8gY292ZXIKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJiYWQgY2hhciBzcGVjIHtzcGVjIXJ9IikKICAgICAgICAgICAgZWRnZVsxXSA9IHN5bXMKCgpkZWYgY29tcGlsZV9ncmFtbWFyKGdyYW1tYXI6IEdyYW1tYXIpIC0+ICJQREEiOgogICAgYiA9IF9CdWlsZGVyKCkKICAgICMgUHJlLXJlZ2lzdGVyIGVudHJ5L2V4aXQgc3RhdGVzIHNvIENhbGwgY2FuIHJlZmVyZW5jZSBydWxlcyBub3QgeWV0IGJ1aWx0LgogICAgZm9yIG5hbWUgaW4gZ3JhbW1hci5ydWxlczoKICAgICAgICBiLnJ1bGVfZW50cnlbbmFtZV0gPSBiLm5ld19zdGF0ZSgpCiAgICAgICAgYi5ydWxlX2V4aXRbbmFtZV0gPSBiLm5ld19zdGF0ZSgpCiAgICBmb3IgbmFtZSwgYm9keSBpbiBncmFtbWFyLnJ1bGVzLml0ZW1zKCk6CiAgICAgICAgYmUsIGJ4ID0gYi5jb21waWxlKGJvZHkpCiAgICAgICAgYi5hZGRfZXBzKGIucnVsZV9lbnRyeVtuYW1lXSwgYmUpCiAgICAgICAgYi5hZGRfZXBzKGJ4LCBiLnJ1bGVfZXhpdFtuYW1lXSkKICAgICAgICBiLnJldHVybnMuYWRkKGIucnVsZV9leGl0W25hbWVdKQogICAgc3RhcnQgPSBiLm5ld19zdGF0ZSgpCiAgICBhY2NlcHQgPSBiLm5ld19zdGF0ZSgpCiAgICBiLmFkZF9jYWxsKHN0YXJ0LCBiLnJ1bGVfZW50cnlbZ3JhbW1hci5zdGFydF0sIGFjY2VwdCkKICAgIGIucmVzb2x2ZV9zcGVjcygpCiAgICByZXR1cm4gUERBKAogICAgICAgIGVkZ2VzPWRpY3QoYi5lZGdlcyksCiAgICAgICAgc3RhcnRfc3RhdGU9c3RhcnQsCiAgICAgICAgYWNjZXB0X3N0YXRlPWFjY2VwdCwKICAgICAgICByZXR1cm5zPWIucmV0dXJucywKICAgICAgICBhbHBoYWJldD1zZXQoYi5hbHBoYWJldCksCiAgICApCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gIwojIFRoZSBhdXRvbWF0b24KIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gIwpDb25maWcgPSBUdXBsZVtpbnQsIFR1cGxlW2ludCwgLi4uXV0gICMgKG5vZGUsIHN0YWNrIG9mIHJldHVybiBzdGF0ZXMpCgpfTUFYX1NUQUNLID0gNDAwICAjIGd1YXJkIGFnYWluc3QgcGF0aG9sb2dpY2FsIHJlY3Vyc2lvbiBpbiBlcHNpbG9uLWNsb3N1cmUKCgpjbGFzcyBQREE6CiAgICAiIiJOb25kZXRlcm1pbmlzdGljIHB1c2hkb3duIGF1dG9tYXRvbiwgc2ltdWxhdGVkIGFzIGEgc2V0IG9mIGNvbmZpZ3VyYXRpb25zLgoKICAgIEEgInN0YXRlIiAod2hhdCB0aGUgYnJpZGdlIHBhc3NlcyBhcm91bmQpIGlzIGEgZnJvemVuc2V0IG9mIGBgKG5vZGUsIHN0YWNrKWBgCiAgICBjb25maWd1cmF0aW9ucywgYWxyZWFkeSBlcHNpbG9uLWNsb3NlZC4gRW1wdHkgc2V0ID09IGRlYWQuIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGVkZ2VzLCBzdGFydF9zdGF0ZSwgYWNjZXB0X3N0YXRlLCByZXR1cm5zLCBhbHBoYWJldCk6CiAgICAgICAgc2VsZi5lZGdlcyA9IGVkZ2VzCiAgICAgICAgc2VsZi5hY2NlcHRfc3RhdGUgPSBhY2NlcHRfc3RhdGUKICAgICAgICBzZWxmLnJldHVybnMgPSByZXR1cm5zCiAgICAgICAgc2VsZi5hbHBoYWJldCA9IHNldChhbHBoYWJldCkKICAgICAgICAjIE1lbW9pemUgKHN0YXRlLCBzeW1ib2wpIC0+IG5leHRfc3RhdGUuIHN0ZXAoKSBpcyBhIHB1cmUgZnVuY3Rpb24gb2YgdGhlCiAgICAgICAgIyAoYWxyZWFkeS1jbG9zZWQpIHN0YXRlIGFuZCB0aGUgc3ltYm9sLCBzbyBjYWNoaW5nIGlzIGV4YWN0LiBUaGlzIHR1cm5zCiAgICAgICAgIyB0aGUgYnJpZGdlJ3MgcGVyLXN0ZXAgTyh2b2NhYikgc2NhbiBmcm9tIH4xNTBrIGNsb3N1cmUgY29tcHV0YXRpb25zCiAgICAgICAgIyBpbnRvIH58ZGlzdGluY3QgY2hhcnN8IG9mIHRoZW0uIFBoYXNlIDMncyBwZXItc3RhdGUgdG9rZW4gaW5kZXggYnVpbGRzCiAgICAgICAgIyBvbiB0aGUgc2FtZSBpZGVhIGF0IHRoZSB3aG9sZS10b2tlbiBsZXZlbC4KICAgICAgICBzZWxmLl9zdGVwX2NhY2hlOiBEaWN0W1R1cGxlW0Zyb3plblNldFtDb25maWddLCBvYmplY3RdLCBGcm96ZW5TZXRbQ29uZmlnXV0gPSB7fQogICAgICAgIHNlbGYuc3RhcnQ6IEZyb3plblNldFtDb25maWddID0gc2VsZi5fY2xvc3VyZShbKHN0YXJ0X3N0YXRlLCAoKSldKQoKICAgICMgLS0gaW50ZXJmYWNlIGV4cGVjdGVkIGJ5IHRva2VuX2JyaWRnZSAvIG1hc2tlciAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiAgICBkZWYgX3N5bWJvbChzZWxmLCBjaDogc3RyKToKICAgICAgICByZXR1cm4gY2ggaWYgY2ggaW4gc2VsZi5hbHBoYWJldCBlbHNlIE9USEVSCgogICAgZGVmIHN0ZXAoc2VsZiwgc3RhdGU6IEZyb3plblNldFtDb25maWddLCBjaDogc3RyKSAtPiBGcm96ZW5TZXRbQ29uZmlnXToKICAgICAgICBzeW0gPSBzZWxmLl9zeW1ib2woY2gpCiAgICAgICAga2V5ID0gKHN0YXRlLCBzeW0pCiAgICAgICAgY2FjaGVkID0gc2VsZi5fc3RlcF9jYWNoZS5nZXQoa2V5KQogICAgICAgIGlmIGNhY2hlZCBpcyBub3QgTm9uZToKICAgICAgICAgICAgcmV0dXJuIGNhY2hlZAogICAgICAgIG54dDogU2V0W0NvbmZpZ10gPSBzZXQoKQogICAgICAgIGZvciAobm9kZSwgc3RhY2spIGluIHN0YXRlOgogICAgICAgICAgICBmb3IgZWRnZSBpbiBzZWxmLmVkZ2VzLmdldChub2RlLCAoKSk6CiAgICAgICAgICAgICAgICBpZiBlZGdlWzBdID09ICJjaGFyIiBhbmQgc3ltIGluIGVkZ2VbMV06CiAgICAgICAgICAgICAgICAgICAgbnh0LmFkZCgoZWRnZVsyXSwgc3RhY2spKQogICAgICAgIHJlc3VsdCA9IHNlbGYuX2Nsb3N1cmUobnh0KQogICAgICAgIHNlbGYuX3N0ZXBfY2FjaGVba2V5XSA9IHJlc3VsdAogICAgICAgIHJldHVybiByZXN1bHQKCiAgICBkZWYgaXNfZGVhZChzZWxmLCBzdGF0ZTogRnJvemVuU2V0W0NvbmZpZ10pIC0+IGJvb2w6CiAgICAgICAgcmV0dXJuIGxlbihzdGF0ZSkgPT0gMAoKICAgIGRlZiBpc19saXZlKHNlbGYsIHN0YXRlOiBGcm96ZW5TZXRbQ29uZmlnXSkgLT4gYm9vbDoKICAgICAgICAjIEV2ZXJ5IHJlYWNoYWJsZSBjb25maWd1cmF0aW9uIGluIHRoaXMgZ3JhbW1hciBjYW4gYmUgY29tcGxldGVkIHRvCiAgICAgICAgIyBhY2NlcHRhbmNlICh0aGUgZ3JhbW1hciBoYXMgbm8gZGVhZC1lbmQgc3RhdGVzKSwgc28gYSBub24tZW1wdHkgc2V0IGlzCiAgICAgICAgIyBsaXZlLiBUZXJtaW5hdGlvbiBvZiBuZXN0ZWQgc3RydWN0dXJlIGlzIGNvdmVyZWQgYnkgdGVzdHMuCiAgICAgICAgcmV0dXJuIGxlbihzdGF0ZSkgPiAwCgogICAgZGVmIGlzX2FjY2VwdGluZyhzZWxmLCBzdGF0ZTogRnJvemVuU2V0W0NvbmZpZ10pIC0+IGJvb2w6CiAgICAgICAgcmV0dXJuIChzZWxmLmFjY2VwdF9zdGF0ZSwgKCkpIGluIHN0YXRlCgogICAgZGVmIHJ1bihzZWxmLCBzdGF0ZTogRnJvemVuU2V0W0NvbmZpZ10sIHRleHQ6IHN0cikgLT4gRnJvemVuU2V0W0NvbmZpZ106CiAgICAgICAgZm9yIGNoIGluIHRleHQ6CiAgICAgICAgICAgIHN0YXRlID0gc2VsZi5zdGVwKHN0YXRlLCBjaCkKICAgICAgICAgICAgaWYgc2VsZi5pc19kZWFkKHN0YXRlKToKICAgICAgICAgICAgICAgIHJldHVybiBzdGF0ZQogICAgICAgIHJldHVybiBzdGF0ZQoKICAgIGRlZiBhY2NlcHRzKHNlbGYsIHRleHQ6IHN0cikgLT4gYm9vbDoKICAgICAgICAiIiJUcnVlIGlmIGBgdGV4dGBgIGlzIGEgY29tcGxldGUgc2VudGVuY2Ugb2YgdGhlIGdyYW1tYXIuIiIiCiAgICAgICAgcmV0dXJuIHNlbGYuaXNfYWNjZXB0aW5nKHNlbGYucnVuKHNlbGYuc3RhcnQsIHRleHQpKQoKICAgICMgLS0gZXBzaWxvbiAvIGNhbGwgLyByZXR1cm4gY2xvc3VyZSAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiAgICBkZWYgX2Nsb3N1cmUoc2VsZiwgY29uZmlncykgLT4gRnJvemVuU2V0W0NvbmZpZ106CiAgICAgICAgc2VlbjogU2V0W0NvbmZpZ10gPSBzZXQoKQogICAgICAgIHdvcmsgPSBsaXN0KGNvbmZpZ3MpCiAgICAgICAgd2hpbGUgd29yazoKICAgICAgICAgICAgY2ZnID0gd29yay5wb3AoKQogICAgICAgICAgICBpZiBjZmcgaW4gc2VlbjoKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIHNlZW4uYWRkKGNmZykKICAgICAgICAgICAgbm9kZSwgc3RhY2sgPSBjZmcKICAgICAgICAgICAgZm9yIGVkZ2UgaW4gc2VsZi5lZGdlcy5nZXQobm9kZSwgKCkpOgogICAgICAgICAgICAgICAgaWYgZWRnZVswXSA9PSAiZXBzIjoKICAgICAgICAgICAgICAgICAgICB3b3JrLmFwcGVuZCgoZWRnZVsxXSwgc3RhY2spKQogICAgICAgICAgICAgICAgZWxpZiBlZGdlWzBdID09ICJjYWxsIjoKICAgICAgICAgICAgICAgICAgICBpZiBsZW4oc3RhY2spIDwgX01BWF9TVEFDSzoKICAgICAgICAgICAgICAgICAgICAgICAgd29yay5hcHBlbmQoKGVkZ2VbMV0sIHN0YWNrICsgKGVkZ2VbMl0sKSkpCiAgICAgICAgICAgICMgUnVsZSBleGl0OiBwb3AgYW5kIGNvbnRpbnVlIGF0IHRoZSBzYXZlZCByZXR1cm4gcG9pbnQuCiAgICAgICAgICAgIGlmIG5vZGUgaW4gc2VsZi5yZXR1cm5zIGFuZCBzdGFjazoKICAgICAgICAgICAgICAgIHdvcmsuYXBwZW5kKChzdGFja1stMV0sIHN0YWNrWzotMV0pKQogICAgICAgIHJldHVybiBmcm96ZW5zZXQoc2VlbikK",
"engine/json_schema.py": "IiIiQ29tcGlsZSBhIEpTT04gU2NoZW1hIGludG8gYSBwdXNoZG93biBncmFtbWFyIChQaGFzZSAyKS4KCk1hcHMgc2NoZW1hIGNvbnN0cnVjdHMgb250byBncmFtbWFyIHJ1bGVzOiBvYmplY3RzIHRvIGBgeyBrZXkgOiB2YWx1ZSAsIC4uLiB9YGAKd2l0aCByZXF1aXJlZCBrZXlzIGluIGRlY2xhcmVkIG9yZGVyIGFuZCBvcHRpb25hbCBrZXlzIHRoYXQgbWF5IGJlIHNraXBwZWQKKHdpdGggdGhlIGNvbW1hIGxvZ2ljIGhhbmRsZWQgc28gb3V0cHV0IGlzIGFsd2F5cyB3ZWxsLWZvcm1lZCk7IGFycmF5cyB0bwpgYFsgdmFsdWUgLCAuLi4gXWBgIG9mIHRoZSBpdGVtIHR5cGU7IGFuZCBzY2FsYXJzIHRvIHNoYXJlZCBTVFJJTkcgLyBJTlQgLwpOVU1CRVIgLyBib29sZWFuIC8gbnVsbCAvIGVudW0gbWF0Y2hlcnMuIEluc2lnbmlmaWNhbnQgSlNPTiB3aGl0ZXNwYWNlIGlzCmFsbG93ZWQgYmV0d2VlbiB0b2tlbnMsIHNvIHRoZSBtb2RlbCBtYXkgZW1pdCBgYHsiYSI6IDF9YGAgb3IgYGB7ICJhIjoxIH1gYC4KCk5lc3RlZCBvYmplY3RzIGFuZCBhcnJheXMgYXJlIGVtaXR0ZWQgYXMgYGBDYWxsYGBcXCBzIGludG8gdGhlaXIgb3duIHJ1bGVzLCBzbwpyZWFsIG5lc3RpbmcgcnVucyB0aHJvdWdoIHRoZSBQREEgc3RhY2suIGBgYnVpbGRfanNvbl9wcm9jZXNzb3JgYCByZXR1cm5zIGEKcmVhZHkgTG9naXRzUHJvY2Vzc29yOyBnZW5lcmF0aW9uIGNvbnN0cmFpbmVkIGJ5IGl0IHBhcnNlcyBhcyB0aGUgc2NoZW1hIGJ5CmNvbnN0cnVjdGlvbi4KClNjb3BlOiBmaXhlZCBzY2hlbWFzIHdpdGggb2JqZWN0L2FycmF5L3N0cmluZy9pbnRlZ2VyL251bWJlci9ib29sZWFuL251bGwsCmBgZW51bWBgIGFuZCBgYGNvbnN0YGAuIFJlY3Vyc2l2ZSBgYCRyZWZgYCBpcyBvdXQgb2Ygc2NvcGUgKFBoYXNlIDMgRUJORiBjb3ZlcnMKcmVjdXJzaW9uKTsgYGBhZGRpdGlvbmFsUHJvcGVydGllc2BgIGlzIHRyZWF0ZWQgYXMgY2xvc2VkIChvbmx5IGxpc3RlZCBrZXlzKS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQganNvbgpmcm9tIHR5cGluZyBpbXBvcnQgRGljdCwgTGlzdCwgT3B0aW9uYWwsIFR1cGxlCgpmcm9tIGVuZ2luZS5wdXNoZG93biBpbXBvcnQgKAogICAgQWx0LAogICAgQ2FsbCwKICAgIENoYXJzLAogICAgRW1wdHksCiAgICBHcmFtbWFyLAogICAgTGl0LAogICAgT3B0LAogICAgUGx1cywKICAgIFNlcSwKICAgIFN0YXIsCiAgICBhbnlfYnV0LAogICAgY2hhcl9zZXQsCiAgICBjb21waWxlX2dyYW1tYXIsCiAgICBsaXRfY2hhciwKKQoKIyBJbnNpZ25pZmljYW50IHdoaXRlc3BhY2UgKHJldXNhYmxlIG5vZGU7IGNvbXBpbGUgbWludHMgZnJlc2ggc3RhdGVzIHBlciB1c2UpLgpfV1MgPSBTdGFyKGNoYXJfc2V0KCIgXHRcblxyIikpCl9ESUdJVCA9IGNoYXJfc2V0KCIwMTIzNDU2Nzg5IikKX05PTlpFUk8gPSBjaGFyX3NldCgiMTIzNDU2Nzg5IikKX0hFWCA9IGNoYXJfc2V0KCIwMTIzNDU2Nzg5YWJjZGVmQUJDREVGIikKCgpkZWYgX3NjYWxhcl9ydWxlcygpIC0+IERpY3Rbc3RyLCBvYmplY3RdOgogICAgIiIiVGhlIHNoYXJlZCBsZWFmIHJ1bGVzIGV2ZXJ5IHNjaGVtYSByZXVzZXMuIiIiCiAgICBzdHJpbmcgPSBTZXEoWwogICAgICAgIExpdCgnIicpLAogICAgICAgIFN0YXIoQWx0KFsKICAgICAgICAgICAgYW55X2J1dCh7JyInLCAiXFwifSksICAjIG9yZGluYXJ5IGNoYXJhY3RlciAoYSAneycgaGVyZSBpcyBqdXN0IHRleHQpCiAgICAgICAgICAgIFNlcShbTGl0KCJcXCIpLCBBbHQoWwogICAgICAgICAgICAgICAgY2hhcl9zZXQoJyJcXC9iZm5ydCcpLAogICAgICAgICAgICAgICAgU2VxKFtMaXQoInUiKSwgX0hFWCwgX0hFWCwgX0hFWCwgX0hFWF0pLAogICAgICAgICAgICBdKV0pLAogICAgICAgIF0pKSwKICAgICAgICBMaXQoJyInKSwKICAgIF0pCiAgICBpbnRfY29yZSA9IFNlcShbCiAgICAgICAgT3B0KExpdCgiLSIpKSwKICAgICAgICBBbHQoW0xpdCgiMCIpLCBTZXEoW19OT05aRVJPLCBTdGFyKF9ESUdJVCldKV0pLAogICAgXSkKICAgIG51bWJlciA9IFNlcShbCiAgICAgICAgaW50X2NvcmUsCiAgICAgICAgT3B0KFNlcShbTGl0KCIuIiksIFBsdXMoX0RJR0lUKV0pKSwKICAgICAgICBPcHQoU2VxKFtjaGFyX3NldCgiZUUiKSwgT3B0KGNoYXJfc2V0KCIrLSIpKSwgUGx1cyhfRElHSVQpXSkpLAogICAgXSkKICAgIHJldHVybiB7IlNUUklORyI6IHN0cmluZywgIklOVCI6IGludF9jb3JlLCAiTlVNQkVSIjogbnVtYmVyfQoKCmNsYXNzIF9TY2hlbWFDb21waWxlcjoKICAgIGRlZiBfX2luaXRfXyhzZWxmKToKICAgICAgICBzZWxmLnJ1bGVzOiBEaWN0W3N0ciwgb2JqZWN0XSA9IF9zY2FsYXJfcnVsZXMoKQogICAgICAgIHNlbGYuX2NvdW50ZXIgPSAwCgogICAgZGVmIF9mcmVzaChzZWxmLCBoaW50OiBzdHIpIC0+IHN0cjoKICAgICAgICBzZWxmLl9jb3VudGVyICs9IDEKICAgICAgICByZXR1cm4gZiJ7aGludH17c2VsZi5fY291bnRlcn0iCgogICAgIyAtLSB2YWx1ZSBvZiBhbnkgc2NoZW1hIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KICAgIGRlZiB2YWx1ZShzZWxmLCBzY2hlbWE6IGRpY3QpIC0+IG9iamVjdDoKICAgICAgICBpZiBzY2hlbWEgaXMgTm9uZToKICAgICAgICAgICAgc2NoZW1hID0ge30KICAgICAgICBpZiAiY29uc3QiIGluIHNjaGVtYToKICAgICAgICAgICAgcmV0dXJuIExpdChqc29uLmR1bXBzKHNjaGVtYVsiY29uc3QiXSkpCiAgICAgICAgaWYgImVudW0iIGluIHNjaGVtYToKICAgICAgICAgICAgcmV0dXJuIEFsdChbTGl0KGpzb24uZHVtcHModikpIGZvciB2IGluIHNjaGVtYVsiZW51bSJdXSkKCiAgICAgICAgdCA9IHNjaGVtYS5nZXQoInR5cGUiKQogICAgICAgIGlmIGlzaW5zdGFuY2UodCwgbGlzdCk6CiAgICAgICAgICAgIHJldHVybiBBbHQoW3NlbGYudmFsdWUoeyoqc2NoZW1hLCAidHlwZSI6IG9uZX0pIGZvciBvbmUgaW4gdF0pCgogICAgICAgIGlmIHQgPT0gIm9iamVjdCI6CiAgICAgICAgICAgIHJldHVybiBDYWxsKHNlbGYuX29iamVjdF9ydWxlKHNjaGVtYSkpCiAgICAgICAgaWYgdCA9PSAiYXJyYXkiOgogICAgICAgICAgICByZXR1cm4gQ2FsbChzZWxmLl9hcnJheV9ydWxlKHNjaGVtYSkpCiAgICAgICAgaWYgdCA9PSAic3RyaW5nIjoKICAgICAgICAgICAgcmV0dXJuIENhbGwoIlNUUklORyIpCiAgICAgICAgaWYgdCA9PSAiaW50ZWdlciI6CiAgICAgICAgICAgIHJldHVybiBDYWxsKCJJTlQiKQogICAgICAgIGlmIHQgPT0gIm51bWJlciI6CiAgICAgICAgICAgIHJldHVybiBDYWxsKCJOVU1CRVIiKQogICAgICAgIGlmIHQgPT0gImJvb2xlYW4iOgogICAgICAgICAgICByZXR1cm4gQWx0KFtMaXQoInRydWUiKSwgTGl0KCJmYWxzZSIpXSkKICAgICAgICBpZiB0ID09ICJudWxsIjoKICAgICAgICAgICAgcmV0dXJuIExpdCgibnVsbCIpCiAgICAgICAgIyBObyB0eXBlL2VudW0vY29uc3Q6IHBlcm1pdCBhbnkgc2NhbGFyIEpTT04gdmFsdWUgKG5vIG5lc3RlZCBjb250YWluZXIpLgogICAgICAgIHJldHVybiBBbHQoWwogICAgICAgICAgICBDYWxsKCJTVFJJTkciKSwgQ2FsbCgiTlVNQkVSIiksCiAgICAgICAgICAgIExpdCgidHJ1ZSIpLCBMaXQoImZhbHNlIiksIExpdCgibnVsbCIpLAogICAgICAgIF0pCgogICAgIyAtLSBvYmplY3QgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KICAgIGRlZiBfb2JqZWN0X3J1bGUoc2VsZiwgc2NoZW1hOiBkaWN0KSAtPiBzdHI6CiAgICAgICAgbmFtZSA9IHNlbGYuX2ZyZXNoKCJvYmoiKQogICAgICAgIHByb3BzOiBkaWN0ID0gc2NoZW1hLmdldCgicHJvcGVydGllcyIsIHt9KQogICAgICAgIHJlcXVpcmVkID0gc2V0KHNjaGVtYS5nZXQoInJlcXVpcmVkIiwgW10pKQogICAgICAgIGtleXM6IExpc3RbVHVwbGVbc3RyLCBkaWN0LCBib29sXV0gPSBbCiAgICAgICAgICAgIChrLCBzdWIsIGsgaW4gcmVxdWlyZWQpIGZvciBrLCBzdWIgaW4gcHJvcHMuaXRlbXMoKQogICAgICAgIF0KICAgICAgICBib2R5ID0gU2VxKFtMaXQoInsiKSwgX1dTLCBzZWxmLl9tZW1iZXJzKGtleXMpLCBfV1MsIExpdCgifSIpXSkKICAgICAgICBzZWxmLnJ1bGVzW25hbWVdID0gYm9keQogICAgICAgIHJldHVybiBuYW1lCgogICAgZGVmIF9tZW1iZXIoc2VsZiwga2V5OiBzdHIsIHN1YjogZGljdCkgLT4gb2JqZWN0OgogICAgICAgIHJldHVybiBTZXEoW0xpdChqc29uLmR1bXBzKGtleSkpLCBfV1MsIExpdCgiOiIpLCBfV1MsIHNlbGYudmFsdWUoc3ViKV0pCgogICAgZGVmIF9tZW1iZXJzKHNlbGYsIGtleXM6IExpc3RbVHVwbGVbc3RyLCBkaWN0LCBib29sXV0sIGVtaXR0ZWQ6IGJvb2wgPSBGYWxzZSkgLT4gb2JqZWN0OgogICAgICAgICIiIkVtaXQga2V5cyBpbiBvcmRlcjsgYSBjb21tYSBwcmVjZWRlcyBldmVyeSBrZXkgYWZ0ZXIgdGhlIGZpcnN0CiAgICAgICAgZW1pdHRlZCBvbmUuIE9wdGlvbmFsIGtleXMgYnJhbmNoIGludG8gcHJlc2VudC9hYnNlbnQsIHNvIHRoZSBjb21tYQogICAgICAgIHBsYWNlbWVudCBpcyBhbHdheXMgY29ycmVjdCByZWdhcmRsZXNzIG9mIHdoaWNoIG9wdGlvbmFscyBhcHBlYXIuIiIiCiAgICAgICAgaWYgbm90IGtleXM6CiAgICAgICAgICAgIHJldHVybiBFbXB0eSgpCiAgICAgICAgKGtleSwgc3ViLCByZXF1aXJlZCksIHJlc3QgPSBrZXlzWzBdLCBrZXlzWzE6XQogICAgICAgIG1lbWJlciA9IHNlbGYuX21lbWJlcihrZXksIHN1YikKICAgICAgICB3aXRoX3NlcCA9IG1lbWJlciBpZiBub3QgZW1pdHRlZCBlbHNlIFNlcShbX1dTLCBMaXQoIiwiKSwgX1dTLCBtZW1iZXJdKQogICAgICAgIHByZXNlbnQgPSBTZXEoW3dpdGhfc2VwLCBzZWxmLl9tZW1iZXJzKHJlc3QsIFRydWUpXSkKICAgICAgICBpZiByZXF1aXJlZDoKICAgICAgICAgICAgcmV0dXJuIHByZXNlbnQKICAgICAgICBhYnNlbnQgPSBzZWxmLl9tZW1iZXJzKHJlc3QsIGVtaXR0ZWQpCiAgICAgICAgcmV0dXJuIEFsdChbcHJlc2VudCwgYWJzZW50XSkKCiAgICAjIC0tIGFycmF5IC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogICAgZGVmIF9hcnJheV9ydWxlKHNlbGYsIHNjaGVtYTogZGljdCkgLT4gc3RyOgogICAgICAgIG5hbWUgPSBzZWxmLl9mcmVzaCgiYXJyIikKICAgICAgICBpdGVtcyA9IHNjaGVtYS5nZXQoIml0ZW1zIiwge30pCiAgICAgICAgZWxlbWVudCA9IHNlbGYudmFsdWUoaXRlbXMpCiAgICAgICAgbW9yZSA9IFN0YXIoU2VxKFtfV1MsIExpdCgiLCIpLCBfV1MsIHNlbGYudmFsdWUoaXRlbXMpXSkpCiAgICAgICAgYm9keSA9IFNlcShbCiAgICAgICAgICAgIExpdCgiWyIpLCBfV1MsCiAgICAgICAgICAgIE9wdChTZXEoW2VsZW1lbnQsIG1vcmVdKSksCiAgICAgICAgICAgIF9XUywgTGl0KCJdIiksCiAgICAgICAgXSkKICAgICAgICBzZWxmLnJ1bGVzW25hbWVdID0gYm9keQogICAgICAgIHJldHVybiBuYW1lCgoKZGVmIHNjaGVtYV90b19ncmFtbWFyKHNjaGVtYTogZGljdCkgLT4gR3JhbW1hcjoKICAgIGMgPSBfU2NoZW1hQ29tcGlsZXIoKQogICAgYy5ydWxlc1siTUFJTiJdID0gU2VxKFtfV1MsIGMudmFsdWUoc2NoZW1hKSwgX1dTXSkKICAgIHJldHVybiBHcmFtbWFyKHJ1bGVzPWMucnVsZXMsIHN0YXJ0PSJNQUlOIikKCgpkZWYgYnVpbGRfanNvbl9hdXRvbWF0b24oc2NoZW1hOiBkaWN0KToKICAgICIiIkNvbXBpbGUgYSBzY2hlbWEgc3RyYWlnaHQgdG8gYSA6Y2xhc3M6YH5lbmdpbmUucHVzaGRvd24uUERBYC4iIiIKICAgIHJldHVybiBjb21waWxlX2dyYW1tYXIoc2NoZW1hX3RvX2dyYW1tYXIoc2NoZW1hKSkKCgpkZWYgYnVpbGRfanNvbl9wcm9jZXNzb3Ioc2NoZW1hOiBkaWN0LCB2b2NhYiwgaW5kZXhlZDogYm9vbCA9IEZhbHNlLCBwcmVjb21wdXRlOiBib29sID0gRmFsc2UpOgogICAgIiIiUmV0dXJuIGEgTG9naXRzUHJvY2Vzc29yIHRoYXQgY29uc3RyYWlucyBnZW5lcmF0aW9uIHRvIGBgc2NoZW1hYGAuCgogICAgYGBpbmRleGVkYGAgYWRkcyB0aGUgUGhhc2UtMyBwZXItc3RhdGUgdG9rZW4gY2FjaGUgKDpjbGFzczpgVG9rZW5JbmRleGApLAogICAgd2hpY2ggcmVtb3ZlcyB0aGUgcGVyLXN0ZXAgZnVsbC12b2NhYiBzY2FuOyBgYHByZWNvbXB1dGVgYCBmaWxscyBpdCBlYWdlcmx5CiAgICBieSB3YWxraW5nIHRoZSByZWFjaGFibGUgc3RhdGUgZ3JhcGggb25jZSB1cCBmcm9udC4iIiIKICAgIGZyb20gZW5naW5lLm1hc2tlciBpbXBvcnQgQXV0b21hdG9uTG9naXRzUHJvY2Vzc29yCiAgICBmcm9tIGVuZ2luZS50b2tlbl9icmlkZ2UgaW1wb3J0IGFsbG93ZWRfdG9rZW5faWRzCgogICAgYXV0b21hdG9uID0gYnVpbGRfanNvbl9hdXRvbWF0b24oc2NoZW1hKQogICAgYWxsb3dlZF9mbiA9IGFsbG93ZWRfdG9rZW5faWRzCiAgICBpZiBpbmRleGVkOgogICAgICAgIGZyb20gZW5naW5lLnRva2VuX2luZGV4IGltcG9ydCBUb2tlbkluZGV4CgogICAgICAgIGlkeCA9IFRva2VuSW5kZXgoYXV0b21hdG9uLCB2b2NhYikKICAgICAgICBpZiBwcmVjb21wdXRlOgogICAgICAgICAgICBpZHgucHJlY29tcHV0ZSgpCiAgICAgICAgYWxsb3dlZF9mbiA9IGlkeAogICAgcmV0dXJuIEF1dG9tYXRvbkxvZ2l0c1Byb2Nlc3NvcihhdXRvbWF0b24sIHZvY2FiLCBhbGxvd2VkX2ZuPWFsbG93ZWRfZm4pCg==",
"engine/token_index.py": "IiIiUHJlY29tcHV0ZWQgcGVyLXN0YXRlIHRva2VuLW1hc2sgaW5kZXggKFBoYXNlIDMpIOKAlCB0aGUgc3BlZWQgZml4LgoKVGhlIFBoYXNlLTEvMiBicmlkZ2UgcmVjb21wdXRlcyB0aGUgYWxsb3dlZC10b2tlbiBzZXQgYnkgc2Nhbm5pbmcgdGhlIHdob2xlCnZvY2FidWxhcnkgKH4xNTBrIHRva2VucykgYXQgKmV2ZXJ5KiBkZWNvZGUgc3RlcC4gQnV0IGBgYWxsb3dlZF90b2tlbl9pZHNgYCBpcyBhCnB1cmUgZnVuY3Rpb24gb2YgdGhlIGF1dG9tYXRvbiBzdGF0ZSwgc28gdGhlIHJlc3VsdCBjYW4gYmUgY2FjaGVkIGJ5IHN0YXRlOiB0aGUKc2NhbiB0aGVuIGhhcHBlbnMgb25jZSBwZXIgKmRpc3RpbmN0IHN0YXRlKiBpbnN0ZWFkIG9mIG9uY2UgcGVyIHN0ZXAsIGFuZApkZWNvZGUtdGltZSBtYXNraW5nIGJlY29tZXMgYSBkaWN0aW9uYXJ5IGxvb2t1cC4KCmBgVG9rZW5JbmRleGBgIGlzIGEgZHJvcC1pbiBmb3IgdGhlIG1hc2tlcidzIGBgYWxsb3dlZF9mbmBgIChzYW1lIGNhbGwKc2lnbmF0dXJlKSwgc28gbm90aGluZyBlbHNlIGNoYW5nZXMuIEl0IGZpbGxzIGxhemlseSBvbiBmaXJzdCB2aXNpdCwgb3IgZWFnZXJseQp2aWEgOm1ldGg6YHByZWNvbXB1dGVgLCB3aGljaCB3YWxrcyB0aGUgcmVhY2hhYmxlIHN0YXRlIGdyYXBoIG9uY2UgdXAgZnJvbnQuCgpDb3JyZWN0bmVzcyBpcyB1bmNoYW5nZWQg4oCUIHRoZSBjYWNoZWQgc2V0IGlzIGV4YWN0bHkgd2hhdCB0aGUgbmFpdmUgc2NhbiB3b3VsZApyZXR1cm4gKGEgdGVzdCBhc3NlcnRzIHRoaXMpLiBPbmx5IGxhdGVuY3kgZGlmZmVycy4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIHR5cGluZyBpbXBvcnQgRGljdCwgRnJvemVuU2V0LCBPcHRpb25hbCwgU2V0Cgpmcm9tIGVuZ2luZS50b2tlbl9icmlkZ2UgaW1wb3J0IFRva2VuVm9jYWIsIGFsbG93ZWRfdG9rZW5faWRzCgoKY2xhc3MgVG9rZW5JbmRleDoKICAgICIiIlBlci1zdGF0ZSBjYWNoZSBvZiBhbGxvd2VkIHRva2VuIGlkcy4KCiAgICBVc2FnZTo6CgogICAgICAgIGluZGV4ID0gVG9rZW5JbmRleChhdXRvbWF0b24sIHZvY2FiKQogICAgICAgIHByb2MgPSBBdXRvbWF0b25Mb2dpdHNQcm9jZXNzb3IoYXV0b21hdG9uLCB2b2NhYiwgYWxsb3dlZF9mbj1pbmRleCkKICAgICIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBhdXRvbWF0b24sIHZvY2FiOiBUb2tlblZvY2FiKToKICAgICAgICBzZWxmLmF1dG9tYXRvbiA9IGF1dG9tYXRvbgogICAgICAgIHNlbGYudm9jYWIgPSB2b2NhYgogICAgICAgIHNlbGYuX2NhY2hlOiBEaWN0W29iamVjdCwgRnJvemVuU2V0W2ludF1dID0ge30KICAgICAgICBzZWxmLmhpdHMgPSAwCiAgICAgICAgc2VsZi5taXNzZXMgPSAwCgogICAgIyBNYXRjaGVzIHRoZSBhbGxvd2VkX2ZuIHNpZ25hdHVyZSAoYXV0b21hdG9uLCBzdGF0ZSwgdm9jYWIpOyB0aGUgcGFzc2VkCiAgICAjIGF1dG9tYXRvbi92b2NhYiBhcmUgaWdub3JlZCBpbiBmYXZvdXIgb2YgdGhlIG9uZXMgdGhpcyBpbmRleCB3YXMgYnVpbHQgZm9yLgogICAgZGVmIF9fY2FsbF9fKHNlbGYsIGF1dG9tYXRvbiwgc3RhdGUsIHZvY2FiKSAtPiBTZXRbaW50XToKICAgICAgICBjYWNoZWQgPSBzZWxmLl9jYWNoZS5nZXQoc3RhdGUpCiAgICAgICAgaWYgY2FjaGVkIGlzIG5vdCBOb25lOgogICAgICAgICAgICBzZWxmLmhpdHMgKz0gMQogICAgICAgICAgICByZXR1cm4gY2FjaGVkCiAgICAgICAgc2VsZi5taXNzZXMgKz0gMQogICAgICAgIHJlc3VsdCA9IGZyb3plbnNldChhbGxvd2VkX3Rva2VuX2lkcyhzZWxmLmF1dG9tYXRvbiwgc3RhdGUsIHNlbGYudm9jYWIpKQogICAgICAgIHNlbGYuX2NhY2hlW3N0YXRlXSA9IHJlc3VsdAogICAgICAgIHJldHVybiByZXN1bHQKCiAgICBkZWYgYWxsb3dlZChzZWxmLCBzdGF0ZSkgLT4gU2V0W2ludF06CiAgICAgICAgcmV0dXJuIHNlbGYoc2VsZi5hdXRvbWF0b24sIHN0YXRlLCBzZWxmLnZvY2FiKQoKICAgIGRlZiBwcmVjb21wdXRlKHNlbGYsIG1heF9zdGF0ZXM6IGludCA9IDEwMF8wMDApIC0+ICJUb2tlbkluZGV4IjoKICAgICAgICAiIiJFYWdlcmx5IGZpbGwgdGhlIGNhY2hlIGJ5IHdhbGtpbmcgZXZlcnkgcmVhY2hhYmxlIHN0YXRlLgoKICAgICAgICBCcmVhZHRoLWZpcnN0IG92ZXIgdGhlIGF1dG9tYXRvbidzIGFscGhhYmV0IChwbHVzIG9uZSByZXByZXNlbnRhdGl2ZQogICAgICAgIG91dC1vZi1hbHBoYWJldCBjaGFyYWN0ZXIsIHRvIGNvdmVyIHRoZSBPVEhFUiBzZW50aW5lbCdzIHRyYW5zaXRpb25zKS4KICAgICAgICBCb3VuZGVkIGJ5IGBgbWF4X3N0YXRlc2BgIHNvIGEgcGF0aG9sb2dpY2FsIGdyYW1tYXIgY2FuJ3QgcnVuIGF3YXkuIiIiCiAgICAgICAgYXV0ID0gc2VsZi5hdXRvbWF0b24KICAgICAgICAjIEEgY2hhcmFjdGVyIHRoYXQgaXMgZ3VhcmFudGVlZCB0byBtYXAgdG8gT1RIRVIgKG5vdCBpbiB0aGUgYWxwaGFiZXQpLgogICAgICAgIG90aGVyX3Byb2JlID0gTm9uZQogICAgICAgIGZvciBjYW5kIGluICgiXHgwMCIsICLvv78iLCAiXHgwMSIpOgogICAgICAgICAgICBpZiBjYW5kIG5vdCBpbiBhdXQuYWxwaGFiZXQ6CiAgICAgICAgICAgICAgICBvdGhlcl9wcm9iZSA9IGNhbmQKICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgcHJvYmVfY2hhcnMgPSBsaXN0KGF1dC5hbHBoYWJldCkKICAgICAgICBpZiBvdGhlcl9wcm9iZSBpcyBub3QgTm9uZToKICAgICAgICAgICAgcHJvYmVfY2hhcnMuYXBwZW5kKG90aGVyX3Byb2JlKQoKICAgICAgICBzZWVuOiBTZXRbb2JqZWN0XSA9IHNldCgpCiAgICAgICAgZnJvbnRpZXIgPSBbYXV0LnN0YXJ0XQogICAgICAgIHNlZW4uYWRkKGF1dC5zdGFydCkKICAgICAgICB3aGlsZSBmcm9udGllcjoKICAgICAgICAgICAgaWYgbGVuKHNlZW4pID4gbWF4X3N0YXRlczoKICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigKICAgICAgICAgICAgICAgICAgICBmInByZWNvbXB1dGUgZXhjZWVkZWQge21heF9zdGF0ZXN9IHN0YXRlczsgZ3JhbW1hciBtYXkgYmUgIgogICAgICAgICAgICAgICAgICAgICJ1bmJvdW5kZWQg4oCUIHVzZSBsYXp5IGluZGV4aW5nIGluc3RlYWQuIgogICAgICAgICAgICAgICAgKQogICAgICAgICAgICBzdGF0ZSA9IGZyb250aWVyLnBvcCgpCiAgICAgICAgICAgIHNlbGYoYXV0LCBzdGF0ZSwgc2VsZi52b2NhYikgICMgY29tcHV0ZSArIGNhY2hlIGFsbG93ZWQgc2V0CiAgICAgICAgICAgIGZvciBjaCBpbiBwcm9iZV9jaGFyczoKICAgICAgICAgICAgICAgIG54dCA9IGF1dC5zdGVwKHN0YXRlLCBjaCkKICAgICAgICAgICAgICAgIGlmIG5vdCBhdXQuaXNfZGVhZChueHQpIGFuZCBueHQgbm90IGluIHNlZW46CiAgICAgICAgICAgICAgICAgICAgc2Vlbi5hZGQobnh0KQogICAgICAgICAgICAgICAgICAgIGZyb250aWVyLmFwcGVuZChueHQpCiAgICAgICAgcmV0dXJuIHNlbGYKCiAgICBAcHJvcGVydHkKICAgIGRlZiBuX3N0YXRlcyhzZWxmKSAtPiBpbnQ6CiAgICAgICAgcmV0dXJuIGxlbihzZWxmLl9jYWNoZSkK",
"engine/ebnf.py": "IiIiRUJORiBncmFtbWFyIGZyb250ZW5kIChQaGFzZSAzKSDigJQgYSBjb250ZXh0LWZyZWUgZ3JhbW1hciBjb21waWxlZCB0byB0aGUKcHVzaGRvd24gYXV0b21hdG9uLCBmb3IgY29uc3RyYWluaW5nIG91dHB1dCB0byBhIHRvb2wtY2FsbCBEU0wuCgpUaGlzIGlzIHdoZXJlIHRoZSBQREEncyBzdGFjayBnZW51aW5lbHkgZWFybnMgaXRzIGtlZXA6IGEgZ3JhbW1hciBydWxlIG1heQpyZWZlcmVuY2UgaXRzZWxmIChkaXJlY3RseSBvciB0aHJvdWdoIG90aGVycyksIHNvIG5lc3RpbmcgZGVwdGggaXMgdW5ib3VuZGVkIGFuZApubyBmaW5pdGUgREZBIGNhbiByZXByZXNlbnQgaXQuIEZvciBleGFtcGxlIGEgYGB2YWx1ZWBgIHRoYXQgbWF5IGJlIGEgYGBjYWxsYGAKbWFrZXMgYGBhZGQobXVsKDIsMyksIG5lZyg0KSlgYCBsZWdhbCBhdCBhbnkgZGVwdGgg4oCUIHRoZSBzdGFjayB0cmFja3MgaXQuCgpTdXBwb3J0ZWQgRUJORiBzdWJzZXQKICAgIHJ1bGUgPSBleHByZXNzaW9uIDsgICAgICAgICAgICBydWxlIGRlZmluaXRpb24gKGZpcnN0IHJ1bGUgaXMgdGhlIHN0YXJ0KQogICAgYSB8IGIgICAgICAgICAgICAgICAgICAgICAgICAgIGFsdGVybmF0aW9uCiAgICBhIGIgICAob3IgIGEgLCBiKSAgICAgICAgICAgICAgY29uY2F0ZW5hdGlvbgogICAgImxpdCIgLyAnbGl0JyAgICAgICAgICAgICAgICAgIHRlcm1pbmFsIHN0cmluZwogICAgL3JlZ2V4LyAgICAgICAgICAgICAgICAgICAgICAgIHRlcm1pbmFsIHBhdHRlcm4gKHJldXNlcyB0aGUgUGhhc2UtMSBwYXJzZXIpCiAgICAoIC4uLiApICAgICAgICAgICAgICAgICAgICAgICAgZ3JvdXBpbmcKICAgIFsgLi4uIF0gICAgICAgICAgICAgICAgICAgICAgICBvcHRpb25hbCAgIHsgLi4uIH0gIHJlcGV0aXRpb24gKHplcm8rKQogICAgeD8gIHgqICB4KyAgICAgICAgICAgICAgICAgICAgIHBvc3RmaXggb3B0aW9uYWwgLyBzdGFyIC8gcGx1cwogICAgKCogY29tbWVudCAqKSAgICAgICAgICAgICAgICAgIGNvbW1lbnRzCiAgICBJREVOVCAgICAgICAgICAgICAgICAgICAgICAgICAgcmVmZXJlbmNlIHRvIGFub3RoZXIgcnVsZQoKRXZlcnl0aGluZyBjb21waWxlcyB0byB0aGUgY29tYmluYXRvcnMgaW4gOm1vZDpgZW5naW5lLnB1c2hkb3duYCwgc28gdGhlIHRva2VuCmJyaWRnZSwgbWFza2VyLCBhbmQgdG9rZW4gaW5kZXggYWxsIGFwcGx5IHVuY2hhbmdlZC4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgcmUKZnJvbSB0eXBpbmcgaW1wb3J0IERpY3QsIExpc3QsIE9wdGlvbmFsLCBUdXBsZQoKZnJvbSBlbmdpbmUucHVzaGRvd24gaW1wb3J0ICgKICAgIEFsdCwKICAgIENhbGwsCiAgICBDaGFycywKICAgIEVtcHR5LAogICAgR3JhbW1hciwKICAgIExpdCwKICAgIE9wdCwKICAgIFBsdXMsCiAgICBTZXEsCiAgICBTdGFyLAogICAgY29tcGlsZV9ncmFtbWFyLAopCmZyb20gZW5naW5lLnJlZ2V4X2NvbXBpbGVyIGltcG9ydCBwYXJzZV9yZWdleF9hc3QKaW1wb3J0IGVuZ2luZS5yZWdleF9jb21waWxlciBhcyBfcngKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSAjCiMgL3JlZ2V4LyB0ZXJtaW5hbCAtPiBwdXNoZG93biBjb21iaW5hdG9ycyAocmV1c2UgdGhlIFBoYXNlLTEgcmVnZXggQVNUKQojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSAjCmRlZiBfcmVnZXhfbm9kZV90b19jb21iaW5hdG9yKG5vZGUpOgogICAgaWYgaXNpbnN0YW5jZShub2RlLCBfcnguX0VtcHR5KToKICAgICAgICByZXR1cm4gRW1wdHkoKQogICAgaWYgaXNpbnN0YW5jZShub2RlLCBfcnguX0xlYWYpOgogICAgICAgIHNwZWMgPSBub2RlLnNwZWMKICAgICAgICBpZiBzcGVjLmtpbmQgPT0gImxpdCI6CiAgICAgICAgICAgIHJldHVybiBDaGFycygoImxpdCIsIGZyb3plbnNldChzcGVjLmNoYXJzKSkpCiAgICAgICAgaWYgc3BlYy5raW5kID09ICJhbnkiOgogICAgICAgICAgICByZXR1cm4gQ2hhcnMoKCJhbnkiLCkpCiAgICAgICAgIyBjbGFzcwogICAgICAgIGlmIHNwZWMubmVnYXRlZDoKICAgICAgICAgICAgcmV0dXJuIENoYXJzKCgiYW55YnV0IiwgZnJvemVuc2V0KHNwZWMuY2hhcnMpKSkKICAgICAgICByZXR1cm4gQ2hhcnMoKCJzZXQiLCBmcm96ZW5zZXQoc3BlYy5jaGFycykpKQogICAgaWYgaXNpbnN0YW5jZShub2RlLCBfcnguX0NvbmNhdCk6CiAgICAgICAgcmV0dXJuIFNlcShbX3JlZ2V4X25vZGVfdG9fY29tYmluYXRvcihwKSBmb3IgcCBpbiBub2RlLnBhcnRzXSkKICAgIGlmIGlzaW5zdGFuY2Uobm9kZSwgX3J4Ll9BbHQpOgogICAgICAgIHJldHVybiBBbHQoW19yZWdleF9ub2RlX3RvX2NvbWJpbmF0b3IobykgZm9yIG8gaW4gbm9kZS5vcHRpb25zXSkKICAgIGlmIGlzaW5zdGFuY2Uobm9kZSwgX3J4Ll9SZXBlYXQpOgogICAgICAgIGlubmVyID0gX3JlZ2V4X25vZGVfdG9fY29tYmluYXRvcihub2RlLm5vZGUpCiAgICAgICAgbSwgbiA9IG5vZGUubSwgbm9kZS5uCiAgICAgICAgaWYgbSA9PSAwIGFuZCBuIGlzIE5vbmU6CiAgICAgICAgICAgIHJldHVybiBTdGFyKGlubmVyKQogICAgICAgIGlmIG0gPT0gMSBhbmQgbiBpcyBOb25lOgogICAgICAgICAgICByZXR1cm4gUGx1cyhpbm5lcikKICAgICAgICBpZiBtID09IDAgYW5kIG4gPT0gMToKICAgICAgICAgICAgcmV0dXJuIE9wdChpbm5lcikKICAgICAgICBpZiBuIGlzIE5vbmU6CiAgICAgICAgICAgIHJldHVybiBTZXEoW2lubmVyXSAqIG0gKyBbU3Rhcihpbm5lcildKQogICAgICAgIHJldHVybiBTZXEoW2lubmVyXSAqIG0gKyBbT3B0KGlubmVyKV0gKiAobiAtIG0pKQogICAgcmFpc2UgVHlwZUVycm9yKGYidW5oYW5kbGVkIHJlZ2V4IG5vZGUge25vZGUhcn0iKQoKCmRlZiByZWdleF90ZXJtaW5hbChwYXR0ZXJuOiBzdHIpOgogICAgYXN0LCBfYWxwaGFiZXQgPSBwYXJzZV9yZWdleF9hc3QocGF0dGVybikKICAgIHJldHVybiBfcmVnZXhfbm9kZV90b19jb21iaW5hdG9yKGFzdCkKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSAjCiMgRUJORiB0b2tlbml6ZXIKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gIwpfVE9LRU4gPSByZS5jb21waWxlKAogICAgciIiIgogICAgICAoP1A8d3M+XHMrKQogICAgfCAoP1A8Y29tbWVudD5cKFwqLio/XCpcKSkKICAgIHwgKD9QPGlkZW50PltBLVphLXpfXVtBLVphLXowLTlfXSopCiAgICB8ICg/UDxkc3RyaW5nPiIoPzpcXC58W14iXFxdKSoiKQogICAgfCAoP1A8c3N0cmluZz4nKD86XFwufFteJ1xcXSkqJykKICAgIHwgKD9QPHJlZ2V4Pi8oPzpcXC58W14vXFxdKSsvKQogICAgfCAoP1A8cHVuY3Q+Wz07fCgpXFtcXXt9Kis/LF0pCiAgICAiIiIsCiAgICByZS5WRVJCT1NFIHwgcmUuRE9UQUxMLAopCgpfRVNDQVBFUyA9IHsibiI6ICJcbiIsICJ0IjogIlx0IiwgInIiOiAiXHIiLCAnIic6ICciJywgIiciOiAiJyIsICJcXCI6ICJcXCIsICIvIjogIi8ifQoKCmRlZiBfdW5lc2NhcGUoczogc3RyKSAtPiBzdHI6CiAgICBvdXQsIGkgPSBbXSwgMAogICAgd2hpbGUgaSA8IGxlbihzKToKICAgICAgICBjID0gc1tpXQogICAgICAgIGlmIGMgPT0gIlxcIiBhbmQgaSArIDEgPCBsZW4ocyk6CiAgICAgICAgICAgIG91dC5hcHBlbmQoX0VTQ0FQRVMuZ2V0KHNbaSArIDFdLCBzW2kgKyAxXSkpCiAgICAgICAgICAgIGkgKz0gMgogICAgICAgIGVsc2U6CiAgICAgICAgICAgIG91dC5hcHBlbmQoYykKICAgICAgICAgICAgaSArPSAxCiAgICByZXR1cm4gIiIuam9pbihvdXQpCgoKZGVmIF90b2tlbml6ZSh0ZXh0OiBzdHIpIC0+IExpc3RbVHVwbGVbc3RyLCBzdHJdXToKICAgIHRva2VuczogTGlzdFtUdXBsZVtzdHIsIHN0cl1dID0gW10KICAgIHBvcyA9IDAKICAgIHdoaWxlIHBvcyA8IGxlbih0ZXh0KToKICAgICAgICBtID0gX1RPS0VOLm1hdGNoKHRleHQsIHBvcykKICAgICAgICBpZiBub3QgbToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkVCTkY6IHVuZXhwZWN0ZWQgY2hhcmFjdGVyIHt0ZXh0W3Bvc10hcn0gYXQge3Bvc30iKQogICAgICAgIHBvcyA9IG0uZW5kKCkKICAgICAgICBraW5kID0gbS5sYXN0Z3JvdXAKICAgICAgICBpZiBraW5kIGluICgid3MiLCAiY29tbWVudCIpOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHZhbCA9IG0uZ3JvdXAoKQogICAgICAgIGlmIGtpbmQgaW4gKCJkc3RyaW5nIiwgInNzdHJpbmciKToKICAgICAgICAgICAgdG9rZW5zLmFwcGVuZCgoInN0cmluZyIsIF91bmVzY2FwZSh2YWxbMTotMV0pKSkKICAgICAgICBlbGlmIGtpbmQgPT0gInJlZ2V4IjoKICAgICAgICAgICAgdG9rZW5zLmFwcGVuZCgoInJlZ2V4IiwgdmFsWzE6LTFdKSkKICAgICAgICBlbGlmIGtpbmQgPT0gImlkZW50IjoKICAgICAgICAgICAgdG9rZW5zLmFwcGVuZCgoImlkZW50IiwgdmFsKSkKICAgICAgICBlbHNlOiAgIyBwdW5jdAogICAgICAgICAgICB0b2tlbnMuYXBwZW5kKCh2YWwsIHZhbCkpCiAgICB0b2tlbnMuYXBwZW5kKCgiZW9mIiwgIiIpKQogICAgcmV0dXJuIHRva2VucwoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tICMKIyBFQk5GIHBhcnNlciAtPiBHcmFtbWFyCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tICMKY2xhc3MgX0VCTkZQYXJzZXI6CiAgICBkZWYgX19pbml0X18oc2VsZiwgdG9rZW5zOiBMaXN0W1R1cGxlW3N0ciwgc3RyXV0pOgogICAgICAgIHNlbGYudG9rcyA9IHRva2VucwogICAgICAgIHNlbGYuaSA9IDAKCiAgICBkZWYgX3BlZWsoc2VsZikgLT4gVHVwbGVbc3RyLCBzdHJdOgogICAgICAgIHJldHVybiBzZWxmLnRva3Nbc2VsZi5pXQoKICAgIGRlZiBfbmV4dChzZWxmKSAtPiBUdXBsZVtzdHIsIHN0cl06CiAgICAgICAgdCA9IHNlbGYudG9rc1tzZWxmLmldCiAgICAgICAgc2VsZi5pICs9IDEKICAgICAgICByZXR1cm4gdAoKICAgIGRlZiBfZXhwZWN0KHNlbGYsIGtpbmQ6IHN0cikgLT4gVHVwbGVbc3RyLCBzdHJdOgogICAgICAgIHQgPSBzZWxmLl9uZXh0KCkKICAgICAgICBpZiB0WzBdICE9IGtpbmQ6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJFQk5GOiBleHBlY3RlZCB7a2luZCFyfSwgZ290IHt0WzBdIXJ9ICh7dFsxXSFyfSkiKQogICAgICAgIHJldHVybiB0CgogICAgZGVmIHBhcnNlKHNlbGYpIC0+IEdyYW1tYXI6CiAgICAgICAgcnVsZXM6IERpY3Rbc3RyLCBvYmplY3RdID0ge30KICAgICAgICBzdGFydDogT3B0aW9uYWxbc3RyXSA9IE5vbmUKICAgICAgICB3aGlsZSBzZWxmLl9wZWVrKClbMF0gIT0gImVvZiI6CiAgICAgICAgICAgIG5hbWUgPSBzZWxmLl9leHBlY3QoImlkZW50IilbMV0KICAgICAgICAgICAgc2VsZi5fZXhwZWN0KCI9IikKICAgICAgICAgICAgYm9keSA9IHNlbGYuX2FsdCgpCiAgICAgICAgICAgIHNlbGYuX2V4cGVjdCgiOyIpCiAgICAgICAgICAgIHJ1bGVzW25hbWVdID0gYm9keQogICAgICAgICAgICBpZiBzdGFydCBpcyBOb25lOgogICAgICAgICAgICAgICAgc3RhcnQgPSBuYW1lCiAgICAgICAgaWYgc3RhcnQgaXMgTm9uZToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiRUJORjogbm8gcnVsZXMgZGVmaW5lZCIpCiAgICAgICAgbWlzc2luZyA9IHsKICAgICAgICAgICAgciBmb3IgYm9keSBpbiBydWxlcy52YWx1ZXMoKSBmb3IgciBpbiBfcmVmZXJlbmNlZChib2R5KSBpZiByIG5vdCBpbiBydWxlcwogICAgICAgIH0KICAgICAgICBpZiBtaXNzaW5nOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiRUJORjogdW5kZWZpbmVkIHJ1bGUocyk6IHtzb3J0ZWQobWlzc2luZyl9IikKICAgICAgICByZXR1cm4gR3JhbW1hcihydWxlcz1ydWxlcywgc3RhcnQ9c3RhcnQpCgogICAgZGVmIF9hbHQoc2VsZik6CiAgICAgICAgb3B0cyA9IFtzZWxmLl9zZXEoKV0KICAgICAgICB3aGlsZSBzZWxmLl9wZWVrKClbMF0gPT0gInwiOgogICAgICAgICAgICBzZWxmLl9uZXh0KCkKICAgICAgICAgICAgb3B0cy5hcHBlbmQoc2VsZi5fc2VxKCkpCiAgICAgICAgcmV0dXJuIG9wdHNbMF0gaWYgbGVuKG9wdHMpID09IDEgZWxzZSBBbHQob3B0cykKCiAgICBfU1RPUCA9IHsifCIsICIpIiwgIl0iLCAifSIsICI7IiwgImVvZiJ9CgogICAgZGVmIF9zZXEoc2VsZik6CiAgICAgICAgcGFydHMgPSBbXQogICAgICAgIHdoaWxlIHNlbGYuX3BlZWsoKVswXSBub3QgaW4gc2VsZi5fU1RPUDoKICAgICAgICAgICAgaWYgc2VsZi5fcGVlaygpWzBdID09ICIsIjogICMgb3B0aW9uYWwgY29uY2F0ZW5hdGlvbiBzZXBhcmF0b3IKICAgICAgICAgICAgICAgIHNlbGYuX25leHQoKQogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgcGFydHMuYXBwZW5kKHNlbGYuX3Bvc3RmaXgoKSkKICAgICAgICBpZiBub3QgcGFydHM6CiAgICAgICAgICAgIHJldHVybiBFbXB0eSgpCiAgICAgICAgcmV0dXJuIHBhcnRzWzBdIGlmIGxlbihwYXJ0cykgPT0gMSBlbHNlIFNlcShwYXJ0cykKCiAgICBkZWYgX3Bvc3RmaXgoc2VsZik6CiAgICAgICAgbm9kZSA9IHNlbGYuX2F0b20oKQogICAgICAgIHdoaWxlIHNlbGYuX3BlZWsoKVswXSBpbiAoIioiLCAiKyIsICI/Iik6CiAgICAgICAgICAgIG9wID0gc2VsZi5fbmV4dCgpWzBdCiAgICAgICAgICAgIG5vZGUgPSB7IioiOiBTdGFyLCAiKyI6IFBsdXMsICI/IjogT3B0fVtvcF0obm9kZSkKICAgICAgICByZXR1cm4gbm9kZQoKICAgIGRlZiBfYXRvbShzZWxmKToKICAgICAgICBraW5kLCB2YWwgPSBzZWxmLl9wZWVrKCkKICAgICAgICBpZiBraW5kID09ICIoIjoKICAgICAgICAgICAgc2VsZi5fbmV4dCgpCiAgICAgICAgICAgIG5vZGUgPSBzZWxmLl9hbHQoKQogICAgICAgICAgICBzZWxmLl9leHBlY3QoIikiKQogICAgICAgICAgICByZXR1cm4gbm9kZQogICAgICAgIGlmIGtpbmQgPT0gIlsiOgogICAgICAgICAgICBzZWxmLl9uZXh0KCkKICAgICAgICAgICAgbm9kZSA9IHNlbGYuX2FsdCgpCiAgICAgICAgICAgIHNlbGYuX2V4cGVjdCgiXSIpCiAgICAgICAgICAgIHJldHVybiBPcHQobm9kZSkKICAgICAgICBpZiBraW5kID09ICJ7IjoKICAgICAgICAgICAgc2VsZi5fbmV4dCgpCiAgICAgICAgICAgIG5vZGUgPSBzZWxmLl9hbHQoKQogICAgICAgICAgICBzZWxmLl9leHBlY3QoIn0iKQogICAgICAgICAgICByZXR1cm4gU3Rhcihub2RlKQogICAgICAgIGlmIGtpbmQgPT0gInN0cmluZyI6CiAgICAgICAgICAgIHNlbGYuX25leHQoKQogICAgICAgICAgICByZXR1cm4gTGl0KHZhbCkKICAgICAgICBpZiBraW5kID09ICJyZWdleCI6CiAgICAgICAgICAgIHNlbGYuX25leHQoKQogICAgICAgICAgICByZXR1cm4gcmVnZXhfdGVybWluYWwodmFsKQogICAgICAgIGlmIGtpbmQgPT0gImlkZW50IjoKICAgICAgICAgICAgc2VsZi5fbmV4dCgpCiAgICAgICAgICAgIHJldHVybiBDYWxsKHZhbCkKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiRUJORjogdW5leHBlY3RlZCB0b2tlbiB7a2luZCFyfSAoe3ZhbCFyfSkiKQoKCmRlZiBfcmVmZXJlbmNlZChub2RlKSAtPiBzZXQ6CiAgICAiIiJDb2xsZWN0IHJ1bGUgbmFtZXMgcmVmZXJlbmNlZCBieSBgYENhbGxgYCBhbnl3aGVyZSBpbiBhIGNvbWJpbmF0b3IgdHJlZS4iIiIKICAgIG91dCA9IHNldCgpCiAgICBzdGFjayA9IFtub2RlXQogICAgd2hpbGUgc3RhY2s6CiAgICAgICAgbiA9IHN0YWNrLnBvcCgpCiAgICAgICAgaWYgaXNpbnN0YW5jZShuLCBDYWxsKToKICAgICAgICAgICAgb3V0LmFkZChuLnJ1bGUpCiAgICAgICAgZWxpZiBpc2luc3RhbmNlKG4sIChTZXEsIEFsdCkpOgogICAgICAgICAgICBzdGFjay5leHRlbmQobi5wYXJ0cyBpZiBpc2luc3RhbmNlKG4sIFNlcSkgZWxzZSBuLm9wdGlvbnMpCiAgICAgICAgZWxpZiBpc2luc3RhbmNlKG4sIChPcHQsIFN0YXIsIFBsdXMpKToKICAgICAgICAgICAgc3RhY2suYXBwZW5kKG4ubm9kZSkKICAgIHJldHVybiBvdXQKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSAjCiMgUHVibGljIEFQSQojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSAjCmRlZiBwYXJzZV9lYm5mKHRleHQ6IHN0cikgLT4gR3JhbW1hcjoKICAgIHJldHVybiBfRUJORlBhcnNlcihfdG9rZW5pemUodGV4dCkpLnBhcnNlKCkKCgpkZWYgY29tcGlsZV9lYm5mKHRleHQ6IHN0cik6CiAgICAiIiJDb21waWxlIEVCTkYgc291cmNlIGludG8gYSA6Y2xhc3M6YH5lbmdpbmUucHVzaGRvd24uUERBYC4iIiIKICAgIHJldHVybiBjb21waWxlX2dyYW1tYXIocGFyc2VfZWJuZih0ZXh0KSkKCgpkZWYgbG9hZF9lYm5mKHBhdGg6IHN0cik6CiAgICB3aXRoIG9wZW4ocGF0aCkgYXMgZjoKICAgICAgICByZXR1cm4gY29tcGlsZV9lYm5mKGYucmVhZCgpKQoKCmRlZiBidWlsZF9lYm5mX3Byb2Nlc3Nvcih0ZXh0OiBzdHIsIHZvY2FiLCBpbmRleGVkOiBib29sID0gRmFsc2UsIHByZWNvbXB1dGU6IGJvb2wgPSBGYWxzZSk6CiAgICAiIiJSZXR1cm4gYSBMb2dpdHNQcm9jZXNzb3IgY29uc3RyYWluaW5nIGdlbmVyYXRpb24gdG8gYW4gRUJORiBncmFtbWFyLiIiIgogICAgZnJvbSBlbmdpbmUubWFza2VyIGltcG9ydCBBdXRvbWF0b25Mb2dpdHNQcm9jZXNzb3IKICAgIGZyb20gZW5naW5lLnRva2VuX2JyaWRnZSBpbXBvcnQgYWxsb3dlZF90b2tlbl9pZHMKCiAgICBhdXRvbWF0b24gPSBjb21waWxlX2VibmYodGV4dCkKICAgIGFsbG93ZWRfZm4gPSBhbGxvd2VkX3Rva2VuX2lkcwogICAgaWYgaW5kZXhlZDoKICAgICAgICBmcm9tIGVuZ2luZS50b2tlbl9pbmRleCBpbXBvcnQgVG9rZW5JbmRleAoKICAgICAgICBpZHggPSBUb2tlbkluZGV4KGF1dG9tYXRvbiwgdm9jYWIpCiAgICAgICAgaWYgcHJlY29tcHV0ZToKICAgICAgICAgICAgaWR4LnByZWNvbXB1dGUoKQogICAgICAgIGFsbG93ZWRfZm4gPSBpZHgKICAgIHJldHVybiBBdXRvbWF0b25Mb2dpdHNQcm9jZXNzb3IoYXV0b21hdG9uLCB2b2NhYiwgYWxsb3dlZF9mbj1hbGxvd2VkX2ZuKQo=",
"engine/streaming.py": "IiIiU3RyZWFtaW5nIG91dHB1dCB3aXRoIGVhcmx5IHZhbGlkYXRpb24gKFBoYXNlIDMpLgoKVHdvIGNvbXBsZW1lbnRhcnkgdG9vbHMgb3ZlciB0aGUgc2FtZSBhdXRvbWF0b246CgoqIDpjbGFzczpgU3RyZWFtVmFsaWRhdG9yYCDigJQgZmVlZCBhbiAqZXhpc3RpbmcqIHRva2VuL3RleHQgc3RyZWFtIGFuZCBsZWFybiwgYXQKICBlYWNoIHRva2VuLCB3aGV0aGVyIHRoZSBvdXRwdXQgaXMgc3RpbGwgb24tZ3JhbW1hciwgd2hldGhlciBpdCBpcyBhIGNvbXBsZXRlCiAgdmFsaWQgdmFsdWUgeWV0LCBhbmQgdGhlIGN1cnJlbnQgbmVzdGluZyBkZXB0aC4gVGhlIGZpcnN0IG9mZi1ncmFtbWFyIHRva2VuIGlzCiAgcmVwb3J0ZWQgaW1tZWRpYXRlbHkgKGZhaWwgZmFzdCksIHJhdGhlciB0aGFuIG9ubHkgYWZ0ZXIgdGhlIHdob2xlIHN0cmluZyBpcwogIHByb2R1Y2VkLgoKKiA6ZnVuYzpgc3RyZWFtX2NvbnN0cmFpbmVkYCDigJQgZHJpdmUgYSBtb2RlbCdzIGRlY29kZSBsb29wIHRva2VuIGJ5IHRva2VuLAogIG1hc2tpbmcgdG8gdGhlIGdyYW1tYXIgYW5kICp5aWVsZGluZyogdGhlIGdyb3dpbmcgb3V0cHV0IHBsdXMgc3RhdHVzIGFzIGl0CiAgZm9ybXMuIEJlY2F1c2UgZXZlcnkgdG9rZW4gaXMgbWFza2VkLCB0aGUgb3V0cHV0IGlzIHZhbGlkIGJ5IGNvbnN0cnVjdGlvbjsgdGhlCiAgc3RhdHVzIHRlbGxzIHlvdSB0aGUgbW9tZW50IGl0IGJlY29tZXMgYSBjb21wbGV0ZSB2YWx1ZSBhbmQgaG93IGRlZXAgdGhlCiAgc3RydWN0dXJlIGN1cnJlbnRseSBpcy4KClRoZSBjb3JlIGlzIG1vZGVsLWFnbm9zdGljIChpdCB0YWtlcyBhIGBgbG9naXRzX2ZuYGApLCBzbyBpdCBpcyB1bml0LXRlc3RlZAp3aXRob3V0IGEgbW9kZWw7IDpmdW5jOmBzdHJlYW1fZ2VuZXJhdGVfaGZgIGlzIGEgdGhpbiBIdWdnaW5nIEZhY2UgYWRhcHRlci4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKZnJvbSB0eXBpbmcgaW1wb3J0IENhbGxhYmxlLCBJdGVyYXRvciwgTGlzdCwgT3B0aW9uYWwKCmZyb20gZW5naW5lLnRva2VuX2JyaWRnZSBpbXBvcnQgVG9rZW5Wb2NhYiwgYWR2YW5jZSwgYWxsb3dlZF90b2tlbl9pZHMKCgpkZWYgX2RlcHRoKHN0YXRlKSAtPiBpbnQ6CiAgICAiIiJDdXJyZW50IG5lc3RpbmcgZGVwdGgsIGFzIHRoZSAqc2hhbGxvd2VzdCogbGl2ZSBQREEgc3RhY2sgKDAgZm9yIGEgREZBKS4KCiAgICBXaXRoIHRoZSBQREEgc2ltdWxhdGVkIGFzIGEgc2V0IG9mIGNvbmZpZ3VyYXRpb25zLCBzZXZlcmFsIHBhcnNlIHBhdGhzCiAgICBjb2V4aXN0OyB0aGUgc2hhbGxvd2VzdCBpcyB0aGUgb25lIG5lYXJlc3QgdG8gY29tcGxldGluZywgc28gaXQgaXMgMCBleGFjdGx5CiAgICB3aGVuIHRoZSBvdXRwdXQgaXMgYSBjb21wbGV0ZSB0b3AtbGV2ZWwgdmFsdWUgYW5kIGdyb3dzIGFzIHN0cnVjdHVyZSBvcGVucy4iIiIKICAgIHRyeToKICAgICAgICByZXR1cm4gbWluKChsZW4oc3RhY2spIGZvciBfbm9kZSwgc3RhY2sgaW4gc3RhdGUpLCBkZWZhdWx0PTApCiAgICBleGNlcHQgVHlwZUVycm9yOgogICAgICAgIHJldHVybiAwCgoKQGRhdGFjbGFzcwpjbGFzcyBTdHJlYW1TdGF0dXM6CiAgICB0ZXh0OiBzdHIgICAgICAgICAgIyBmdWxsIG91dHB1dCBzbyBmYXIKICAgIHBpZWNlOiBzdHIgICAgICAgICAjIHRoZSB0ZXh0IGp1c3QgYWRkZWQKICAgIGlzX2NvbXBsZXRlOiBib29sICAjIG91dHB1dCBzbyBmYXIgaXMgYSBjb21wbGV0ZSB2YWxpZCB2YWx1ZSAoYXV0b21hdG9uIGFjY2VwdGluZykKICAgIGRlcHRoOiBpbnQgICAgICAgICAjIGN1cnJlbnQgc3RydWN0dXJhbCBuZXN0aW5nIGRlcHRoCiAgICBuX3Rva2VuczogaW50CiAgICBkb25lOiBib29sID0gRmFsc2UgICMgZ2VuZXJhdGlvbiBmaW5pc2hlZCAoRU9TIG9yIGJ1ZGdldCkKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSAjCiMgVmFsaWRhdGUgYW4gZXhpc3Rpbmcgc3RyZWFtCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tICMKY2xhc3MgU3RyZWFtVmFsaWRhdG9yOgogICAgIiIiSW5jcmVtZW50YWxseSB2YWxpZGF0ZSBhIHN0cmVhbSBhZ2FpbnN0IGFuIGF1dG9tYXRvbi4KCiAgICBGZWVkIGRlY29kZWQgcGllY2VzIChwZXIgdG9rZW4sIG9yIGFueSBjaHVuaykuIFJldHVybnMgYSA6Y2xhc3M6YFN0cmVhbVN0YXR1c2AKICAgIHdoZXJlIGBgaXNfY29tcGxldGVgYCBzYXlzIHRoZSBvdXRwdXQgaXMgYSBjb21wbGV0ZSB2YWxpZCB2YWx1ZSBzbyBmYXIuIElmIGEKICAgIHBpZWNlIGxlYXZlcyB0aGUgZ3JhbW1hciwgOm1ldGg6YGZlZWRgIHJhaXNlcyA6Y2xhc3M6YFN0cmVhbVZpb2xhdGlvbmAgbmFtaW5nCiAgICB0aGUgb2ZmZW5kaW5nIHRleHQg4oCUIHRoZSBlYXJseS1yZWplY3Rpb24gc2lnbmFsLiIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBhdXRvbWF0b24pOgogICAgICAgIHNlbGYuYXV0b21hdG9uID0gYXV0b21hdG9uCiAgICAgICAgc2VsZi5zdGF0ZSA9IGF1dG9tYXRvbi5zdGFydAogICAgICAgIHNlbGYudGV4dCA9ICIiCiAgICAgICAgc2VsZi5uID0gMAoKICAgIGRlZiBmZWVkKHNlbGYsIHBpZWNlOiBzdHIpIC0+IFN0cmVhbVN0YXR1czoKICAgICAgICBzdGF0ZSA9IHNlbGYuc3RhdGUKICAgICAgICBmb3IgaSwgY2ggaW4gZW51bWVyYXRlKHBpZWNlKToKICAgICAgICAgICAgc3RhdGUgPSBzZWxmLmF1dG9tYXRvbi5zdGVwKHN0YXRlLCBjaCkKICAgICAgICAgICAgaWYgc2VsZi5hdXRvbWF0b24uaXNfZGVhZChzdGF0ZSk6CiAgICAgICAgICAgICAgICByYWlzZSBTdHJlYW1WaW9sYXRpb24oc2VsZi50ZXh0ICsgcGllY2VbOiBpICsgMV0sIHBpZWNlLCBpKQogICAgICAgIHNlbGYuc3RhdGUgPSBzdGF0ZQogICAgICAgIHNlbGYudGV4dCArPSBwaWVjZQogICAgICAgIHNlbGYubiArPSAxCiAgICAgICAgcmV0dXJuIFN0cmVhbVN0YXR1cygKICAgICAgICAgICAgdGV4dD1zZWxmLnRleHQsIHBpZWNlPXBpZWNlLAogICAgICAgICAgICBpc19jb21wbGV0ZT1zZWxmLmF1dG9tYXRvbi5pc19hY2NlcHRpbmcoc3RhdGUpLAogICAgICAgICAgICBkZXB0aD1fZGVwdGgoc3RhdGUpLCBuX3Rva2Vucz1zZWxmLm4sCiAgICAgICAgKQoKICAgIGRlZiBpc19jb21wbGV0ZShzZWxmKSAtPiBib29sOgogICAgICAgIHJldHVybiBzZWxmLmF1dG9tYXRvbi5pc19hY2NlcHRpbmcoc2VsZi5zdGF0ZSkKCgpjbGFzcyBTdHJlYW1WaW9sYXRpb24oVmFsdWVFcnJvcik6CiAgICBkZWYgX19pbml0X18oc2VsZiwgcHJlZml4OiBzdHIsIHBpZWNlOiBzdHIsIG9mZnNldDogaW50KToKICAgICAgICBzZWxmLnByZWZpeCA9IHByZWZpeAogICAgICAgIHNlbGYucGllY2UgPSBwaWVjZQogICAgICAgIHNlbGYub2Zmc2V0ID0gb2Zmc2V0CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygKICAgICAgICAgICAgZiJzdHJlYW0gbGVmdCB0aGUgZ3JhbW1hciBhdCB7cHJlZml4Wy00MDpdIXJ9ICIKICAgICAgICAgICAgZiIob2ZmZW5kaW5nIGNoYXIge3BpZWNlW29mZnNldF0hcn0gaW4gcGllY2Uge3BpZWNlIXJ9KSIKICAgICAgICApCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gIwojIENvbnN0cmFpbmVkIHN0cmVhbWluZyBnZW5lcmF0aW9uIChtb2RlbC1hZ25vc3RpYyBjb3JlKQojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSAjCmRlZiBfZ3JlZWR5KGxvZ2l0cywgYWxsb3dlZF9pZHMpIC0+IGludDoKICAgIGJlc3QsIGJlc3RfdiA9IE5vbmUsIE5vbmUKICAgIGZvciB0aWQgaW4gYWxsb3dlZF9pZHM6CiAgICAgICAgdiA9IGxvZ2l0c1t0aWRdCiAgICAgICAgaWYgYmVzdF92IGlzIE5vbmUgb3IgdiA+IGJlc3RfdjoKICAgICAgICAgICAgYmVzdCwgYmVzdF92ID0gdGlkLCB2CiAgICByZXR1cm4gYmVzdAoKCmRlZiBzdHJlYW1fY29uc3RyYWluZWQoCiAgICBsb2dpdHNfZm46IENhbGxhYmxlW1tMaXN0W2ludF1dLCAib2JqZWN0Il0sCiAgICBhdXRvbWF0b24sCiAgICB2b2NhYjogVG9rZW5Wb2NhYiwKICAgICosCiAgICBhbGxvd2VkX2ZuPU5vbmUsCiAgICBtYXhfbmV3X3Rva2VuczogaW50ID0gMjU2LAogICAgc2FtcGxlOiBPcHRpb25hbFtDYWxsYWJsZVtbIm9iamVjdCIsIHNldF0sIGludF1dID0gTm9uZSwKKSAtPiBJdGVyYXRvcltTdHJlYW1TdGF0dXNdOgogICAgIiIiWWllbGQgYSA6Y2xhc3M6YFN0cmVhbVN0YXR1c2AgYWZ0ZXIgZWFjaCBnZW5lcmF0ZWQgdG9rZW4uCgogICAgYGBsb2dpdHNfZm4oZ2VuZXJhdGVkX2lkcylgYCByZXR1cm5zIG5leHQtdG9rZW4gc2NvcmVzIChpbmRleGFibGUgYnkgaWQpLgogICAgYGBzYW1wbGUobG9naXRzLCBhbGxvd2VkX2lkcylgYCBwaWNrcyBhIHRva2VuIGlkIGZyb20gdGhlIGFsbG93ZWQgc2V0OwogICAgZGVmYXVsdHMgdG8gZ3JlZWR5IGFyZ21heCBvdmVyIHRoZSBhbGxvd2VkIHNldC4iIiIKICAgIGFsbG93ZWRfZm4gPSBhbGxvd2VkX2ZuIG9yIGFsbG93ZWRfdG9rZW5faWRzCiAgICBjaG9vc2UgPSBzYW1wbGUgb3IgX2dyZWVkeQogICAgc3RhdGUgPSBhdXRvbWF0b24uc3RhcnQKICAgIGdlbmVyYXRlZDogTGlzdFtpbnRdID0gW10KICAgIHRleHQgPSAiIgogICAgZm9yIF8gaW4gcmFuZ2UobWF4X25ld190b2tlbnMpOgogICAgICAgIGFsbG93ZWQgPSBhbGxvd2VkX2ZuKGF1dG9tYXRvbiwgc3RhdGUsIHZvY2FiKQogICAgICAgIGlmIG5vdCBhbGxvd2VkOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoInN0cmVhbV9jb25zdHJhaW5lZDogbm8gbGVnYWwgdG9rZW4gYW5kIG5vIGxlZ2FsIHN0b3AiKQogICAgICAgIGxvZ2l0cyA9IGxvZ2l0c19mbihnZW5lcmF0ZWQpCiAgICAgICAgdGlkID0gY2hvb3NlKGxvZ2l0cywgYWxsb3dlZCkKICAgICAgICBpZiB0aWQgPT0gdm9jYWIuZW9zX2lkOgogICAgICAgICAgICB5aWVsZCBTdHJlYW1TdGF0dXModGV4dD10ZXh0LCBwaWVjZT0iIiwgaXNfY29tcGxldGU9VHJ1ZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGRlcHRoPV9kZXB0aChzdGF0ZSksIG5fdG9rZW5zPWxlbihnZW5lcmF0ZWQpLCBkb25lPVRydWUpCiAgICAgICAgICAgIHJldHVybgogICAgICAgIHBpZWNlID0gdm9jYWIucGllY2VzW3RpZF0KICAgICAgICBzdGF0ZSA9IGFkdmFuY2UoYXV0b21hdG9uLCBzdGF0ZSwgcGllY2UpCiAgICAgICAgZ2VuZXJhdGVkLmFwcGVuZCh0aWQpCiAgICAgICAgdGV4dCArPSBwaWVjZQogICAgICAgIHlpZWxkIFN0cmVhbVN0YXR1cyh0ZXh0PXRleHQsIHBpZWNlPXBpZWNlLAogICAgICAgICAgICAgICAgICAgICAgICAgICBpc19jb21wbGV0ZT1hdXRvbWF0b24uaXNfYWNjZXB0aW5nKHN0YXRlKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgZGVwdGg9X2RlcHRoKHN0YXRlKSwgbl90b2tlbnM9bGVuKGdlbmVyYXRlZCksIGRvbmU9RmFsc2UpCiAgICAjIEJ1ZGdldCBleGhhdXN0ZWQuCiAgICB5aWVsZCBTdHJlYW1TdGF0dXModGV4dD10ZXh0LCBwaWVjZT0iIiwgaXNfY29tcGxldGU9YXV0b21hdG9uLmlzX2FjY2VwdGluZyhzdGF0ZSksCiAgICAgICAgICAgICAgICAgICAgICAgZGVwdGg9X2RlcHRoKHN0YXRlKSwgbl90b2tlbnM9bGVuKGdlbmVyYXRlZCksIGRvbmU9VHJ1ZSkKCgpkZWYgc3RyZWFtX2dlbmVyYXRlX2hmKG1vZGVsLCB0b2tlbml6ZXIsIHByb21wdDogc3RyLCBhdXRvbWF0b24sIHZvY2FiOiBUb2tlblZvY2FiLAogICAgICAgICAgICAgICAgICAgICAgICosIG1heF9uZXdfdG9rZW5zOiBpbnQgPSAyNTYsICoqa3cpIC0+IEl0ZXJhdG9yW1N0cmVhbVN0YXR1c106CiAgICAiIiJIdWdnaW5nIEZhY2UgYWRhcHRlcjogc3RyZWFtIGNvbnN0cmFpbmVkIGdlbmVyYXRpb24gZm9yIGBgcHJvbXB0YGAuCgogICAgQWNjZXB0cyBlaXRoZXIgYSByYXcgSEYgbW9kZWwgb3IgdGhpcyBwcm9qZWN0J3MgYGBoYXJuZXNzLk1vZGVsYGAgd3JhcHBlcgogICAgKHVud3JhcHBlZCB2aWEgaXRzIGBgbW9kZWxfaWRgYCBtYXJrZXIsIHNvIGEgcmVhbCBIRiBtb2RlbCdzIG93biBgYC5tb2RlbGBgCiAgICBhdHRyaWJ1dGUgaXMgbmV2ZXIgbWlzdGFrZW4gZm9yIHRoZSB3cmFwcGVyKS4gRGVtby1ncmFkZSBsb29wOiByZS1ydW5zIHRoZQogICAgcHJvbXB0K2dlbmVyYXRlZCBwcmVmaXggZWFjaCBzdGVwIOKAlCBmaW5lIGZvciB0aGUgc2hvcnQgb3V0cHV0cyBoZXJlLiIiIgogICAgaW1wb3J0IHRvcmNoCgogICAgaGZfbW9kZWwgPSBtb2RlbC5tb2RlbCBpZiAoaGFzYXR0cihtb2RlbCwgIm1vZGVsX2lkIikgYW5kIGhhc2F0dHIobW9kZWwsICJtb2RlbCIpKSBlbHNlIG1vZGVsCiAgICBkZXZpY2UgPSBnZXRhdHRyKGhmX21vZGVsLCAiZGV2aWNlIiwgImNwdSIpCiAgICBiYXNlID0gdG9rZW5pemVyKHByb21wdCwgcmV0dXJuX3RlbnNvcnM9InB0IikuaW5wdXRfaWRzLnRvKGRldmljZSkKCiAgICBkZWYgbG9naXRzX2ZuKGdlbmVyYXRlZF9pZHMpOgogICAgICAgIGlmIGdlbmVyYXRlZF9pZHM6CiAgICAgICAgICAgIGV4dHJhID0gdG9yY2gudGVuc29yKFtnZW5lcmF0ZWRfaWRzXSwgZGV2aWNlPWRldmljZSkKICAgICAgICAgICAgaWRzID0gdG9yY2guY2F0KFtiYXNlLCBleHRyYV0sIGRpbT0xKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGlkcyA9IGJhc2UKICAgICAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICAgICAgb3V0ID0gaGZfbW9kZWwoaWRzKQogICAgICAgIHJldHVybiBvdXQubG9naXRzWzAsIC0xXQoKICAgIHlpZWxkIGZyb20gc3RyZWFtX2NvbnN0cmFpbmVkKGxvZ2l0c19mbiwgYXV0b21hdG9uLCB2b2NhYiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG1heF9uZXdfdG9rZW5zPW1heF9uZXdfdG9rZW5zLCAqKmt3KQo=",
"bench/__init__.py": "IiIiQmVuY2htYXJrIGhhcm5lc3M6IGJhc2VsaW5lIChwcm9tcHQtb25seSAvIHByb21wdC1hbmQtcmV0cnkpIHZzLiBjb25zdHJhaW5lZApnZW5lcmF0aW9uLiBCdWlsdCBpbiBQaGFzZSAwIGFuZCBmcm96ZW4g4oCUIHJldXNlZCB1bmNoYW5nZWQgYnkgZXZlcnkgbGF0ZXIgcGhhc2UuCiIiIgo=",
"bench/metrics.py": "IiIiTWV0cmljIGhlbHBlcnMgc2hhcmVkIGJ5IGV2ZXJ5IHBoYXNlOiBkb2VzIHRoZSBvdXRwdXQgcGFyc2UsIGRvZXMgaXQgc2F0aXNmeQp0aGUgc2NoZW1hLCBhbmQgbGF0ZW5jeSBhZ2dyZWdhdGlvbi4gUHVyZSBzdGRsaWIgKyBqc29uc2NoZW1hIChubyB0b3JjaCksIHNvIGl0CmlzIHVuaXQtdGVzdGFibGUgb24gaXRzIG93bi4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQganNvbgppbXBvcnQgcmUKaW1wb3J0IHN0YXRpc3RpY3MKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBmaWVsZApmcm9tIHR5cGluZyBpbXBvcnQgTGlzdCwgT3B0aW9uYWwKCnRyeToKICAgIGltcG9ydCBqc29uc2NoZW1hCgogICAgX0hBVkVfSlNPTlNDSEVNQSA9IFRydWUKZXhjZXB0IEV4Y2VwdGlvbjogICMgcHJhZ21hOiBubyBjb3ZlcgogICAgX0hBVkVfSlNPTlNDSEVNQSA9IEZhbHNlCgoKX0ZFTkNFID0gcmUuY29tcGlsZShyImBgYCg/Ompzb24pP1xzKiguKj8pYGBgIiwgcmUuRE9UQUxMKQoKCmRlZiBleHRyYWN0X2pzb25fdGV4dCh0ZXh0OiBzdHIpIC0+IE9wdGlvbmFsW3N0cl06CiAgICAiIiJCZXN0LWVmZm9ydCBleHRyYWN0aW9uIG9mIGEgSlNPTiB2YWx1ZSBmcm9tIG1vZGVsIG91dHB1dC4KCiAgICBUcmllcywgaW4gb3JkZXI6IGEgYGBganNvbiBmZW5jZWQgYmxvY2ssIHRoZSBmaXJzdCBiYWxhbmNlZCB7Li4ufSBvciBbLi4uXQogICAgc3BhbiwgdGhlbiB0aGUgc3RyaXBwZWQgd2hvbGUgc3RyaW5nLiBUaGlzIGlzIGludGVudGlvbmFsbHkgZ2VuZXJvdXMg4oCUIGl0IGlzCiAgICB0aGUgKmJhc2VsaW5lJ3MqIGJlc3Qgc2hvdCBhdCBwYXJzaW5nLCBzbyB0aGUgcHJvbXB0LW9ubHkgbnVtYmVyIGlzIG5vdAogICAgdW5mYWlybHkgbG93LiIiIgogICAgaWYgdGV4dCBpcyBOb25lOgogICAgICAgIHJldHVybiBOb25lCiAgICBtID0gX0ZFTkNFLnNlYXJjaCh0ZXh0KQogICAgaWYgbToKICAgICAgICBpbm5lciA9IG0uZ3JvdXAoMSkuc3RyaXAoKQogICAgICAgIGlmIGlubmVyOgogICAgICAgICAgICByZXR1cm4gaW5uZXIKICAgIHNwYW4gPSBfZmlyc3RfYmFsYW5jZWRfc3Bhbih0ZXh0KQogICAgaWYgc3BhbiBpcyBub3QgTm9uZToKICAgICAgICByZXR1cm4gc3BhbgogICAgc3RyaXBwZWQgPSB0ZXh0LnN0cmlwKCkKICAgIHJldHVybiBzdHJpcHBlZCBvciBOb25lCgoKZGVmIF9maXJzdF9iYWxhbmNlZF9zcGFuKHRleHQ6IHN0cikgLT4gT3B0aW9uYWxbc3RyXToKICAgICIiIlJldHVybiB0aGUgZmlyc3QgYmFsYW5jZWQgey4uLn0vWy4uLl0gc3Vic3RyaW5nLCByZXNwZWN0aW5nIHN0cmluZ3MuIiIiCiAgICBvcGVucyA9IHsieyI6ICJ9IiwgIlsiOiAiXSJ9CiAgICBmb3Igc3RhcnQsIGNoIGluIGVudW1lcmF0ZSh0ZXh0KToKICAgICAgICBpZiBjaCBpbiBvcGVuczoKICAgICAgICAgICAgY2xvc2UgPSBvcGVuc1tjaF0KICAgICAgICAgICAgZGVwdGggPSAwCiAgICAgICAgICAgIGluX3N0ciA9IEZhbHNlCiAgICAgICAgICAgIGVzYyA9IEZhbHNlCiAgICAgICAgICAgIGZvciBqIGluIHJhbmdlKHN0YXJ0LCBsZW4odGV4dCkpOgogICAgICAgICAgICAgICAgYyA9IHRleHRbal0KICAgICAgICAgICAgICAgIGlmIGluX3N0cjoKICAgICAgICAgICAgICAgICAgICBpZiBlc2M6CiAgICAgICAgICAgICAgICAgICAgICAgIGVzYyA9IEZhbHNlCiAgICAgICAgICAgICAgICAgICAgZWxpZiBjID09ICJcXCI6CiAgICAgICAgICAgICAgICAgICAgICAgIGVzYyA9IFRydWUKICAgICAgICAgICAgICAgICAgICBlbGlmIGMgPT0gJyInOgogICAgICAgICAgICAgICAgICAgICAgICBpbl9zdHIgPSBGYWxzZQogICAgICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgICAgICBpZiBjID09ICciJzoKICAgICAgICAgICAgICAgICAgICBpbl9zdHIgPSBUcnVlCiAgICAgICAgICAgICAgICBlbGlmIGMgPT0gY2g6CiAgICAgICAgICAgICAgICAgICAgZGVwdGggKz0gMQogICAgICAgICAgICAgICAgZWxpZiBjID09IGNsb3NlOgogICAgICAgICAgICAgICAgICAgIGRlcHRoIC09IDEKICAgICAgICAgICAgICAgICAgICBpZiBkZXB0aCA9PSAwOgogICAgICAgICAgICAgICAgICAgICAgICByZXR1cm4gdGV4dFtzdGFydCA6IGogKyAxXQogICAgICAgICAgICByZXR1cm4gTm9uZSAgIyBvcGVuZWQgYnV0IG5ldmVyIGNsb3NlZAogICAgcmV0dXJuIE5vbmUKCgpkZWYgcGFyc2VfanNvbih0ZXh0OiBzdHIpOgogICAgIiIiUGFyc2UgbW9kZWwgb3V0cHV0IGludG8gYSBQeXRob24gb2JqZWN0LCBvciBOb25lIGlmIGl0IGRvZXMgbm90IHBhcnNlLiIiIgogICAgY2FuZGlkYXRlID0gZXh0cmFjdF9qc29uX3RleHQodGV4dCkKICAgIGlmIGNhbmRpZGF0ZSBpcyBOb25lOgogICAgICAgIHJldHVybiBOb25lCiAgICB0cnk6CiAgICAgICAgcmV0dXJuIGpzb24ubG9hZHMoY2FuZGlkYXRlKQogICAgZXhjZXB0IChqc29uLkpTT05EZWNvZGVFcnJvciwgVmFsdWVFcnJvcik6CiAgICAgICAgcmV0dXJuIE5vbmUKCgpkZWYgaXNfdmFsaWRfanNvbih0ZXh0OiBzdHIpIC0+IGJvb2w6CiAgICByZXR1cm4gcGFyc2VfanNvbih0ZXh0KSBpcyBub3QgTm9uZQoKCmRlZiBzYXRpc2ZpZXNfc2NoZW1hKG9iaiwgc2NoZW1hOiBkaWN0KSAtPiBib29sOgogICAgIiIiVHJ1ZSBpZiBgYG9iamBgIHZhbGlkYXRlcyBhZ2FpbnN0IGBgc2NoZW1hYGAuIFJlcXVpcmVzIGpzb25zY2hlbWEuIiIiCiAgICBpZiBvYmogaXMgTm9uZToKICAgICAgICByZXR1cm4gRmFsc2UKICAgIGlmIG5vdCBfSEFWRV9KU09OU0NIRU1BOiAgIyBwcmFnbWE6IG5vIGNvdmVyCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJqc29uc2NoZW1hIG5vdCBpbnN0YWxsZWQ7IGNhbm5vdCB2YWxpZGF0ZSBzY2hlbWEiKQogICAgdHJ5OgogICAgICAgIGpzb25zY2hlbWEudmFsaWRhdGUob2JqLCBzY2hlbWEpCiAgICAgICAgcmV0dXJuIFRydWUKICAgIGV4Y2VwdCBqc29uc2NoZW1hLlZhbGlkYXRpb25FcnJvcjoKICAgICAgICByZXR1cm4gRmFsc2UKICAgIGV4Y2VwdCBqc29uc2NoZW1hLlNjaGVtYUVycm9yIGFzIGU6ICAjIHByYWdtYTogbm8gY292ZXIKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJiYWQgc2NoZW1hOiB7ZX0iKSBmcm9tIGUKCgpAZGF0YWNsYXNzCmNsYXNzIFJ1blJlc3VsdDoKICAgICIiIk9uZSBnZW5lcmF0aW9uIGF0dGVtcHQncyBvdXRjb21lLiIiIgoKICAgIHRhc2s6IHN0cgogICAgbW9kZTogc3RyCiAgICB2YWxpZF9qc29uOiBib29sCiAgICBzY2hlbWFfdmFsaWQ6IGJvb2wKICAgIGxhdGVuY3lfczogZmxvYXQKICAgIG5ld190b2tlbnM6IGludAogICAgYXR0ZW1wdHM6IGludCA9IDEKICAgIG91dHB1dDogc3RyID0gIiIKCiAgICBAcHJvcGVydHkKICAgIGRlZiBtc19wZXJfdG9rZW4oc2VsZikgLT4gZmxvYXQ6CiAgICAgICAgcmV0dXJuIChzZWxmLmxhdGVuY3lfcyAqIDEwMDAuMCAvIHNlbGYubmV3X3Rva2VucykgaWYgc2VsZi5uZXdfdG9rZW5zIGVsc2UgMC4wCgoKQGRhdGFjbGFzcwpjbGFzcyBBZ2dyZWdhdGU6CiAgICAiIiJSb2xsZWQtdXAgbWV0cmljcyBmb3IgYSAodGFzaywgbW9kZSkgZ3JvdXAuIiIiCgogICAgdGFzazogc3RyCiAgICBtb2RlOiBzdHIKICAgIG46IGludAogICAgdmFsaWRfanNvbl9yYXRlOiBmbG9hdAogICAgc2NoZW1hX3ZhbGlkX3JhdGU6IGZsb2F0CiAgICBtZWFuX2xhdGVuY3lfczogZmxvYXQKICAgIG1lYW5fbXNfcGVyX3Rva2VuOiBmbG9hdAogICAgbWVhbl90b2tlbnNfcGVyX3M6IGZsb2F0CiAgICBtZWFuX2F0dGVtcHRzOiBmbG9hdAoKCmRlZiBhZ2dyZWdhdGUocmVzdWx0czogTGlzdFtSdW5SZXN1bHRdKSAtPiBMaXN0W0FnZ3JlZ2F0ZV06CiAgICAiIiJHcm91cCByZXN1bHRzIGJ5ICh0YXNrLCBtb2RlKSBhbmQgY29tcHV0ZSByYXRlcyArIGxhdGVuY3kuIiIiCiAgICBncm91cHM6IGRpY3QgPSB7fQogICAgZm9yIHIgaW4gcmVzdWx0czoKICAgICAgICBncm91cHMuc2V0ZGVmYXVsdCgoci50YXNrLCByLm1vZGUpLCBbXSkuYXBwZW5kKHIpCiAgICBvdXQ6IExpc3RbQWdncmVnYXRlXSA9IFtdCiAgICBmb3IgKHRhc2ssIG1vZGUpLCBycyBpbiBzb3J0ZWQoZ3JvdXBzLml0ZW1zKCkpOgogICAgICAgIG4gPSBsZW4ocnMpCiAgICAgICAgdG9rX3Blcl9zID0gWwogICAgICAgICAgICAoci5uZXdfdG9rZW5zIC8gci5sYXRlbmN5X3MpIGZvciByIGluIHJzIGlmIHIubGF0ZW5jeV9zID4gMCBhbmQgci5uZXdfdG9rZW5zCiAgICAgICAgXQogICAgICAgIG91dC5hcHBlbmQoCiAgICAgICAgICAgIEFnZ3JlZ2F0ZSgKICAgICAgICAgICAgICAgIHRhc2s9dGFzaywKICAgICAgICAgICAgICAgIG1vZGU9bW9kZSwKICAgICAgICAgICAgICAgIG49biwKICAgICAgICAgICAgICAgIHZhbGlkX2pzb25fcmF0ZT1zdW0oci52YWxpZF9qc29uIGZvciByIGluIHJzKSAvIG4sCiAgICAgICAgICAgICAgICBzY2hlbWFfdmFsaWRfcmF0ZT1zdW0oci5zY2hlbWFfdmFsaWQgZm9yIHIgaW4gcnMpIC8gbiwKICAgICAgICAgICAgICAgIG1lYW5fbGF0ZW5jeV9zPXN0YXRpc3RpY3MuZm1lYW4oci5sYXRlbmN5X3MgZm9yIHIgaW4gcnMpLAogICAgICAgICAgICAgICAgbWVhbl9tc19wZXJfdG9rZW49c3RhdGlzdGljcy5mbWVhbihyLm1zX3Blcl90b2tlbiBmb3IgciBpbiBycyBpZiByLm5ld190b2tlbnMpCiAgICAgICAgICAgICAgICBpZiBhbnkoci5uZXdfdG9rZW5zIGZvciByIGluIHJzKQogICAgICAgICAgICAgICAgZWxzZSAwLjAsCiAgICAgICAgICAgICAgICBtZWFuX3Rva2Vuc19wZXJfcz1zdGF0aXN0aWNzLmZtZWFuKHRva19wZXJfcykgaWYgdG9rX3Blcl9zIGVsc2UgMC4wLAogICAgICAgICAgICAgICAgbWVhbl9hdHRlbXB0cz1zdGF0aXN0aWNzLmZtZWFuKHIuYXR0ZW1wdHMgZm9yIHIgaW4gcnMpLAogICAgICAgICAgICApCiAgICAgICAgKQogICAgcmV0dXJuIG91dAoKCiMgQ1NWIGZpZWxkIG9yZGVyIGZvciByZXN1bHRzLyouY3N2IChzdGRsaWIgY3N2LCBubyBwYW5kYXMgZGVwZW5kZW5jeSByZXF1aXJlZCkuCkNTVl9GSUVMRFMgPSBbCiAgICAidGFzayIsCiAgICAibW9kZSIsCiAgICAibiIsCiAgICAidmFsaWRfanNvbl9yYXRlIiwKICAgICJzY2hlbWFfdmFsaWRfcmF0ZSIsCiAgICAibWVhbl9sYXRlbmN5X3MiLAogICAgIm1lYW5fbXNfcGVyX3Rva2VuIiwKICAgICJtZWFuX3Rva2Vuc19wZXJfcyIsCiAgICAibWVhbl9hdHRlbXB0cyIsCl0KCgpkZWYgd3JpdGVfY3N2KHBhdGg6IHN0ciwgYWdnczogTGlzdFtBZ2dyZWdhdGVdKSAtPiBOb25lOgogICAgaW1wb3J0IGNzdgoKICAgIHdpdGggb3BlbihwYXRoLCAidyIsIG5ld2xpbmU9IiIpIGFzIGY6CiAgICAgICAgdyA9IGNzdi5EaWN0V3JpdGVyKGYsIGZpZWxkbmFtZXM9Q1NWX0ZJRUxEUykKICAgICAgICB3LndyaXRlaGVhZGVyKCkKICAgICAgICBmb3IgYSBpbiBhZ2dzOgogICAgICAgICAgICB3LndyaXRlcm93KHtrOiBnZXRhdHRyKGEsIGspIGZvciBrIGluIENTVl9GSUVMRFN9KQo=",
"bench/schemas.py": "IiIiVGhlIHZhcmllZCB0ZXN0IHNldC4KCmBgUkVHRVhfVEFTS1NgYCBkcml2ZSBQaGFzZSAxIChmaW5pdGUtYXV0b21hdG9uIGNvbnN0cmFpbnRzOiBkYXRlcywgZW51bXMsCmZpeGVkIGZvcm1hdHMpLiBgYEpTT05fU0NIRU1BU2BgIGRyaXZlIHRoZSBQaGFzZSAwIGJhc2VsaW5lIGFuZCBQaGFzZSAyCmNvbnN0cmFpbmVkIEpTT04gZ2VuZXJhdGlvbiwgc3Bhbm5pbmcgZmxhdCBvYmplY3RzLCBuZXN0ZWQgb2JqZWN0cywgYXJyYXlzLAplbnVtcywgYW5kIG9wdGlvbmFsIGZpZWxkcyDigJQgdGhlIGNvbXBsZXhpdHkgZ3JhZGllbnQgdGhlIGd1aWRlIGNhbGxzIGZvci4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MsIGZpZWxkCmZyb20gdHlwaW5nIGltcG9ydCBEaWN0LCBMaXN0LCBPcHRpb25hbAoKCkBkYXRhY2xhc3MKY2xhc3MgUmVnZXhUYXNrOgogICAgbmFtZTogc3RyCiAgICBwYXR0ZXJuOiBzdHIKICAgIGluc3RydWN0aW9uOiBzdHIKCgpAZGF0YWNsYXNzCmNsYXNzIEpzb25UYXNrOgogICAgbmFtZTogc3RyCiAgICBzY2hlbWE6IGRpY3QKICAgIGluc3RydWN0aW9uOiBzdHIKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSAjCiMgUGhhc2UgMTogcmVnZXggLyBmaW5pdGUtYXV0b21hdG9uIHRhc2tzCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tICMKUkVHRVhfVEFTS1M6IExpc3RbUmVnZXhUYXNrXSA9IFsKICAgIFJlZ2V4VGFzaygKICAgICAgICBuYW1lPSJpc29fZGF0ZSIsCiAgICAgICAgcGF0dGVybj1yIlxkezR9LVxkezJ9LVxkezJ9IiwKICAgICAgICBpbnN0cnVjdGlvbj0iR2l2ZSB0aGUgZGF0ZSA2IE5vdmVtYmVyIDIwMjYgaW4gSVNPIGZvcm1hdCBZWVlZLU1NLURELiAiCiAgICAgICAgIlJlcGx5IHdpdGggb25seSB0aGUgZGF0ZS4iLAogICAgKSwKICAgIFJlZ2V4VGFzaygKICAgICAgICBuYW1lPSJlbnVtX2NvbG9yIiwKICAgICAgICBwYXR0ZXJuPXIiKHJlZHxncmVlbnxibHVlKSIsCiAgICAgICAgaW5zdHJ1Y3Rpb249IlBpY2sgYSBwcmltYXJ5IGFkZGl0aXZlIGNvbG9yIGZyb20gcmVkLCBncmVlbiwgb3IgYmx1ZS4gIgogICAgICAgICJSZXBseSB3aXRoIG9ubHkgdGhlIHNpbmdsZSBsb3dlcmNhc2Ugd29yZC4iLAogICAgKSwKICAgIFJlZ2V4VGFzaygKICAgICAgICBuYW1lPSJ1c19waG9uZSIsCiAgICAgICAgcGF0dGVybj1yIlwoXGR7M31cKSBcZHszfS1cZHs0fSIsCiAgICAgICAgaW5zdHJ1Y3Rpb249IlByb2R1Y2UgYSBVUyBwaG9uZSBudW1iZXIgaW4gdGhlIGZvcm1hdCAoMTIzKSA0NTYtNzg5MC4gIgogICAgICAgICJSZXBseSB3aXRoIG9ubHkgdGhlIG51bWJlci4iLAogICAgKSwKICAgIFJlZ2V4VGFzaygKICAgICAgICBuYW1lPSJzZW12ZXIiLAogICAgICAgIHBhdHRlcm49ciJcZCtcLlxkK1wuXGQrIiwKICAgICAgICBpbnN0cnVjdGlvbj0iR2l2ZSBhIHNlbWFudGljIHZlcnNpb24gbnVtYmVyIGxpa2UgMi41LjEuIFJlcGx5IHdpdGggb25seSAiCiAgICAgICAgInRoZSB2ZXJzaW9uLiIsCiAgICApLAogICAgUmVnZXhUYXNrKAogICAgICAgIG5hbWU9ImhleF9jb2xvciIsCiAgICAgICAgcGF0dGVybj1yIiNbMC05YS1mQS1GXXs2fSIsCiAgICAgICAgaW5zdHJ1Y3Rpb249IkdpdmUgYSBoZXggY29sb3IgY29kZSBmb3IgcHVyZSByZWQsIGxpa2UgI1JSR0dCQi4gUmVwbHkgIgogICAgICAgICJ3aXRoIG9ubHkgdGhlIGNvZGUuIiwKICAgICksCiAgICBSZWdleFRhc2soCiAgICAgICAgbmFtZT0ieWVzX25vIiwKICAgICAgICBwYXR0ZXJuPXIiKHllc3xubykiLAogICAgICAgIGluc3RydWN0aW9uPSJJcyAxNyBhIHByaW1lIG51bWJlcj8gUmVwbHkgd2l0aCBvbmx5ICd5ZXMnIG9yICdubycuIiwKICAgICksCl0KCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSAjCiMgUGhhc2UgMCAvIFBoYXNlIDI6IEpTT04tc2NoZW1hIHRhc2tzLCBpbmNyZWFzaW5nIGluIGNvbXBsZXhpdHkKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gIwpKU09OX1NDSEVNQVM6IExpc3RbSnNvblRhc2tdID0gWwogICAgSnNvblRhc2soCiAgICAgICAgbmFtZT0iZmxhdF9vYmplY3QiLAogICAgICAgIHNjaGVtYT17CiAgICAgICAgICAgICJ0eXBlIjogIm9iamVjdCIsCiAgICAgICAgICAgICJwcm9wZXJ0aWVzIjogewogICAgICAgICAgICAgICAgIm5hbWUiOiB7InR5cGUiOiAic3RyaW5nIn0sCiAgICAgICAgICAgICAgICAiYWdlIjogeyJ0eXBlIjogImludGVnZXIifSwKICAgICAgICAgICAgfSwKICAgICAgICAgICAgInJlcXVpcmVkIjogWyJuYW1lIiwgImFnZSJdLAogICAgICAgICAgICAiYWRkaXRpb25hbFByb3BlcnRpZXMiOiBGYWxzZSwKICAgICAgICB9LAogICAgICAgIGluc3RydWN0aW9uPSJSZXR1cm4gSlNPTiBmb3IgYSBwZXJzb24gbmFtZWQgQWRhIHdobyBpcyAzNiwgd2l0aCBrZXlzICIKICAgICAgICAiJ25hbWUnIChzdHJpbmcpIGFuZCAnYWdlJyAoaW50ZWdlcikuIiwKICAgICksCiAgICBKc29uVGFzaygKICAgICAgICBuYW1lPSJlbnVtX2ZpZWxkIiwKICAgICAgICBzY2hlbWE9ewogICAgICAgICAgICAidHlwZSI6ICJvYmplY3QiLAogICAgICAgICAgICAicHJvcGVydGllcyI6IHsKICAgICAgICAgICAgICAgICJzdGF0dXMiOiB7InR5cGUiOiAic3RyaW5nIiwgImVudW0iOiBbImFjdGl2ZSIsICJpbmFjdGl2ZSIsICJwZW5kaW5nIl19LAogICAgICAgICAgICAgICAgImlkIjogeyJ0eXBlIjogImludGVnZXIifSwKICAgICAgICAgICAgfSwKICAgICAgICAgICAgInJlcXVpcmVkIjogWyJzdGF0dXMiLCAiaWQiXSwKICAgICAgICAgICAgImFkZGl0aW9uYWxQcm9wZXJ0aWVzIjogRmFsc2UsCiAgICAgICAgfSwKICAgICAgICBpbnN0cnVjdGlvbj0iUmV0dXJuIEpTT04gd2l0aCAnc3RhdHVzJyAob25lIG9mIGFjdGl2ZSwgaW5hY3RpdmUsICIKICAgICAgICAicGVuZGluZykgYW5kICdpZCcgKGludGVnZXIpLiBVc2Ugc3RhdHVzIGFjdGl2ZSBhbmQgaWQgNy4iLAogICAgKSwKICAgIEpzb25UYXNrKAogICAgICAgIG5hbWU9ImFycmF5X29mX3N0cmluZ3MiLAogICAgICAgIHNjaGVtYT17CiAgICAgICAgICAgICJ0eXBlIjogIm9iamVjdCIsCiAgICAgICAgICAgICJwcm9wZXJ0aWVzIjogewogICAgICAgICAgICAgICAgInRhZ3MiOiB7InR5cGUiOiAiYXJyYXkiLCAiaXRlbXMiOiB7InR5cGUiOiAic3RyaW5nIn19LAogICAgICAgICAgICB9LAogICAgICAgICAgICAicmVxdWlyZWQiOiBbInRhZ3MiXSwKICAgICAgICAgICAgImFkZGl0aW9uYWxQcm9wZXJ0aWVzIjogRmFsc2UsCiAgICAgICAgfSwKICAgICAgICBpbnN0cnVjdGlvbj0iUmV0dXJuIEpTT04gd2l0aCBhICd0YWdzJyBhcnJheSBvZiB0aHJlZSBzdHJpbmcgdGFncyBmb3IgYSAiCiAgICAgICAgImNvb2tpbmcgYmxvZy4iLAogICAgKSwKICAgIEpzb25UYXNrKAogICAgICAgIG5hbWU9Im9wdGlvbmFsX2ZpZWxkIiwKICAgICAgICBzY2hlbWE9ewogICAgICAgICAgICAidHlwZSI6ICJvYmplY3QiLAogICAgICAgICAgICAicHJvcGVydGllcyI6IHsKICAgICAgICAgICAgICAgICJ0aXRsZSI6IHsidHlwZSI6ICJzdHJpbmcifSwKICAgICAgICAgICAgICAgICJzdWJ0aXRsZSI6IHsidHlwZSI6ICJzdHJpbmcifSwKICAgICAgICAgICAgfSwKICAgICAgICAgICAgInJlcXVpcmVkIjogWyJ0aXRsZSJdLAogICAgICAgICAgICAiYWRkaXRpb25hbFByb3BlcnRpZXMiOiBGYWxzZSwKICAgICAgICB9LAogICAgICAgIGluc3RydWN0aW9uPSJSZXR1cm4gSlNPTiB3aXRoIGEgcmVxdWlyZWQgJ3RpdGxlJyAoc3RyaW5nKSBhbmQgYW4gIgogICAgICAgICJvcHRpb25hbCAnc3VidGl0bGUnIChzdHJpbmcpIGZvciBhIGJvb2sgY2FsbGVkIER1bmUuIiwKICAgICksCiAgICBKc29uVGFzaygKICAgICAgICBuYW1lPSJuZXN0ZWRfb2JqZWN0IiwKICAgICAgICBzY2hlbWE9ewogICAgICAgICAgICAidHlwZSI6ICJvYmplY3QiLAogICAgICAgICAgICAicHJvcGVydGllcyI6IHsKICAgICAgICAgICAgICAgICJ1c2VyIjogewogICAgICAgICAgICAgICAgICAgICJ0eXBlIjogIm9iamVjdCIsCiAgICAgICAgICAgICAgICAgICAgInByb3BlcnRpZXMiOiB7CiAgICAgICAgICAgICAgICAgICAgICAgICJuYW1lIjogeyJ0eXBlIjogInN0cmluZyJ9LAogICAgICAgICAgICAgICAgICAgICAgICAiYWN0aXZlIjogeyJ0eXBlIjogImJvb2xlYW4ifSwKICAgICAgICAgICAgICAgICAgICB9LAogICAgICAgICAgICAgICAgICAgICJyZXF1aXJlZCI6IFsibmFtZSIsICJhY3RpdmUiXSwKICAgICAgICAgICAgICAgICAgICAiYWRkaXRpb25hbFByb3BlcnRpZXMiOiBGYWxzZSwKICAgICAgICAgICAgICAgIH0sCiAgICAgICAgICAgICAgICAic2NvcmUiOiB7InR5cGUiOiAibnVtYmVyIn0sCiAgICAgICAgICAgIH0sCiAgICAgICAgICAgICJyZXF1aXJlZCI6IFsidXNlciIsICJzY29yZSJdLAogICAgICAgICAgICAiYWRkaXRpb25hbFByb3BlcnRpZXMiOiBGYWxzZSwKICAgICAgICB9LAogICAgICAgIGluc3RydWN0aW9uPSJSZXR1cm4gSlNPTiB3aXRoIGEgbmVzdGVkICd1c2VyJyBvYmplY3QgKGtleXMgJ25hbWUnIHN0cmluZywgIgogICAgICAgICInYWN0aXZlJyBib29sZWFuKSBhbmQgYSB0b3AtbGV2ZWwgJ3Njb3JlJyBudW1iZXIuIFVzZSBuYW1lIEdyYWNlLCAiCiAgICAgICAgImFjdGl2ZSB0cnVlLCBzY29yZSA5LjUuIiwKICAgICksCiAgICBKc29uVGFzaygKICAgICAgICBuYW1lPSJhcnJheV9vZl9vYmplY3RzIiwKICAgICAgICBzY2hlbWE9ewogICAgICAgICAgICAidHlwZSI6ICJvYmplY3QiLAogICAgICAgICAgICAicHJvcGVydGllcyI6IHsKICAgICAgICAgICAgICAgICJpdGVtcyI6IHsKICAgICAgICAgICAgICAgICAgICAidHlwZSI6ICJhcnJheSIsCiAgICAgICAgICAgICAgICAgICAgIml0ZW1zIjogewogICAgICAgICAgICAgICAgICAgICAgICAidHlwZSI6ICJvYmplY3QiLAogICAgICAgICAgICAgICAgICAgICAgICAicHJvcGVydGllcyI6IHsKICAgICAgICAgICAgICAgICAgICAgICAgICAgICJza3UiOiB7InR5cGUiOiAic3RyaW5nIn0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAicXR5IjogeyJ0eXBlIjogImludGVnZXIifSwKICAgICAgICAgICAgICAgICAgICAgICAgfSwKICAgICAgICAgICAgICAgICAgICAgICAgInJlcXVpcmVkIjogWyJza3UiLCAicXR5Il0sCiAgICAgICAgICAgICAgICAgICAgICAgICJhZGRpdGlvbmFsUHJvcGVydGllcyI6IEZhbHNlLAogICAgICAgICAgICAgICAgICAgIH0sCiAgICAgICAgICAgICAgICB9LAogICAgICAgICAgICB9LAogICAgICAgICAgICAicmVxdWlyZWQiOiBbIml0ZW1zIl0sCiAgICAgICAgICAgICJhZGRpdGlvbmFsUHJvcGVydGllcyI6IEZhbHNlLAogICAgICAgIH0sCiAgICAgICAgaW5zdHJ1Y3Rpb249IlJldHVybiBKU09OIHdpdGggYW4gJ2l0ZW1zJyBhcnJheSBvZiB0d28gb2JqZWN0cywgZWFjaCB3aXRoICIKICAgICAgICAiJ3NrdScgKHN0cmluZykgYW5kICdxdHknIChpbnRlZ2VyKS4iLAogICAgKSwKXQo=",
"bench/harness.py": "IiIiQmVuY2htYXJrIHJ1bm5lci4KCk1vZGVzCiAgYmFzZWxpbmUgICAgIHByb21wdC1vbmx5LCB1bmNvbnN0cmFpbmVkIGdlbmVyYXRpb24KICByZXRyeSAgICAgICAgcHJvbXB0LWFuZC1yZXRyeTogdXAgdG8gTiBhdHRlbXB0cywgc3RvcCBhdCBmaXJzdCBzY2hlbWEtdmFsaWQgb25lCiAgY29uc3RyYWluZWQgIGdlbmVyYXRpb24gd2l0aCB0aGUgZW5naW5lJ3MgTG9naXRzUHJvY2Vzc29yICh2YWxpZCBieSBjb25zdHJ1Y3Rpb24pCgpHcmFtbWFycwogIHJlZ2V4ICAgICAgICBQaGFzZS0xIGZpbml0ZS1hdXRvbWF0b24gdGFza3MgKFJFR0VYX1RBU0tTKQogIGpzb24gICAgICAgICBKU09OLXNjaGVtYSB0YXNrcyAoSlNPTl9TQ0hFTUFTKTsgY29uc3RyYWluZWQgcGF0aCBuZWVkcyBQaGFzZSAyCgpFdmVyeXRoaW5nIGlzIGRldmljZS1hZ25vc3RpYyAoQ1BVIGxvY2FsbHk7IGEgZnJlZSBDb2xhYi9LYWdnbGUgVDQgZm9yIGxhdGVuY3kpLgpUaGUgaGFybmVzcyBpcyBmcm96ZW4gYWZ0ZXIgUGhhc2UgMCBhbmQgcmV1c2VkIHVuY2hhbmdlZCBieSBsYXRlciBwaGFzZXMuCgpVc2FnZToKICBweXRob24gLW0gYmVuY2guaGFybmVzcyAtLW1vZGUgYmFzZWxpbmUgICAgLS1ncmFtbWFyIGpzb24gIC0tc2FtcGxlcyA1CiAgcHl0aG9uIC1tIGJlbmNoLmhhcm5lc3MgLS1tb2RlIGNvbnN0cmFpbmVkIC0tZ3JhbW1hciByZWdleCAtLXNhbXBsZXMgNQoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgb3MKaW1wb3J0IHRpbWUKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmZyb20gdHlwaW5nIGltcG9ydCBMaXN0LCBPcHRpb25hbAoKZnJvbSBiZW5jaCBpbXBvcnQgbWV0cmljcwpmcm9tIGJlbmNoLm1ldHJpY3MgaW1wb3J0IFJ1blJlc3VsdApmcm9tIGJlbmNoLnNjaGVtYXMgaW1wb3J0IEpTT05fU0NIRU1BUywgUkVHRVhfVEFTS1MsIEpzb25UYXNrLCBSZWdleFRhc2sKZnJvbSBlbmdpbmUgaW1wb3J0IGNvbmZpZwoKUkVTVUxUU19ESVIgPSBvcy5wYXRoLmpvaW4ob3MucGF0aC5kaXJuYW1lKG9zLnBhdGguZGlybmFtZShfX2ZpbGVfXykpLCAicmVzdWx0cyIpCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gIwojIE1vZGVsIGxvYWRpbmcgLyBnZW5lcmF0aW9uCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tICMKQGRhdGFjbGFzcwpjbGFzcyBNb2RlbDoKICAgIG1vZGVsOiBvYmplY3QKICAgIHRva2VuaXplcjogb2JqZWN0CiAgICBkZXZpY2U6IHN0cgogICAgbW9kZWxfaWQ6IHN0cgoKCmRlZiBsb2FkX21vZGVsKG1vZGVsX2lkOiBPcHRpb25hbFtzdHJdID0gTm9uZSwgZGV2aWNlOiBPcHRpb25hbFtzdHJdID0gTm9uZSkgLT4gTW9kZWw6CiAgICBpbXBvcnQgdG9yY2gKICAgIGZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCBBdXRvTW9kZWxGb3JDYXVzYWxMTSwgQXV0b1Rva2VuaXplcgoKICAgIG1vZGVsX2lkID0gbW9kZWxfaWQgb3IgY29uZmlnLkRFRkFVTFRfTU9ERUwKICAgIGRldmljZSA9IGRldmljZSBvciBjb25maWcucGlja19kZXZpY2UoKQogICAgdG9rID0gQXV0b1Rva2VuaXplci5mcm9tX3ByZXRyYWluZWQobW9kZWxfaWQpCiAgICBkdHlwZSA9IHRvcmNoLmZsb2F0MzIgaWYgZGV2aWNlID09ICJjcHUiIGVsc2UgdG9yY2guZmxvYXQxNgogICAgbW9kZWwgPSBBdXRvTW9kZWxGb3JDYXVzYWxMTS5mcm9tX3ByZXRyYWluZWQobW9kZWxfaWQsIHRvcmNoX2R0eXBlPWR0eXBlKQogICAgbW9kZWwudG8oZGV2aWNlKQogICAgbW9kZWwuZXZhbCgpCiAgICBpZiB0b2sucGFkX3Rva2VuX2lkIGlzIE5vbmU6CiAgICAgICAgdG9rLnBhZF90b2tlbiA9IHRvay5lb3NfdG9rZW4KICAgIHJldHVybiBNb2RlbChtb2RlbD1tb2RlbCwgdG9rZW5pemVyPXRvaywgZGV2aWNlPWRldmljZSwgbW9kZWxfaWQ9bW9kZWxfaWQpCgoKZGVmIF9idWlsZF9wcm9tcHQodG9rLCBzeXN0ZW06IHN0ciwgaW5zdHJ1Y3Rpb246IHN0cikgLT4gc3RyOgogICAgbWVzc2FnZXMgPSBbeyJyb2xlIjogInN5c3RlbSIsICJjb250ZW50Ijogc3lzdGVtfSwgeyJyb2xlIjogInVzZXIiLCAiY29udGVudCI6IGluc3RydWN0aW9ufV0KICAgIHJldHVybiB0b2suYXBwbHlfY2hhdF90ZW1wbGF0ZShtZXNzYWdlcywgdG9rZW5pemU9RmFsc2UsIGFkZF9nZW5lcmF0aW9uX3Byb21wdD1UcnVlKQoKCmRlZiBnZW5lcmF0ZSgKICAgIG06IE1vZGVsLAogICAgcHJvbXB0OiBzdHIsCiAgICBtYXhfbmV3X3Rva2VuczogaW50LAogICAgZG9fc2FtcGxlOiBib29sLAogICAgdGVtcGVyYXR1cmU6IGZsb2F0LAogICAgc2VlZDogT3B0aW9uYWxbaW50XSwKICAgIGxvZ2l0c19wcm9jZXNzb3I9Tm9uZSwKKToKICAgICIiIkdlbmVyYXRlIG9uY2UuIFJldHVybnMgKGRlY29kZWRfdGV4dCwgbl9uZXdfdG9rZW5zLCBsYXRlbmN5X3NlY29uZHMpLiIiIgogICAgaW1wb3J0IHRvcmNoCgogICAgaWYgc2VlZCBpcyBub3QgTm9uZToKICAgICAgICB0b3JjaC5tYW51YWxfc2VlZChzZWVkKQogICAgZW5jID0gbS50b2tlbml6ZXIocHJvbXB0LCByZXR1cm5fdGVuc29ycz0icHQiKS50byhtLmRldmljZSkKICAgIGdlbl9rd2FyZ3MgPSBkaWN0KAogICAgICAgIG1heF9uZXdfdG9rZW5zPW1heF9uZXdfdG9rZW5zLAogICAgICAgIGRvX3NhbXBsZT1kb19zYW1wbGUsCiAgICAgICAgcGFkX3Rva2VuX2lkPW0udG9rZW5pemVyLnBhZF90b2tlbl9pZCwKICAgICkKICAgIGlmIGRvX3NhbXBsZToKICAgICAgICBnZW5fa3dhcmdzWyJ0ZW1wZXJhdHVyZSJdID0gdGVtcGVyYXR1cmUKICAgIGlmIGxvZ2l0c19wcm9jZXNzb3IgaXMgbm90IE5vbmU6CiAgICAgICAgZnJvbSB0cmFuc2Zvcm1lcnMgaW1wb3J0IExvZ2l0c1Byb2Nlc3Nvckxpc3QKCiAgICAgICAgaWYgaGFzYXR0cihsb2dpdHNfcHJvY2Vzc29yLCAicmVzZXQiKToKICAgICAgICAgICAgbG9naXRzX3Byb2Nlc3Nvci5yZXNldCgpCiAgICAgICAgZ2VuX2t3YXJnc1sibG9naXRzX3Byb2Nlc3NvciJdID0gTG9naXRzUHJvY2Vzc29yTGlzdChbbG9naXRzX3Byb2Nlc3Nvcl0pCgogICAgdDAgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICBvdXQgPSBtLm1vZGVsLmdlbmVyYXRlKCoqZW5jLCAqKmdlbl9rd2FyZ3MpCiAgICBsYXRlbmN5ID0gdGltZS5wZXJmX2NvdW50ZXIoKSAtIHQwCgogICAgaW5fbGVuID0gZW5jWyJpbnB1dF9pZHMiXS5zaGFwZVsxXQogICAgbmV3X2lkcyA9IG91dFswXVtpbl9sZW46XQogICAgdGV4dCA9IG0udG9rZW5pemVyLmRlY29kZShuZXdfaWRzLCBza2lwX3NwZWNpYWxfdG9rZW5zPVRydWUpCiAgICByZXR1cm4gdGV4dCwgaW50KG5ld19pZHMuc2hhcGVbMF0pLCBsYXRlbmN5CgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gIwojIFRhc2sgcnVubmVycwojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSAjCl9SRUdFWF9TWVNURU0gPSAiWW91IG91dHB1dCBvbmx5IHRoZSByZXF1ZXN0ZWQgdmFsdWUgaW4gdGhlIGV4YWN0IGZvcm1hdCBhc2tlZCwgd2l0aCBubyBleHBsYW5hdGlvbiwgbGFiZWxzLCBvciBleHRyYSB0ZXh0LiIKX0pTT05fU1lTVEVNID0gIllvdSBvdXRwdXQgb25seSBhIHNpbmdsZSBKU09OIHZhbHVlIG1hdGNoaW5nIHRoZSB1c2VyJ3MgcmVxdWVzdC4gTm8gcHJvc2UsIG5vIG1hcmtkb3duIGZlbmNlcywgbm8gZXhwbGFuYXRpb24uIgoKCmRlZiBydW5fcmVnZXgobTogTW9kZWwsIHRhc2tzOiBMaXN0W1JlZ2V4VGFza10sIG1vZGU6IHN0ciwgc2FtcGxlczogaW50LCBtYXhfbmV3X3Rva2VuczogaW50KSAtPiBMaXN0W1J1blJlc3VsdF06CiAgICBpbXBvcnQgcmUgYXMgX3JlCgogICAgZnJvbSBlbmdpbmUucmVnZXhfY29tcGlsZXIgaW1wb3J0IGNvbXBpbGVfcmVnZXgKICAgIGZyb20gZW5naW5lLnRva2VuX2JyaWRnZSBpbXBvcnQgVG9rZW5Wb2NhYgogICAgZnJvbSBlbmdpbmUubWFza2VyIGltcG9ydCBBdXRvbWF0b25Mb2dpdHNQcm9jZXNzb3IKCiAgICB2b2NhYiA9IFRva2VuVm9jYWIuZnJvbV9oZihtLnRva2VuaXplcikgaWYgbW9kZSA9PSAiY29uc3RyYWluZWQiIGVsc2UgTm9uZQogICAgcmVzdWx0czogTGlzdFtSdW5SZXN1bHRdID0gW10KICAgIGZvciB0YXNrIGluIHRhc2tzOgogICAgICAgIHByb21wdCA9IF9idWlsZF9wcm9tcHQobS50b2tlbml6ZXIsIF9SRUdFWF9TWVNURU0sIHRhc2suaW5zdHJ1Y3Rpb24pCiAgICAgICAgcHJvYyA9IE5vbmUKICAgICAgICBpZiBtb2RlID09ICJjb25zdHJhaW5lZCI6CiAgICAgICAgICAgIHByb2MgPSBBdXRvbWF0b25Mb2dpdHNQcm9jZXNzb3IoY29tcGlsZV9yZWdleCh0YXNrLnBhdHRlcm4pLCB2b2NhYikKICAgICAgICBwYXQgPSBfcmUuY29tcGlsZSh0YXNrLnBhdHRlcm4pCiAgICAgICAgZm9yIHMgaW4gcmFuZ2Uoc2FtcGxlcyk6CiAgICAgICAgICAgIHRleHQsIG5fbmV3LCBsYXQgPSBnZW5lcmF0ZSgKICAgICAgICAgICAgICAgIG0sIHByb21wdCwgbWF4X25ld190b2tlbnMsCiAgICAgICAgICAgICAgICBkb19zYW1wbGU9KHNhbXBsZXMgPiAxKSwgdGVtcGVyYXR1cmU9MC43LCBzZWVkPTEwMDAgKyBzLAogICAgICAgICAgICAgICAgbG9naXRzX3Byb2Nlc3Nvcj1wcm9jLAogICAgICAgICAgICApCiAgICAgICAgICAgIHZhbGlkID0gcGF0LmZ1bGxtYXRjaCh0ZXh0LnN0cmlwKCkpIGlzIG5vdCBOb25lCiAgICAgICAgICAgIHJlc3VsdHMuYXBwZW5kKFJ1blJlc3VsdCgKICAgICAgICAgICAgICAgIHRhc2s9dGFzay5uYW1lLCBtb2RlPW1vZGUsIHZhbGlkX2pzb249dmFsaWQsIHNjaGVtYV92YWxpZD12YWxpZCwKICAgICAgICAgICAgICAgIGxhdGVuY3lfcz1sYXQsIG5ld190b2tlbnM9bl9uZXcsIGF0dGVtcHRzPTEsIG91dHB1dD10ZXh0LnN0cmlwKCksCiAgICAgICAgICAgICkpCiAgICByZXR1cm4gcmVzdWx0cwoKCmRlZiBydW5fanNvbihtOiBNb2RlbCwgdGFza3M6IExpc3RbSnNvblRhc2tdLCBtb2RlOiBzdHIsIHNhbXBsZXM6IGludCwgbWF4X25ld190b2tlbnM6IGludCwgcmV0cmllczogaW50KSAtPiBMaXN0W1J1blJlc3VsdF06CiAgICBpZiBtb2RlID09ICJjb25zdHJhaW5lZCI6CiAgICAgICAgdHJ5OgogICAgICAgICAgICBmcm9tIGVuZ2luZS5qc29uX3NjaGVtYSBpbXBvcnQgYnVpbGRfanNvbl9wcm9jZXNzb3IgICMgUGhhc2UgMgogICAgICAgICAgICBmcm9tIGVuZ2luZS50b2tlbl9icmlkZ2UgaW1wb3J0IFRva2VuVm9jYWIKICAgICAgICBleGNlcHQgSW1wb3J0RXJyb3I6CiAgICAgICAgICAgIHJhaXNlIE5vdEltcGxlbWVudGVkRXJyb3IoCiAgICAgICAgICAgICAgICAiY29uc3RyYWluZWQgSlNPTiBuZWVkcyBQaGFzZSAyIChlbmdpbmUvanNvbl9zY2hlbWEucHkpLiAiCiAgICAgICAgICAgICAgICAiQmFzZWxpbmUgYW5kIHJldHJ5IG1vZGVzIHdvcmsgbm93OyByZWdleCBjb25zdHJhaW5lZCB3b3JrcyBub3cuIgogICAgICAgICAgICApCiAgICAgICAgdm9jYWIgPSBUb2tlblZvY2FiLmZyb21faGYobS50b2tlbml6ZXIpCgogICAgcmVzdWx0czogTGlzdFtSdW5SZXN1bHRdID0gW10KICAgIGZvciB0YXNrIGluIHRhc2tzOgogICAgICAgIHByb21wdCA9IF9idWlsZF9wcm9tcHQobS50b2tlbml6ZXIsIF9KU09OX1NZU1RFTSwgdGFzay5pbnN0cnVjdGlvbikKICAgICAgICBwcm9jID0gTm9uZQogICAgICAgIGlmIG1vZGUgPT0gImNvbnN0cmFpbmVkIjoKICAgICAgICAgICAgcHJvYyA9IGJ1aWxkX2pzb25fcHJvY2Vzc29yKHRhc2suc2NoZW1hLCB2b2NhYikKICAgICAgICBmb3IgcyBpbiByYW5nZShzYW1wbGVzKToKICAgICAgICAgICAgYXR0ZW1wdHMgPSAxCiAgICAgICAgICAgIG1heF9hdHRlbXB0cyA9IHJldHJpZXMgaWYgbW9kZSA9PSAicmV0cnkiIGVsc2UgMQogICAgICAgICAgICB3aGlsZSBUcnVlOgogICAgICAgICAgICAgICAgdGV4dCwgbl9uZXcsIGxhdCA9IGdlbmVyYXRlKAogICAgICAgICAgICAgICAgICAgIG0sIHByb21wdCwgbWF4X25ld190b2tlbnMsCiAgICAgICAgICAgICAgICAgICAgZG9fc2FtcGxlPShzYW1wbGVzID4gMSBvciBtb2RlID09ICJyZXRyeSIpLAogICAgICAgICAgICAgICAgICAgIHRlbXBlcmF0dXJlPTAuNywgc2VlZD0xMDAwICsgcyAqIDEwMCArIGF0dGVtcHRzLAogICAgICAgICAgICAgICAgICAgIGxvZ2l0c19wcm9jZXNzb3I9cHJvYywKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgICAgIG9iaiA9IG1ldHJpY3MucGFyc2VfanNvbih0ZXh0KQogICAgICAgICAgICAgICAgdmFsaWRfanNvbiA9IG9iaiBpcyBub3QgTm9uZQogICAgICAgICAgICAgICAgc2NoZW1hX3ZhbGlkID0gbWV0cmljcy5zYXRpc2ZpZXNfc2NoZW1hKG9iaiwgdGFzay5zY2hlbWEpCiAgICAgICAgICAgICAgICBpZiBzY2hlbWFfdmFsaWQgb3IgYXR0ZW1wdHMgPj0gbWF4X2F0dGVtcHRzOgogICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgICAgICBhdHRlbXB0cyArPSAxCiAgICAgICAgICAgIHJlc3VsdHMuYXBwZW5kKFJ1blJlc3VsdCgKICAgICAgICAgICAgICAgIHRhc2s9dGFzay5uYW1lLCBtb2RlPW1vZGUsIHZhbGlkX2pzb249dmFsaWRfanNvbiwgc2NoZW1hX3ZhbGlkPXNjaGVtYV92YWxpZCwKICAgICAgICAgICAgICAgIGxhdGVuY3lfcz1sYXQsIG5ld190b2tlbnM9bl9uZXcsIGF0dGVtcHRzPWF0dGVtcHRzLCBvdXRwdXQ9dGV4dC5zdHJpcCgpLAogICAgICAgICAgICApKQogICAgcmV0dXJuIHJlc3VsdHMKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSAjCiMgQ0xJCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tICMKZGVmIG1haW4oKSAtPiBOb25lOgogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj0iQ29uc3RyYWluZWQgRGVjb2RpbmcgRW5naW5lIGJlbmNobWFyayIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbW9kZSIsIGNob2ljZXM9WyJiYXNlbGluZSIsICJyZXRyeSIsICJjb25zdHJhaW5lZCJdLCBkZWZhdWx0PSJiYXNlbGluZSIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZ3JhbW1hciIsIGNob2ljZXM9WyJyZWdleCIsICJqc29uIl0sIGRlZmF1bHQ9Impzb24iKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW1vZGVsIiwgZGVmYXVsdD1Ob25lLCBoZWxwPSJIRiBtb2RlbCBpZCAoZGVmYXVsdDogZW5naW5lLmNvbmZpZy5ERUZBVUxUX01PREVMKSIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tc2FtcGxlcyIsIHR5cGU9aW50LCBkZWZhdWx0PTUsIGhlbHA9ImdlbmVyYXRpb25zIHBlciB0YXNrIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1yZXRyaWVzIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MywgaGVscD0ibWF4IGF0dGVtcHRzIGluIHJldHJ5IG1vZGUiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW1heC1uZXctdG9rZW5zIiwgdHlwZT1pbnQsIGRlZmF1bHQ9Tm9uZSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1vdXQiLCBkZWZhdWx0PU5vbmUsIGhlbHA9IkNTViBwYXRoIChkZWZhdWx0OiByZXN1bHRzLzxncmFtbWFyPl88bW9kZT4uY3N2KSIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tdmVyYm9zZSIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIsIGhlbHA9InByaW50IGVhY2ggZ2VuZXJhdGlvbiIpCiAgICBhcmdzID0gYXAucGFyc2VfYXJncygpCgogICAgb3MubWFrZWRpcnMoUkVTVUxUU19ESVIsIGV4aXN0X29rPVRydWUpCiAgICBtID0gbG9hZF9tb2RlbChhcmdzLm1vZGVsKQogICAgcHJpbnQoZiJbaGFybmVzc10gbW9kZWw9e20ubW9kZWxfaWR9IGRldmljZT17bS5kZXZpY2V9IG1vZGU9e2FyZ3MubW9kZX0gZ3JhbW1hcj17YXJncy5ncmFtbWFyfSIpCgogICAgaWYgYXJncy5ncmFtbWFyID09ICJyZWdleCI6CiAgICAgICAgbW50ID0gYXJncy5tYXhfbmV3X3Rva2VucyBvciAzMgogICAgICAgIHJlc3VsdHMgPSBydW5fcmVnZXgobSwgUkVHRVhfVEFTS1MsIGFyZ3MubW9kZSwgYXJncy5zYW1wbGVzLCBtbnQpCiAgICBlbHNlOgogICAgICAgIG1udCA9IGFyZ3MubWF4X25ld190b2tlbnMgb3IgMTYwCiAgICAgICAgcmVzdWx0cyA9IHJ1bl9qc29uKG0sIEpTT05fU0NIRU1BUywgYXJncy5tb2RlLCBhcmdzLnNhbXBsZXMsIG1udCwgYXJncy5yZXRyaWVzKQoKICAgIGlmIGFyZ3MudmVyYm9zZToKICAgICAgICBmb3IgciBpbiByZXN1bHRzOgogICAgICAgICAgICBmbGFnID0gIk9LICIgaWYgci5zY2hlbWFfdmFsaWQgZWxzZSAiQkFEIgogICAgICAgICAgICBwcmludChmIiAgW3tmbGFnfV0ge3IudGFzazoxOHN9IHtyLm91dHB1dFs6NzBdIXJ9IikKCiAgICBhZ2dzID0gbWV0cmljcy5hZ2dyZWdhdGUocmVzdWx0cykKICAgIG91dF9wYXRoID0gYXJncy5vdXQgb3Igb3MucGF0aC5qb2luKFJFU1VMVFNfRElSLCBmInthcmdzLmdyYW1tYXJ9X3thcmdzLm1vZGV9LmNzdiIpCiAgICBtZXRyaWNzLndyaXRlX2NzdihvdXRfcGF0aCwgYWdncykKCiAgICBvdmVyYWxsX3ZhbGlkID0gc3VtKGEuc2NoZW1hX3ZhbGlkX3JhdGUgKiBhLm4gZm9yIGEgaW4gYWdncykgLyBzdW0oYS5uIGZvciBhIGluIGFnZ3MpCiAgICBwcmludChmIlxuW2hhcm5lc3NdIHsndGFzayc6MThzfSB7J3NjaGVtYV92YWxpZCc6PjEyc30geydtcy90b2snOj45c30geyd0b2svcyc6PjhzfSIpCiAgICBmb3IgYSBpbiBhZ2dzOgogICAgICAgIHByaW50KGYiICAgICAgICAgIHthLnRhc2s6MThzfSB7YS5zY2hlbWFfdmFsaWRfcmF0ZToxMi4wJX0ge2EubWVhbl9tc19wZXJfdG9rZW46OS4xZn0ge2EubWVhbl90b2tlbnNfcGVyX3M6OC4xZn0iKQogICAgcHJpbnQoZiJcbltoYXJuZXNzXSBvdmVyYWxsIHNjaGVtYS12YWxpZCByYXRlOiB7b3ZlcmFsbF92YWxpZDouMSV9IikKICAgIHByaW50KGYiW2hhcm5lc3NdIHdyb3RlIHtvdXRfcGF0aH0iKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK",
"grammars/tool_call.ebnf": "KCogQSBuZXN0ZWQgdG9vbC1jYWxsIERTTC4gQW4gYXJndW1lbnQgbWF5IGl0c2VsZiBiZSBhIGNhbGwsIHNvIGNhbGxzIG5lc3QgdG8KICAgYW55IGRlcHRoLCBlLmcuICBhZGQobXVsKDIsIDMpLCBuZWcoLTQpLCAibGFiZWwiKSAg4oCUIHRoaXMgaXMgZ2VudWluZWx5CiAgIHJlY3Vyc2l2ZSAoY29udGV4dC1mcmVlKSwgd2hpY2ggdGhlIHB1c2hkb3duIHN0YWNrIGhhbmRsZXMgYW5kIGEgZmxhdCBERkEKICAgY2Fubm90LiBUaGUgZmlyc3QgcnVsZSwgYGNhbGxgLCBpcyB0aGUgc3RhcnQgc3ltYm9sLiAqKQoKY2FsbCAgICA9IG5hbWUgIigiIGFyZ3MgIikiIDsKYXJncyAgICA9IFsgdmFsdWUgeyAiLCIgd3MgdmFsdWUgfSBdIDsKdmFsdWUgICA9IGNhbGwgfCBudW1iZXIgfCBzdHJpbmcgfCBib29sZWFuIDsKd3MgICAgICA9IHsgIiAiIH0gOwoKbmFtZSAgICA9IC9bYS16X11bYS16MC05X10qLyA7Cm51bWJlciAgPSAvLT9bMC05XSsoXC5bMC05XSspPy8gOwpzdHJpbmcgID0gLyJbXiJdKiIvIDsKYm9vbGVhbiA9ICJ0cnVlIiB8ICJmYWxzZSIgOwo="
}
for path, b64 in FILES.items():
    d = os.path.dirname(path)
    if d: os.makedirs(d, exist_ok=True)
    with open(path, "wb") as f:
        f.write(base64.b64decode(b64))
os.makedirs("results", exist_ok=True)
print("wrote", len(FILES), "files + results/")

## 3. Quick self-check (no model)

In [ ]:
from engine.regex_compiler import compile_regex
from engine.json_schema import build_json_automaton
from engine.ebnf import load_ebnf
assert compile_regex(r"\d{4}-\d{2}-\d{2}").accepts("2026-11-11")
assert build_json_automaton({"type":"object","properties":{"a":{"type":"integer"}},
       "required":["a"],"additionalProperties":False}).accepts('{"a": 1}')
assert load_ebnf("grammars/tool_call.ebnf").accepts("add(mul(2,3),neg(-4))")
print("engine self-check OK (regex, json, recursive EBNF)")

## 4. Load the model

In [ ]:
from bench.harness import load_model
from engine import config
MODEL_ID = config.DEFAULT_MODEL   # change here to swap models
m = load_model(MODEL_ID)
print("loaded", m.model_id, "on", m.device)

## 5. Run the benchmark (baseline vs. retry vs. constrained)

In [ ]:
import time
from bench.harness import run_regex, run_json
from bench.schemas import REGEX_TASKS, JSON_SCHEMAS

SAMPLES, RETRIES = 5, 3
all_results = []
def section(t): print("\n"+"="*68+"\n"+t+"\n"+"="*68)

section("REGEX / FSA tasks"); t=time.perf_counter()
all_results += run_regex(m, REGEX_TASKS, "baseline",    SAMPLES, max_new_tokens=32)
all_results += run_regex(m, REGEX_TASKS, "constrained", SAMPLES, max_new_tokens=32)
print(f"regex done in {time.perf_counter()-t:.0f}s")

section("JSON-SCHEMA tasks"); t=time.perf_counter()
all_results += run_json(m, JSON_SCHEMAS, "baseline",    SAMPLES, max_new_tokens=160, retries=1)
all_results += run_json(m, JSON_SCHEMAS, "retry",       SAMPLES, max_new_tokens=160, retries=RETRIES)
all_results += run_json(m, JSON_SCHEMAS, "constrained", SAMPLES, max_new_tokens=160, retries=1)
print(f"json done in {time.perf_counter()-t:.0f}s | total generations: {len(all_results)}")

## 6. Results — the headline comparison

In [ ]:
from bench import metrics
from bench.schemas import REGEX_TASKS
import statistics
regex_tasks = {t.name for t in REGEX_TASKS}
aggs = metrics.aggregate(all_results)
metrics.write_csv("results/regex_all.csv", [a for a in aggs if a.task in regex_tasks])
metrics.write_csv("results/json_all.csv",  [a for a in aggs if a.task not in regex_tasks])

def overall(grammar, mode):
    rs=[r for r in all_results if r.mode==mode and ((r.task in regex_tasks)==(grammar=="regex"))]
    if not rs: return None
    mpt=[r.ms_per_token for r in rs if r.new_tokens]
    return sum(r.schema_valid for r in rs)/len(rs), (statistics.fmean(mpt) if mpt else 0.0)

print("GRAMMAR      MODE          schema-valid    mean ms/token")
print("-"*58)
for g in ("regex","json"):
    for mode in ("baseline","retry","constrained"):
        o=overall(g,mode)
        if o: print(f"{g:11s}  {mode:12s}   {o[0]:9.0%}      {o[1]:9.1f}")
jb,jc=overall("json","baseline"),overall("json","constrained")
if jb and jc:
    print("\n"+f'HEADLINE: 100% schema-valid vs. ~{jb[0]:.0%} from prompting, zero retries '
          f'(naive masking +{(jc[1]/jb[1]-1)*100:.0f}% ms/token; Phase 3 index removes it).')

## 7. See actual outputs (baseline failures vs. constrained)

In [ ]:
def show(mode, grammar, n=6):
    want=(grammar=="regex")
    rs=[r for r in all_results if r.mode==mode and ((r.task in regex_tasks)==want)]
    print(f"--- {grammar} / {mode} ---")
    for r in rs[:n]:
        print(f"[{'OK ' if r.schema_valid else 'BAD'}] {r.task:16s} {r.output[:64]!r}")
show("baseline","json"); print(); show("constrained","json")

## 8. Phase 3 demo A — the token index speedup (on the real model)

Same schema, same greedy output, two maskers: **naive** (rescans the full vocab
every step) vs. **indexed** (one-time precompute, then O(1) lookups).

In [ ]:
import time, json, jsonschema
from engine.token_bridge import TokenVocab
from engine.json_schema import build_json_processor, build_json_automaton
from bench.harness import generate, _build_prompt, _JSON_SYSTEM
from bench.schemas import JSON_SCHEMAS

vocab = TokenVocab.from_hf(m.tokenizer)
task = [t for t in JSON_SCHEMAS if t.name=="nested_object"][0]
prompt = _build_prompt(m.tokenizer, _JSON_SYSTEM, task.instruction)
mpt = lambda lat,n: lat/n*1000 if n else 0

pn = build_json_processor(task.schema, vocab, indexed=False)
tn,ntn,latn = generate(m, prompt, 160, do_sample=False, temperature=1.0, seed=0, logits_processor=pn)

t0=time.perf_counter()
pi = build_json_processor(task.schema, vocab, indexed=True, precompute=True)  # finite schema -> precompute OK
pre=time.perf_counter()-t0
ti,nti,lati = generate(m, prompt, 160, do_sample=False, temperature=1.0, seed=0, logits_processor=pi)

print(f"naive   : {mpt(latn,ntn):6.1f} ms/token   ({ntn} tok, {latn:.2f}s)")
print(f"indexed : {mpt(lati,nti):6.1f} ms/token   ({nti} tok, {lati:.2f}s)   + {pre:.1f}s one-time precompute")
print(f"per-token speedup: {mpt(latn,ntn)/max(mpt(lati,nti),1e-9):.1f}x")
print("identical output:", tn.strip()==ti.strip())
jsonschema.validate(json.loads(ti), task.schema); print("valid:", ti.strip())

## 9. Phase 3 demo B — constrained tool-call DSL (recursive EBNF)

The grammar in `grammars/tool_call.ebnf` is recursive (an argument can be another
call), so the index must be **lazy** (`precompute=False`) — the state space is
unbounded. Every generation is a valid DSL call by construction.

In [ ]:
from engine.ebnf import compile_ebnf, build_ebnf_processor
from engine.token_bridge import TokenVocab
from bench.harness import generate, _build_prompt

grammar = open("grammars/tool_call.ebnf").read()
vocab = TokenVocab.from_hf(m.tokenizer)
proc = build_ebnf_processor(grammar, vocab, indexed=True, precompute=False)  # recursive -> lazy
checker = compile_ebnf(grammar)
sysmsg = "You output only one tool call in this DSL, e.g. add(mul(2,3),4). No prose."
instr  = "Add two numbers, but make the first argument a multiplication of 4 and 5."
prompt = _build_prompt(m.tokenizer, sysmsg, instr)
print("constrained tool-call generations (all valid DSL by construction):")
for s in range(5):
    text,_,_ = generate(m, prompt, 48, do_sample=True, temperature=0.7, seed=200+s, logits_processor=proc)
    print(f"  valid={checker.accepts(text.strip())!s:5s}  {text.strip()!r}")

## 10. Phase 3 demo C — streaming + early validation

Stream a constrained generation token by token (always valid; watch `depth` and
`complete`), then show `StreamValidator` rejecting an external stream the moment
it leaves the grammar.

In [ ]:
from engine.json_schema import build_json_automaton
from engine.streaming import stream_generate_hf, StreamValidator, StreamViolation
from engine.token_bridge import TokenVocab
from bench.harness import _build_prompt, _JSON_SYSTEM
from bench.schemas import JSON_SCHEMAS

vocab = TokenVocab.from_hf(m.tokenizer)
task = [t for t in JSON_SCHEMAS if t.name=="nested_object"][0]
aut = build_json_automaton(task.schema)
prompt = _build_prompt(m.tokenizer, _JSON_SYSTEM, task.instruction)
print("streaming constrained generation (piece | depth | complete):")
final=None
for status in stream_generate_hf(m.model, m.tokenizer, prompt, aut, vocab, max_new_tokens=120):
    if status.piece: print(f"  {status.piece!r:16s} depth={status.depth} complete={status.is_complete}")
    final=status
    if status.done: break
print("final:", final.text)

print("\nearly validation of a malformed external stream:")
v = StreamValidator(build_json_automaton(task.schema))
try:
    v.feed('{"user": {"name": "Grace", "actve":')   # typo key 'actve'
    print("  (no violation?)")
except StreamViolation as e:
    print("  caught early:", e)

## 11. Download the CSVs

In [ ]:
for p in ["results/regex_all.csv","results/json_all.csv"]:
    print("\n==",p,"=="); print(open(p).read())
try:
    from google.colab import files
    files.download("results/regex_all.csv"); files.download("results/json_all.csv")
except Exception:
    print("(On Kaggle, CSVs are in /kaggle/working/results/ via the Output tab.)")